# Deney 01 — Video Analiz & Hot-Moment Tespiti

**Amaç:** Uzun bir kaynak videodan (a) transcript çıkarımı, (b) sahne tespiti, (c) ses enerjisi analizi ve (d) LLM tabanlı "vurucu an" (hot-moment) skorlaması için aday model/sağlayıcıları **kalite + maliyet + hız** üzerinden karşılaştırmak.

**Tek kaynak politikası:** Bu notebook algoritmaları kendi içinde tekrar tanımlamaz — transkripsiyon, sahne tespiti, ses tepe analizi, hot-moment skorlama, önizleme klibi üretimi ve `hot_moments.json` şeması hepsi `local_pipeline/video_analysis.py` içindeki **kanonik** fonksiyonlardan gelir. `00_youtube_automation_master.ipynb` ve `verify_local.py` de aynı fonksiyonları çağırır. Bu notebook'un kendine özgü katkısı **çok-aday karşılaştırma (benchmark) döngüsüdür**; master notebook ise tek varsayılan adayla uçtan uca çalışır.

**Nasıl çalışır:**
- Colab'da Runtime → Change runtime type → **GPU (T4)** seçip baştan sona çalıştırabilirsin.
- Bazı adaylar API anahtarı gerektirir (OpenAI / Anthropic / Google). Anahtarı olmayanlar otomatik atlanır, notebook hata vermez.
- API anahtarlarını Colab'da soldaki 🔑 **Secrets** panelinden ekle: `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY`. Hiçbiri yoksa sadece ücretsiz/lokal adaylar (faster-whisper, PySceneDetect, librosa) çalışır.


## Google model router (bu notebook'taki tum Gemini cagrilari icin tek merkez)

Bu notebook artik `genai.Client(...)` kurmuyor ve sabit `model='gemini-...'` yazmiyor.
Her Google cagrisi `local_pipeline/google_ai.py` uzerinden gider:

- **Colab Secret adlari:** `GeminiKey` (eski ad `GOOGLE_API_KEY` hala calisir),
  `YoutubeKEY` (eski ad `YOUTUBE_DATA_API_KEY`). Deger hicbir zaman yazdirilmaz;
  yalnizca `configured (GeminiKey)` / `missing` durumu gosterilir.
- **Tek anahtar, cok model:** ayni `GeminiKey` projenin yetkili oldugu tum modellere erisir.
- **Kota PROJE bazlidir, anahtar bazli DEGILDIR** (ai.google.dev/gemini-api/docs/rate-limits);
  ikinci bir anahtar limit artirmaz. Hesap/proje dondurerek kota asmak yasaktir
  (`ALLOW_ACCOUNT_QUOTA_ROTATION = False`).
- **Rota:** ilk basarili cagri rotayi bitirir; hicbir zaman tum katalog denenmez.
- **Ucretsiz katman veri politikasi:** gonderilen icerik Google tarafindan urun
  gelistirmede kullanilabilir ve insan gozden gecirenler okuyabilir. Ham telifli
  video/ses varsayilan olarak YUKLENMEZ; yalnizca yerel olarak turetilmis
  transcript/skor ozetleri gonderilir.
- **Kota defteri** (`results/google_quota_usage.json`) YEREL TAHMINDIR; sunucunun
  yetkili kotasi degildir. Gunluk sayaclar Pasifik gece yarisinda sifirlanir.

Bu notebook'ta kullanilan gorev: **hot-moment incelemesi** -> Rota A (`gemini-3.6-flash` -> `gemini-3.5-flash` -> `gemini-3.5-flash-lite` -> `gemma-4-31b-it`).


## 0. Kurulum

In [ ]:
import os, subprocess, sys
if os.path.isdir('/content'):
    packages = ['yt-dlp==2026.7.4', 'faster-whisper==1.2.1', 'scenedetect-headless==0.7.1', 'librosa==0.11.0', 'openai==2.50.0', 'anthropic==0.120.2', 'google-genai==2.15.0', 'pandas==3.0.5', 'tqdm==4.70.0', 'moviepy==2.2.1', 'imageio-ffmpeg==0.6.0']
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
    print('Colab bagimliliklari kuruldu.')
else:
    print('Yerel calisma: proje .venv paketleri bir sonraki hucrede otomatik etkinlestirilecek.')

In [ ]:
import os, time, json, subprocess, shutil, sys, site
from pathlib import Path
if 'google.colab' not in sys.modules:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        venv_site_packages = candidate / '.venv' / 'Lib' / 'site-packages'
        if venv_site_packages.is_dir():
            site.addsitedir(str(venv_site_packages))
            print('Yerel .venv paketleri etkinlestirildi:', venv_site_packages)
            break
import pandas as pd
import imageio_ffmpeg

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RUNTIME_DIR = Path('/content') if IS_COLAB else PROJECT_ROOT / '.local_runtime'
RESULTS_DIR = PROJECT_ROOT / 'results'
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FFMPEG_BIN = shutil.which('ffmpeg') or imageio_ffmpeg.get_ffmpeg_exe()
FFPROBE_BIN = shutil.which('ffprobe')
print('Calisma ortami:', 'Colab' if IS_COLAB else 'Yerel Jupyter')
print('FFmpeg:', FFMPEG_BIN)
print('FFprobe:', FFPROBE_BIN or 'bulunamadi; on kontrol FFmpeg donusumunde yapilacak')

def get_secret(name):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)

OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
ANTHROPIC_API_KEY = get_secret('ANTHROPIC_API_KEY')
GOOGLE_API_KEY = get_secret('GeminiKey') or get_secret('GOOGLE_API_KEY')  # Colab Secret adi: GeminiKey (eski ad: GOOGLE_API_KEY)

print('OpenAI key:', 'var' if OPENAI_API_KEY else 'yok (bu adaylar atlanacak)')
print('Anthropic key:', 'var' if ANTHROPIC_API_KEY else 'yok (bu adaylar atlanacak)')
print('Google key:', 'var' if GOOGLE_API_KEY else 'yok (bu adaylar atlanacak)')

# Paylasilan durum sozlugu (Faz 6) - tum notebooklarda (01-06) ayni degerler kullanilir.
STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}

results = []  # her satır: task, candidate, quality_score, cost_usd, latency_sec, notes, status

def log_result(task, candidate, quality_score, cost_usd, latency_sec, notes='', status='experimental'):
    assert status in STATUS_VALUES, f'Bilinmeyen status: {status}'
    results.append({
        'task': task,
        'candidate': candidate,
        'quality_score_1_5': quality_score,
        'cost_usd_estimate': round(cost_usd, 5),
        'latency_sec': round(latency_sec, 2),
        'notes': notes,
        'status': status,
    })
    print(f"[{task}] {candidate} -> latency={latency_sec:.2f}s cost=${cost_usd:.5f} quality={quality_score} status={status}")

## `local_pipeline.video_analysis` modülünü yükle (kanonik tek kaynak)

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['video_analysis', 'render_utils', 'google_ai']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'video_analysis': {
        'sha256': 'bcee971d6885d7e58b40f37514f93ca63c0e10cb44904470d936a7608d6f45f0',
        'encoding': 'zlib+b85',
        'data': (
            'c-qZ<YjfK;w%~XD3XHmHEv6+!c6a*Db&q;0lQ=V}$)hVyy6>#lB~ub*aZHiAeArQPUA4b#zijRQ+VcVkfTXNsy0@mQW@3@V0dR0|'
            'aNg+m`|qNv2<J(pZsTQ?jlwicc12vM$7-FGqfNGn(o!v>GFp^zmZ~JXju)y}hk3MA^IdOpJfns)-EaoYxYan`?$Wsm)1{hBX1lB^'
            't9djFt1{b!2y3<pi!#djvvIHYCM%<PmffhyvAT((ZK2|_P}yCoHdRu_qXj^Umth&Hd6X{Jn=rqL(`)tNr5Y@<%{I*OR}tMtd6@Km'
            'UB|^X%GD-YMu{roPf;;c#Ue^0nM{_$0~}kn#leJ$diClxe39i*UO<B-bhD#>gJEx8m8x7vDqNHmd>JO!Sss_`4Gys$O`%33rw@P&'
            'eN@Xh8*QWTrnik3IArb@;U(Du!145o_!>uHVI2p9b6RCNd=et`B)*N*I?5weWW9M-u2m{nD-?iA2x{~Z#;J;M6R8DEU&SflIfR+6'
            ')FzD668?u^m(@0j7l6-UFHV=yHiG|4%ValN=JBe;FBaK0jtW?s_kc}AWd(GSB8C;T&B7%tPp-DrJc)~SRKT=$<r*6Ph-k^uWT)~7'
            'KMG}{L@B*h3{VgQcYs4^#qYLxbQ?!^YLUcSq&}QT9+g#|0-?p}XcKL+e5YXjLsgV{1uKz9<K8ovG%nb{;2}^zo6xk_W@!;4Wdr`G'
            'FPsvfcgE-~#{VdQpvz1x<KkxA>-YP;Ri16sY__Tp_-v-)%{I$R7%V_1nk{-gd6q}|ucDf7^K20ng?>`Tn}}g;!*ZR(a}DTc_{lGj'
            'j`1`7M*kjC<4;*?z^}ssLFk`2Xfuek+H7|!EL6Jf_0HcvegENn_RG^(AI{FzRDJF#_^%&*gnba>jKicqRQ)BaUy^O{!EBp^i)fuC'
            '%P7ZJyacm6`p+tc{oNljgm4S<yT#wC542RRR`DXn=C~O6a23Lm(rfr|SrOsSaOrmd;vPo!;pd;<zI%W6e0Ki!!@Fl^vtM35KYKen'
            '`;Yf$Z_Z!7ePfMwyxD$_fX18bmjAusfA6CChW-d|V}9}dEkl3t_8Cn3-PvCR(x6|IVP4`#n|QfQBK!-88~<5U1+brf(ChV<(F&;~'
            'OKzi@dF5ex9St4<tH`Q+5zUrW&b+FK7DFH;Xb%&+h9}cEuttoaj{lI7YXF;AnkXK_j0Raheh?CW5963lh8}Euc=G)uP@|tz2|F{o'
            'Si$ZrVZiv$<td{Uc|A=R_T#$Rgz0FN!!j;mn<iO$J)+&Icy_ogfZt|mjC&K{XXz3d0RlSv7#Fmc^p+#egPRb!Hg4QFmnNfmSitrT'
            '^XrPYoO%!2`WcW0@LvM<4E9t8dlWh|e5!y!sjDj++gDd=n^eGq;<N&u!Ek7?6mX_6xeIqik5LtFw$LSv-(uLjn(&Y6CvfPf<B5jU'
            '$DUy#&}no7+ZA4(y?rs(7^8pVm9z3NPBq}A?DvCHvv9OPK!1evfxBkNCntgVRIi--wY{F$0GHG>fkAKwe^L{CX+ZpGswSrn_2gk('
            'MCz9?siLzy&+<Xv8_ZA8L;)Q%d;JN4Cz~*d`*qjMhPnHp>9`d&h)F<?vnA`nH3qio&*LOcH_<Li9YCk*v;FfA`Mr#%%y$;X-A!t+'
            '*|K@<+5Be}0ii8t53B-ENT436A131oHI6<mqHU=L?{`}s{?M9AfLs{9*fZHaZ?FQ6mLf*<BiD&P=?6u)iwmGtjebSv1c20t_GjxZ'
            'e6zdiQ<d_OtFmQy9V_74sk#YwX?UZYvG5|qyDS$h$E!9pJ=CE)pJjLx--IxoFo*soM@!h@^J-V!xCCY`0wCO9x?#&Wf>z3#PL=Te'
            '=8iNgn6$N$L!#NtAB-BA(FP!)oa}{)2Tl`YH$(wdM0F#F)>}A?caAv3`V|D*zNpzJke`8#C%|k$^5<23kX(I^^|igE(KTRag;cZ&'
            'KUq9Uqk?FL8Pg{GIDqL7_aHf70K(AntV)-Ix*38AhUz=`AHS-p@H;Ir%xLVhcr}A_V-dl*w;X`ba}6Y4(D{o81J8D`jutl{Do@pQ'
            '1kz)fn-5XcYJdUp1lt<wmgY6)<iVgGN|nw;@|^L@<zTT{4wV^Deewe<2mwNzER{4-cWV^H*ICM11|F9SK8=q+naDB_8)5dqr-011'
            'Ky%@>2mn`Cpl@xel#~ZJnKwX)Y6&xf4UQ_wHr|3*|Lb30zB(hQx<eQ*oZLTWNkUppoEIg*LtOx+MSL2rKqb5j^JPII!4e*WesBYW'
            'g%je%3pm@aRkT6HFiNxPdJV#Rc%5b-kSq!~^a&m)7@(ukFvcLQpq2yr$cOhYMt_8UNaiK|Doj8iK)nM+6V}3TWGFDa{@X7EY6B}r'
            'DhHmp5o%MQQo>xqYnUNX{7uZ$3DvKlMP+vd^-J>s-PVyK5D&-$jJ`1b8IYU~>J`StE6OFP%2WHRmp`A;^C-_7&ueZGeJrQ%^9sZg'
            'P)IYx=CogxtI;2UvLf7O#k4QeZ#ztx91wuJSsPr4w;v8t4+<uQV;WmzF+ep2=P?<d1dZnN9Mxg_iSR0GcJCPgtRLzk%^`rSRyYs$'
            'Zq0@;Q+M<%+knUnC(O_M!0e&Lm_Np4A9_;`Y?0v4AmyIXpD4Q643MB~f#L@N3aT}8Ftfl9faHOXA=ZL!=HH3o?}FafVUZvr{SJct'
            'Jq7Ne?<r5hZb_mT*gs~=I5*swpP)WG1*sW1B`U)W)dcxPoNlX9B$T_jT*LSk>!iHV(14Jlbpv=i8Y6bdS%fz|g-3eFUtKl&25xl+'
            'Y6i2DqT0fj(Q>TbLbv%H9!l7M8r?<0nm_{@MVoE8qmM!52GxQG2F%Wb53>dEu*yZ0M2iyPzIUPJXg`4u&^m-Gj(XHW4K||q<Cyf^'
            '0yYw9%?f$p7N$>TDC-P^;o!8?JX&QrOO&X`HuRSUB2F!u5p+#yMFx#e9UTM(eBELU3Nwy(5VRfdFW}9atbBndsh0NtW(%M5Dyd+q'
            '!1qDg*D&w<PIsm&IGoEt0RPO;mUoEK<M);o9o+~=(65+|v<wmFt$2aw=Ab6+(R_|bfq!hA9Egt_P))PEa`93F#v!v@q3xgr4DF&<'
            '^FyEmG<DG*K%VxPfZk|O4sU|*>LoO9knHFc8G1gBlk5(K4utri9j66|yKbOg(l?hj^6Cj$0<+m}AWNgU=N}nq2l#04;5dTdLF7Bc'
            'v+CQ3CPir(xvEmVDr5B=6_Oj+sIXzHPhkpjLzv@Y=|SK34R~2JAu<#;>*)^FO|+YWXggnqgsxLH@UTX_VV1(D^a5FfUJ;mN?;CZF'
            'j4;|{c=U|!vV5u5QI#`i0SO=Z8OV3nG1`_>%fhaW%Xm<ZduOlTz6Zq3etrAyxn+$j)*yQ!`sok&9RHq2WghP8zu2sZ=ZOB7ZGj-W'
            'm23IK*b+gsBa2~ty(SACsOEqg{Hx3>{+tZV_(wR;=vaiA>E=mxOXj>apif&KLER+NBVvWRL$eV5HBYjPhmwca2n2dr=WqsVaPc}!'
            'x%VQYd1gtq;eSE)r|02qSl}Pm@u!3gj{tiS(Z{H+Xh>zWPU0IH$W2H+MgIUfVNLeYGQ`QEt&$E-5Y+N;QO?NR0K_mO7r|h)Lg{uk'
            'kJD4MaDx&O-dSvT$b>k9(@=g>v-<Zw3!*Z{pHn5aLEAcF9%^}sY?7Wn1INj;3Iyc~18Z1={njzIC_J*Wl{W)mA!VjQIaqVl8$ZDY'
            '|LgeVhwmRhefo^-QF0Cnj?i~-{KL&(KdBAqDeBj!zp&I@0s+wHEbs%@gU{1qk;f#1J!YqiIS$o3<N}2PH3;_;aR3r6pjnlojI->C'
            'Sc~E;$Q8U8O6Y5>k5YFD^wHoxW!~d1j{eep8nnP1Z^Im&XPX;j*!-i=ay?pJXW0$?Z0qI=8uyH08N^-ZAgSpPNOwdVeYA!#&gNm7'
            '^2Qn^*>!?`1R`|AX1^xCZ`4Z^;6~&_BeN=qJU~-0mKc~2E*9FG<$Quu8Afluzq%R~#nqJs>@KT7X=UY$7k@H2olI1<MI-M3!9^cK'
            '6yXCg>Z+6PwLm0p9}65ZL{dG~d{=^g_$%saBhr}Au?V~sMOLk90viQgQY77?_O4j~je&|hH?Sc#Th?M}?xa~~q6SBch?HF5`l}o{'
            'xdGFMS@&@}<pZ;&0z~u*^=@K7-(E-QwxmBtv3%?9uV84wE9DbsU)mpR!Y#s@oZg;J2&j1wo=*OI-~RM=MXV5n(P&B=W)R;!J_#P3'
            'oXW2?{c_qYPRVbR#Pg4^y{Q90K*8J=h4xigl>9jUI9qM#?_G8K{mEqO5YzMQF5Sc*5sT<9j<T_Os-BFGjZweKZ)08@q%&T~5M8(*'
            ')nbh#n-n%Vtuv4ZUOoN$+Yj$+^*%;7WKq_h*%TO%LY^c5LHZ#B$r`O3qmc#BI4-NQB<F69j?^5zwZzHtXjLKa0STKp&e4ao7H(vK'
            '7PurtPr2&<U<Thm!Wjd~N^-`WuhH{|UJtch19Jv0w;av$?5==Lfpp@POP^u7y9;+j)QmrDOLiKz_~byApjL$WbyS+^lk--FLMoO8'
            'a0v?nlof>=K~9bkud5I^a2c`hC0w8)YSVYPzy<bcgpS9M-;KiCEMCso<Og~s)A}a7iDsXoJYyQoZtD$!6eO-b%PyGv?09HNoOwik'
            ')2TV=L`j#w3zA3i$Rks%n!Tx#<Y#$3FcN=TgHsDbsP+kU-jPv=-CHFnhAZGPAO)s7?SLX)I9BIX4(k>P-#M}=b`ynYQeO(6BsQbi'
            'as?coO`OXt;-Mrc-T{~V1e6Cy<mB-W^&$}S94q6%MlzjMz~PGhRtG?r{qY#>j(kWA8Zw7{c<9sR;>qcyC0g(rG{u!NE5pQfYO<l*'
            'VPOqphjXTP+qCV^K=aZnS3Hju2z>=e=T%ZGql<grqV1aI1ug9+Sef=vsLduD+oYua(Be<|l3@}&7FjMv-%a3HyfjK8%LTvn(O#cC'
            '{vN|qB5fQaPE(frkAHxH@^+_Pe@uT1C)h4Fv*P#%a@k`qp15T1<b5k(CjXl)0Xnaci7(=tFpnQx0TT1IL`p$VDB$z3SazqlB+`jV'
            '*hek50D%~>zJ&8+Q9oX6tA3Ykk8w}HL}uk~TQ>kH8%@4E%UPVRvQyyqSpx4~gh0hzz+Bni$o%ikGzwMEfX}iOMR1MoIM@orZ0fTB'
            'sSNn$3@y^Ij?p#w{&@}VD0nFf7i;|jnds*uBlR2&)e$fo{U^e}f5Y4B8BivSael-TI7+h_^cpSJGk&U$Q2i~APSu42uLegSz(4qX'
            'h)&OPcXWSw?^?@f*Ym_?GBM3EC8+E*t{rbP4lkQJZwB?+>cyP$KSO81ruGlUj>gpt4K_oCYll8Agg36qY2ci8^_svgME_Y8UPlsY'
            'GKWK&1~PpD>SMS>TMs~Hjle*`7kH9v&@5R@KlkBif>sHf?v@jMQ}j>OlgZ?Ms1S`aBkdL*I1K!VsY!-g6K0-O1rfuP>CbVsSdD|<'
            '>WJ28MBaC4G(<yhsf*`&|F?M%YzAV(9*0|um>bA%aj2lw2wTwDtD@^MJ_8;Bn&}|W<FLm-=2`DMXBsn<?HW_DL0C>4Tp9O!-G*#K'
            '7`lR;(N009*2Se~yoJ%=%P~7x#b=?U{TL&2TKyQZ@E`$Kk@LcgNF(fXqg~%@R-iEUyz;?c4}{&a&6cvY+uFgau2%RSn$Ty`2cf;?'
            'Q;Su!bz;4y5TEe``uyGlbl;$!JqYPU1}<9{E;{glP1@1WM{0wweZup;vbb&wXzOjutots1ZfWm+&<29YR91>`R=>ZuB30^x&Nu2i'
            'M-W3$9=<ip?9%}{(7t*EH?-bTuotc|{A0E={A>pAnKrXF;%Q3EgpZDH9S}pJuXZ}S@9mwnh$I}AJfj%pVqn`Abx?FIcEDza*6gZF'
            ';-b80i7dRd0}KlF^X?oWKPOl!ybdv<Pu1Owt!|E_gCm^J5oqCCRQitiEY@;FNIjz^abOu8h1dfg1V=m;W9R&)8FFX(@TQ@?4pYD?'
            '9IfbhXVXb*qp=AR*Ug{G2%pP~A%-sE1oQ(I2z0yIB0(Pf+2HZi{~i@zfe9~t@b_b&h=GP?Xx#zdV8r|5>xytqk?G6m1|U=*zG`TH'
            '$nVu)E*_0jB|{!6J-$F|6|9a-E$j_yWq?EwIk^y%vkfB~zPE8dX!&FuE|)Vsnt?sZK^w$z3mw~^5z$P&Hh|Ar{dy4S^;)T!#J7@1'
            '9o2$hPYjl9G2XSh0RtB(O%=5zHlP7YGXM>eeIp1^_Py&!x)_-L^r2uLAj=m~gCnEp)J`qrTnazX5~#?KQFv20T1uI1XFw3>cb=-_'
            '$q<bevn2pP@yK!<;M?(&JsyZP00Qc|Mi&=Ddvbh2tYsN5O2b87QBHxSWq|lqkZqT&O5_O$BwmO(1U3A8M6xId{^@p{F2g)Wd)HCE'
            'X*{ACvVuT<HP#C7Ly&66<MHLaIDg4FH{U7t%)_41xe`mXyrh{b?=nTuDBA#)AwaN?9mA7^hkzQNXGO>hi6%MR8}e|B`4oeOKwsoj'
            '(wIdy-V|1gRTNSZ1Uwz=rU0+T^0fXlxR|gYNJz4qqr?w_c}AlA01y>&_&c;40L*~9FY+$HJP*VI?%gm>JSjC^enhe(z}JXVf_ia!'
            'Iy%0zOexr&Y+6%mR}~k~`qJ@NgLXU!=f$9IFj64DZ!Y?JE&G=N#y0!Qz=MG=aWwXDJ<->GWrVXDxIV#6{~RjV&q*_~g&TOn!yl?2'
            'f_rBYh+Ag}7%a-$tk)hP<&283XgQ5>R;NuY*qnlAfJ3Bz>j7`@Z=zxNW%r`}lo(Ws4(nSyP&Bgm2dot?Ye5@JK7f5fYAxY_hd|zx'
            'b$q=xYs;UGF>Dj*%Y6ua1?H%uW+Mcl{-lnh(eX(`x`QF0p@9fi28q1#i8FtB!=Ny3cThtPj7N5heppTnn$2-3!#s|PLDOw2!iWC7'
            '*XgRV+bB<fl0em+8D)*7Ddc;X4>V_nT@Oc^vg|c9(LP2;KWtEdW;qB!OpMaP^#Q|5{V<+9G%yj{;n?P>S*<7I@18u+d<9#19xiUW'
            '0Q}pNj#QQ*9T&CKc}D97Vj|`3Ew!~>2aQpt8i?wf4QI2Zq@R<l7nc$+c1}}ahq5de`}qVG2w8EuNGdo3*l);DU928&!W}z#Hqey7'
            'k>dpChB2(<2~lYj=9o%IiGi}1z^pmml01qq1%!k^2!Q6G2xF{%#STico00WH1FS*8{Lyx3@Tvf>_-$20OBSdpfzO1xl7}Z*5}>Fk'
            'M;elvN9z#HH8~?mTw44+utV)h1$MefvV!t0*IB8({b8yy6G(TRWlPMqQ<4no*x2l#e;t0!te`>D!Vq)}%gh9R>vqUWM5nYor(Mkv'
            '*WB2JXv5L^g#)E$(LJ?rqp!Sa$Y#H=w0St2M*3}I`&crgXR2yrAx?-rKO=R)>MgJ>WtQ1ox`ruw<A>ST;qKlfv~>Zd@a^wC1>INI'
            '1Y{S_>;!-de=Q$53nk~FZdsGSg-sT9D@4wvta#0QDSLy$&py*a*VuYAXp%uqO?q=V<U}Yk+s?@5b#H7C71_ZUQJv9TnD#y}>oY*F'
            'dGn{Ki#;u=C2$hr+Wd3Sf127DzHA-%P7vQRp@7Z2&54qgSEsc!2D=jyXq!dANkT2tBbX-h#$;SLq%~?8{hzcL({YAc0%e%Q*J-pg'
            'blL#c^Fp&q7)^AI^XZ#!9je&%TO5&oH--TH$?&+_PTnBcGZeo;zgK5pHI`!J?6I#v7H`ca<XWLd?qCfsr70eaCE5B8ud|2z?O|y0'
            'N6~?#Z`AW>h4a0|_>o&6y^!QT8t-FfHas-qlqdp__`uRAow_X72sD;R#_DCsDL0&nPf|l^L=q7j@Ql{7g%(Hn3~o>iV8eMf^bXsg'
            '*MdSIGwaFFIbZ`fb!KW=T>IM$^k}q(_IdszLH}qO=Ug!%Oh$yD(e1JAK6c0CC}ckOw~gI>dtmLYX#7qaIR~6A<g|6(>~HLMYd!ZH'
            'I-8Ho(e?xDOPl`SJNqYpb>B>L^Om?ya&wjuAM1q^-H1qVX4Op?)S;H`N{v=DNyFT)!*uBhovMwO3vV7fma1<3gSU{YZ+s`&e>nz1'
            'xHhfH9!=<<Qlq|FfqGbV+B;wo#b$(5yWPp2c5yl*#bE}sMXz|f0r0yg<4Nr-aP_oj7>jvWu@h5UMGa!CM?x`g96`x$e*!ZiFbrjM'
            'dvb(Cy9R~Bj&Qf?vS?R?UF${c#Z>Pl7^`R5W*(;?8wg)s;*yp-A>a|HLiHoE(80Ega=qc40DSXZy;1O#S>b#z2mqOT5`CnZE#Yv;'
            '@oPIc`Yyuocu+VeqvOeh2dOX)p-5s1AZ0R^Q0ePDik2~)8=QAkaAYi-{BT!DAJ8#oGg-p8o||w0fvU?qG<`63L7*5r8o>N+hVF@i'
            'MDbXd-q22^X};x&VWdh};7MGgivuM<@K5n{hmT-|2-b6qb9?jly}Cy1V=W-JZVVRSi6ZlMCT}U!z8D0q7D|*flUdiMOU)y+l2;o>'
            'S+d8Y&u32{`qZpN{B+;qKa*pdwHV_TTCALY$PQ?QBDD?zZEY6-R4XUO#7|aE5Y3%dr>?vbEwrfYD$6j=dgwumRjHPH+7vlQixd~P'
            '9iz~4SorY<@Z8cd)lb%(aI4U3OOIB6(i1X(E}bk`F9PF&o!yuAs9I;Dhr*v5FRv3^QBW}Ccm)_kdvxBvcslwY;po%<`Cp^|`@fGa'
            'zwHlc2qs6zjm$uAZWCkp>cWXMxOB6V_K&;qt&a3SA3N#P>_#7Aut#gG2YqDs-&DR`EVGMZ6BajECV?Fbp#Rkd81>)uW3yO+PuuLw'
            'KwEhY?y^xLMH4RUI!(qSWjTY7$5xRPXGec&e1&B_IxxPXHWW4ED-7AA;T`?)?)$cdgfH73x5Pw%bXk#4nfM5<uHU~17;+RjU#iCr'
            'a{Y!R($wvWYGd(=KcP*qUQ;4Be9~gUc%UC%b3Q}+B1NDNYK!Srm{AuEYC0LJf1d<_$2ifMWUK@JdhvPe%|mYtH`~qLG6q9PZx(XZ'
            'arXkx>a*RR$R-N8Iq!NNzQqMPMlOQE%m@YmlSWcRrHrP1d|3Vi*}JpQCv;@t9&sXPdOCQNkJy`N=W-=?b%s7w3>)-iD<>Z6hh4ZD'
            'woTR!@wIZqK!=1uo*I|b#oYz`W|n@33+L?oOSZ4U7x>DZ=e;Fi6|QdTktE3BlVNj1kgVLY!6xJ5CwtH!A9NEAyq*l(*B=(~iLKa?'
            'KQmoMABV<Ek;0a&HYD^699&pd7GI_^Shaatjda5B1@&`@#08@0P@O!8+M2^(^!*;;5S})M(P%}F=~EsUZCze4UiC)}7&SG>$mII~'
            'M_fmGIO-4Vlod#tIeutiDMlTqkaCxvp8;D~qKY(j4CEu(nk(1%Z$7mh0+!(olX{{;+he?b&{^3D(Eh5W58r3%Opp19?tF-D)_=Fq'
            'C5)EUz5#lfgN6@wz|Ln|KKoW2(iT>@T2hZgL5wBA4Hu~Eds6Z`nc5vvcG9J7&$kc^Y7(UBaEha~q=jrJj4?p5t=HnD6{0Ys038HV'
            '#nJ$dyV{QFeC$e7<cXXY0~c!sUQp`@`W3@(mPh%?SWfq;-a$j~V}OW03&~a{x+VA3C~@NsJTsAF9MTq6ZQBB5b~?_R!ND329u-^9'
            '>wVSTdc4iHgLdhO<lCmfsR^v1;S9TsIn$2kO_)uC&b0Zc*{sc+Gi`h5Hg%WYLLn;T#5qw4u!sP)(c`X04gKBXz2cR%zaVk<Ay_q6'
            'YKNr5lDpveMgt3wSIqER)i7;70JJ%_PLN_0?E>iU2m<LpbQ*u7p4vHqb2N)pOD=~CLR*444x<s;_hV3Q=3!YTXfaY;!C<VOqKx7}'
            'WQhgf<>cH#xdqKN#(%)MpA|YAbl*rPg~Eyjm>=rFBPYJaj;TsASM$A%!lcldjGW~h>V#J~lc9GcTn`?l3Z*!Nc^Q`&tE<jS;`3M~'
            'nA1>euFDxh_>hW}da$^lCyWJn9Wc9M+_g08u#o6-S(GfPd2y{4izXFx?<g`(5sa4DZI&6QpX&xkzG!lEd2wV*mPeQOAm+7zB0lpV'
            'C`TOahD9@f7?>l(c0v<b_Cau-%nR_E(7siJexJOp9cx8;QD6O+`rF$#AI@JtReyi`HwAp7+=ZWFRe{(6y5T1J6!e2VJkhT6EOvnU'
            'B=!UH97g$2Jf6rs_6f_5k`AL|qpXJ=c5gv(T__p`meAT4Oywl)vFByBilby{0{MlN^*gG0gp}*1{_SM4b!@3*Z?5Ba*gS2I3V0OO'
            '#FmgC%ILF<!xU<XrRF5*UAM$(AE^Px{Od=QWO@{s;)3-YuC9zt`)(a$loq^7;uOn7I5oZELLqa+z${pauu4j)TrG2#dh%Q>EYkIh'
            'xww|9Qp~i!jsRuZY6WU`Ar-{KCFT=PxQH>wGh_v2wqZ;dm8(EDMFD15<ye=_NA!)=iz-P*k_B#Ft+7@&^KT5FN+P!vacba-Sldp@'
            'P2}O7RSHeX8P?aC`s|yX7g>UZFHA+UM|h3_V(bZFG?=cj+$a!MBVE_<9T@BlO|QYBxdt{Jey)Fbi3ZYvt`VnqG#|tu>vOavYIIX9'
            '(Nx2^w*-Ts|4JY4zg0G&MfAFii}k4S&7qZBDBZn9QnQ9Vro(A48S=+NAl0~>_HTev`a@lZa=PkI9)G3}?qL_ipO8BWCQ&I^5@RsC'
            '*s>E?y-FpY>2YA^xYt`&2e1el9s|zn*0az2@5K>{6-Sq+lau8=6>_iRqWFWp4U^7P$G~j5u)+hmq0j;9xbnmh$)#OBvte8RXEWZt'
            ';w+8MgJB~n-Ydo8mSgd!=zaK&dL3>}DY|4gR=<kygn9KtU6uwfl3@10u1G1zfgHjEhu^$@d;Z>36CoB8>hcsQGwzTVQbCYVR|w&t'
            '+9%^<A6kylrCgEcI+nN-eQGS@)hf!Vuqq9zgdS`51|U6PA~phB8mpuKek44a@{V;uy#j?|tf4dZ(0Oq2nq~o;(a9+sc2ad<PmLNl'
            'MDmb})OZ6wXC$p8#D*@~U@=iwTyvQEDF((?=}n4Pnfx&FOTIW|U$EK<gQ#oI<Wz-fC`TY8pRt5ua8~#gTHLZUa!!D%L3MPsx1Ekv'
            'pg$k6%`z>mrCPe`usAgCZG_9is%Q9B8#*dCOdAy-J9$iFOtoV)@$DZz5atJ$y27ol^>9qJ9(uM`2jX9aq6zVA)W23(;B$>deza?v'
            'E9*oe7{%7^V5KK?%1Ys-DC`UYF3%^3rS3Q<1oQa)hs0<6=)Ucw==Huib;OkdRnxM<cyBB{q^~_-!bI>Spr{k8#6_V+7m#oe90-e~'
            '+X8EmLd|%iL`Ij-pc=ZyOCN+7>1GEk>C7SWqBe7!1lcALpoW5R`leg12MZf&xvCP;c{g~J?F!g5Xb_7+lw*+<(WvZ(93EH3ITi#<'
            'Hsjul53gR$&YwMfBR3b!o_~1vlpICpXU{Mk=#LnN3SiC=Kbwq%#A^}<iPy0rOETqMNF}AUWY2f~R7<q*3O04c9)!pDR~Yo**K_<-'
            '@ROyuCK4`)I3YA~@C%D}#9LY>@cP56_b-2b_44c;4FAM>|N7~FSU08|JFnipK6~^2eD?F%JNX2IdvKV)l1*Kk7Os<Q&YCz~zC?FO'
            'i6ug!d#K1R$QsuyY@4d3xP}Ptt}X?_=^M3B4KfCDOUBh$okt+s&Ja{2HTJdOa%Io@FR$MI`RS|K*_*R>fBF0D*Rz*@`RjX%Zu;J<'
            'yh!Cq4y?N9RatzfMB*8DK`8q5uWw(S$x(JBd5w5PSQQO_e)gLD`p6YvPl*!rGj6YA%BfH*e`BUmv@EC`S`_(2%$tTg;{94@rO2w#'
            'o^UoL<$#hI92CAPFR)+2Kx^GZiIK5uC+fP1cao{1r3=<}lw2@oiz=@Ead@r7uP%4kCLjKG)sXx;eE;98A^j$5&KDfYUsp;=3hEYk'
            'J;;|-QbMwy*)}vBAeAk)la(o-)lW*EG?R><Wg&Wk-n~B8{0fhVE7SZ+g^N5ZP-N9x>({5h4A~$Miy)<U>hlPsM^dL?hb?t<8pyOd'
            '&GuAbVlN+5<y|(y9VQHbrRflD7rae**c9hQf?c83Rl3Y^yW`z$(RGM5E9Kt0t?&rk+RPF{+3V4WbEZcyuMsDO!$j!TuLx^2@o-ok'
            '#5^+kgQah|WesFIsaOtuid8z(Rm2si|9U0_&Tzg9Z=*UxfMe-3UV{fOkTIWjUl&6`MNSNH?L?&2j5}Ww&%yzY>DGlts&zk~_vuy?'
            'W>NOV8>~99D2d$bIx18B)D94fm&Cs6uW%vgsZ&G0jGT{X+PPb3A7e!b8f>eR2P>UxlL`>lXbwUcoGL@9Se>fR4Cs;i<2}A~3#`*%'
            '4!J_=I$B^n!Y7sGDfR|B&pQ5OUlT=iG~=vtahxAx4XbiMojhvLgJpEbi$|&r1$s;#ypaM;Vg`S>E4j~sJ%C1_8!|v>czVEH6uf&R'
            '>Jk$n=M`$U$mMm8GrWL_9sdxFLHB_N81O^W`|bpt-QS)3+Yf)VZsIXlx(Fftq>h^dkz*3?9pV9usbu=&$EF=dVKhpbHb*UgU7Rw+'
            'OTbaF203^zIv&=rI6x_cV{&@jz?L(DrmEGBH`<%p+HSHPcwHaM>LV$T<In9<Z(ZHmL{Sz0nb*-&^t*y%USlW-0^f6^JN$O)Ov1^C'
            'ua9A!8gC`DOY(x9%Wj?9oI7uQu1DV4!qB@34y$*7^_d+i=Y<(^J%rIQMpjsD0g2jj>qnE-8T^1YIWq@xW}eb%I8+a%w$~K^Ujq<8'
            'ES;KiVv?|k<{fQ$mZO2<q=&jL5(|W4`Y^>X>g*7fS8!15kmJ<|hVD~Dd0^YA3%e{Cg0X(0HI=^c1FB{1jlHoEyy{1RIx(znyr*;Z'
            '19=t^+Bi&igCOYc(y8~NX&U2SDe^Ti(hk{XW=3HWji>Mzu>F?G!)gGC64lZWt3>Y8gAW5tW(l2+^L<xZ$j>nd9(>U&tI=XsxWl5`'
            'wQXT&bZ1JyZ{#tEDE0!hQT9KhTBAW0Taw!_iUtHFk?gn#c(O4Xi^;8LmRXAvJ57tgmDw_hRc5KWJV{03xnSRu$-f6!@6Py&3Z<`h'
            '<BMuo`3E86%9!A0oD+97l!QKG+uLBU&kfy-h;DY)-(0cl3vNKvb;G{)I>9pB=o>LgEEb8^v(b%$`>ztT3Br>KtT%!4v3Dn+oE%M('
            '4H`SZDhlU9dT4g@EV+*KBwD4<f0B2hk6S3pa2;DOqoPKe<}-_-{iG(nwp6*L6+K;UxpWVT7<Dk8u&R0t)<r8xMuB*UXnRG5_EPz@'
            '0y2q*U@A*@Y)3S1!?fkX!^hlV&*c{_<ti!e6xA(Gy`fe?`*nyobpHGib-VW}K!4mpugSjl)vBb9`lm9!Z`JYekb0JFmkc)BwaNh2'
            '!qo9<&3WwwOkx0wF71~M_kC`J8M;*(<JIFc&DQ#zWi554DPvD+Wl+|WY<Yh>Qaq@YywOO)@~`e>T|nO_=}q6sbh0mng!Op{b?$Xj'
            '&|FX{g`^%tJ<8;h@src0IEx#-O#mBOW!V~nx3p|6EbqoiTP3P&dK1`I#tOx-*(AVy+&5<a)3h70O^II+Je7rZ7>$h3PiLd`b%)!T'
            'VQWp`pk?TQVoke!EofKzOyT}gQY0v#){e}1IAr;uU}ZC+?s#QWhiHn?d4t}=akQ`}&?g)%?Rapq63J8d+qR(FMl>-ur7(X&Lwg{x'
            '9_wjuZz*5~@o*@Xj=7#rT~V@y5Umy3TNkYn4X;Zm8kk;&*aIxSGlQ(zT}xuv9zD;Cnm6U`9)M_(coSMpZB&fTpGgbXcT<170<&KB'
            'i7u*F$CSwWiD}wsp-sol!2Rh*zd4#to!sB0NxTcw8{fs$^e?@9bN=%AS<}BHk!Ouc2Uo~?MsR-BsYvXoG{KK`YB~9KQIl}aGaZJD'
            'Ap$1AD^&k}!UeWM^`BK78J{4BUFetEd4r0|QPK1t5~s!Wt-HV~l-HEYz76%Iz8l4*HKeSV7eBBHZ_yx4u?3`k2>3PaR4%)TH<n<S'
            '0i}+l=<}0GfO)Jhxug|lMT+iS0v!q~OcC<UvcMz+I%l-A6Un%!v`dp0@8x-IT(Gn@b{v!LHX6xV*oFzjTu)yQtB&y_E*H9><XsAL'
            'q5DM(9eS)IA~FfLOX<!g@*x|SQ;AAp9g8EDI!{5Y6<L{|RA5tKBEmeA89Pq3Cxbq{5}!3m{jiVL;<E<S<6z<xb3h_5YEN69+#{(D'
            '^zt@XuglifelF@jh-M&u<6WKS!~32iU0_sHd67oNe-WpUK?A>+hqb7qOO%nu4q6gQjs%R-sz3_hA7u98PMqRUSSoUAeb@4tho+&C'
            'w^H>f-g-P~Xhs{H_F82$ZFQ0%gOceLs<cB&TGqW80@u>OPluD#AiW0smJ{8&<67)q5w$W0QxsEXKv{mC;rgNRyBX2-OLs^IOspZn'
            '4M0DDA?|nZ2gbADqogJF{DD}3N9v(z=stu}zF=*n4&gG&$;~0sqEB%hGK`BcPjD1{OKu%LP+FCde7VrsOw6fePoE()^5eV@JEhXS'
            'fL!5J&-<SePtBFh4+^OIOA=AhHbTNl&?%teV7W|6TL`X())s$jNKNN-;UBgDYi2I@apSdI7}z4n)4^pAhq&bIrsuH!M-SD0{aTx|'
            'T&6Q?x%w!<+vi5N-;H2i^Q?a$ptg&W%*#DjBOOw7xAQ)PgG){Q&W3i(OcZ+ykLADkG@5o;c)Y9;6U!}$Bh9ELCuZUW$g@^9NMVIY'
            'oV|Jf^4*zzrRR|9zF>iLN@LLCHPR<jou<qyy1A38E@cw<7hhLdRp>Oe!O8cN0%H^;tAh-_+M;g7Q;<9u(@oM+LJvh*n9~+7d(>mr'
            'Qngw|cjo#%Eq_Yjo-OxoGsG)^4YBC2A3A7BqLNf0vh|z+l0q=2I*#zwM`C{>>x3)d3QLqBPxZ}FAkWiX$9Tz>2_8nFTZ@epyo@P('
            'ShN>rNW!xwD_zSt(_$}1%oK9LBRWA#?eB>7y^j<HCIa_BWa!W&7fNj_S{4@5sQeVgk_^T5jyzEF^ulrD+`-v~(d1YpSQH|>mvGE+'
            'Z70ZYxWU<fygYyZ^37im`bgG%8*e$s2Sj&DM3SIAwE7SzjWLerz``B%RaBDxEB9n@YOh^zOjj8L*g{1m-3O{ij>H4{Q>{pboQueD'
            'jCcnlD-E9C)A$B(;6tNJ5Sxy$xN{n>P<I0bPHs|jXcDjJWFJ)OXnl5#=#N{eB#9C6qApOTDPL4@W$}Y6uY#9R5ACG5e_*&e?l_88'
            'q?9f6O0{fK#ySP*!uGl{+0%eaWUYPc%;{fAG?7e{YL$^Hj$t(mj6Cp^nBO39%{skw=%Dqe@y(&pg8!nfwaA1*nTv;Kmf^$ku(4?3'
            'xYp*?67P9s(Z!+}oC0c2W|Xb5!t7Z&8MgQRNbLzg4s;FeZ(#F%?SI1F>=da2qvHocgbTiU{e5NdNwJ2A)YS4xK6KAp9UiO$Qf(T0'
            '?2!EUF1+pQ07|k0Osy*ytUwVePSA`W!N>k#?Zk{6qk-Bt8v?c6>tlh!HQ_AT*t(>1Z_VDY=cYL5O}}w<YhWjG$dtQ7G(BkCST4pH'
            '*a4zLYflW-+jIIOXk6)vxfbSfUAD)d>gt3TWBdTHpeVnk^VA%vKjLZrkE*cmM@FuMVH}aVu43V)Xlnzh1TT)4?SVcp6I+161i(8%'
            '#=y)BJF|%st_;5~kV_X?oweaD7iKPE^cYMq7|&)YuKMg=-$3i<dSG$O!KzQ0q2WAClUUe=+J##QrNt^8$#sOH3{XBW_V79ebc74o'
            '3xm&g_xC#8aut(J&%eN!;?6nL6XRcz7XwcZ%f<TC2}^zbXExZu=+OFTT@USMvn&lZB_W1#&n#msE#A2M<|U7i#uXqSy%}uc0#j*S'
            '=55iK5R!M*3-FO%&vig|#@+bj-Wqo^^_~rTdsKjF%d?^5@R&OMx;-hB?&;w%20HDBR=-Wfu@ZsocK3M6*G@_JL$rvKr3rTU=Q;I>'
            '6V*ZIv&Ttg9DVP@gG2hWWS`Z;s<jo~hOiAdz>=;Z^*_*8?zvpF@!GzT&;C@#&TCXOhvVhr>YOUu7$CT4vjrS-GpTzZ%6K&A>(Y8x'
            '{Xf&v>+CR>He&26qNB5=FoXaYe6FK;F06}pFELEyhe>ch1}=0H--LF>2CBHC57(MLFtJ#Ee%27l*VLL->>kly0L7fytm~9?jQ}o|'
            'Oe5&#Li#MF31ID`htE?p*P%7y2Q+k>b@+9^<e?)HT<$rgbzRi@E@r#d?o4hOZd_O0teez&fAzF!*D==~yOx@kvQGAO$ZeJz=NfBa'
            '1AFI*A)?lOrp@4q4witR8{fvORtC{|0d6-(@zBk|Vdiep;7~{&^Vtj4xjNMC;Sg?OOksF5y7w(w^sLQfQ`c(N0>-;t1mjJR8_wL@'
            '35yH79PW~oDDIrpz6;NiOoS+OyZ;Wupc$dA!}n?^7b34;4xNYOHt+4N9M%${=x@y6pq;L~P;XQ0RAT?M2dmAYLc&X<N`Es!px*p@'
            '>hrf0fNoBpl`oODIo+2?+N@Yt+r%4gO5Y4_hQ*zeMf`Jn(2ow_(8J{49k&g*;T$a!Yj5SpakMk#M%#PO4<X{qn99p72Tl<X=YSel'
            'TNFw$+KX1qt^mwi1=Y)1c(vbasQW0<sY|*J^FrD>*OCwULTu0;B$DvHA#Q(-O2@+H+Z2<_jV>n=65n)^VSq)R-Kp@7uUxY;uS>Ft'
            'Y0J_<SevxsYBFp^!`|EQEQIXW=I;GSQ(LEF`P&y(i~tsGBFd7aIb9q{mq(N$X||KE#+lOLthG^1pH@+%^_1MTA8!KeLFMZZCCpLA'
            'JZ(wjXzh=qi3j_NAAFHA<S9h0vM(=*EI_RkNrYpqc<f7g^-4xJ&3wRg)jD3x<J`Vo0adbC^7l}w_hfGrx!cA{X0Bx%{3iMv*}Gc|'
            'ZDK)3Guu-bW)h0`1*@MWNVlqp^Dy}$dz)Jlwmn?TAbeP>;cZq(b4{{6Xz_OyTrA43xKtOzP&Y|D*G1;OU;+=8erqZPcJZa&VXiu~'
            '(GEQQ*{Z^1CK()7F^(r-*NmH*I$u4>krF%B6o@fSmbQE`(=Tfxcj`&md{i$E>@}<vgT5Y?^+%tZ%EP_2Ve5-2rv5EGQ@Wf-*Lj{k'
            'KYQ`?!>jia$Z>92UmUzo8`T}ORAD&d)b*dbr9r2Bw;Dck?Kxbi+A{$;v6{ABiO6>9g`FB|CUK%k*<S1>tmCp;(APde`(K*aPOFy0'
            '2%V?MdEI-qxNM#;0Xo5|duPd46qsd$M&C7S+}Ep!_U-0-?Xzr|8V3gNL(V<_88{S-fv|W%5J`=lXz-f2p02v7c6NE6+p!nqiEt&<'
            'ev8rYFr4<;Jv+l;LJR*s3-WB%k-H$(&yIb2k0qtpHel}~b~|Wk2(3DFT{W{?qa+1#Rbw;xsp+qqG}c8(y<z_Juh4GI_HwY|SV)Zs'
            'F25V!6*KWa3Q-wZq4^=Y6)e({%Oj4RYs6hKg(_;p8^(q&SOzVf)Z-Np_Hvu9XLAat<r7f`ZVo~RDu=Dk+P6A;WJVN}>s_m-8=@SF'
            'gBtj^fp}`=o=9j!o0PKQsgWaG#Q1&-T=uTBG-XN%NnnXi#8NL%tmSAmu3ABX#a67XFB`eB#jRzE+>l|46sO8YKB&+8!v*hubmEDr'
            '5EBZ!FetkmDrh@>pVQaDgA_v15?aa8##j?B+!Jry3b-r$xUC!7vOp^18j?*{CbFW7Ky;z(M|@oq0MsQ%sbH+NP87>Q+L}b;Tfx`f'
            'GzM7N$kX?OAGtAsseMs8tOlfq2rCguO=@X7ZqS`o9+gIG0y&&vyHWzj<s(ty0HkbT-#;>!rsN)0G;E;r-a9SY?2vR~w7uwVYix*K'
            'Q~mtmmX4jU1B(5Ptp&A>)e2xV-2w@f6|SVz(*71s(jV2YPv5<H`Q|UD_C0wRC=zY9<xUN-Vz|DGuVjwbx5FPb_+GEKNYLBl1c#W='
            'wKv%)+p26%-r(E>=a`!UG4M;_G8Yt81FB=0Rf~1Bq=E(JE)3Qru<khp>DHC?`c`FSH6D*m3)W7#g_dZRrb^!<t)%u307X^;h9uuo'
            'Tw=%9Ey@bE^}d$eoMIiGJo#Zpu6i;C;8y^f@wbexS^i&##Ts;@F@HV?>WVs4OoxT(&I&qwIYpJ;;L#c5-4`6=d}>#585~bezNZNE'
            'lkefIpZEKoVS)#vO*#b6N$8UI>*yo@HP96mY_&z=IAz|pY&pYAPdVktMFP;lIc0zl78Iy+QqPGB)SIf;(IUqCAuvFI8Lou!0X0@h'
            'Ij++-uv`{~fMW>Ocv)?>MXL@)<3aN{#{bO1ViCtv@>~unc$}`>VV8&%#*9Z?ndQZF&>y0Xeu^pS5<NM^;9nMw$2^KbzbaRwKlU9o'
            'dP^<i4O;NZN};HqCYNl`YnzDVayg;ux+w}FBySF79LPrD^?b(x%kuOejYfr~tcjjX=ah-s?GH_;RGb_2{FI7zh}XyKGc38W7f#z!'
            'Mhs|H0!Wz<MjD2?@=-$3c2{tQl<_KDloFqZQ5O0Uo?&E!r-Z9agclUzEW=PyYvlTZ=;$Lfc93e0A*vOI^*gsoNY}P!2_}nC{mmrA'
            'V-lHzm9)i?FdSRE1&)6kaphV#jZ?~UL1P%qQd>KgLi%1w3gDfakPD}xIp-Ec1QLdAir|kLyRVL*vw`d_I^KwBEFy}Z)Ag!AUvhxm'
            '8)B!$%B4Q6MRPb+;62XukGg=AG};%;$8L|=yR%<jp8b0EeD>$R1L_NvVSLV~AL|0Vi+B@b<S?N41~x5d#IZIvIo4ORS&-l*aIb9x'
            'xq(&hciW~3zG-=H4*&e@)!DQ6Fm64Mr*HnIw%Ml`=v1Md60Mj8+9!2o431$1uszEq1$fj0rtknanISmUcyOISK0RF6%iF)aH{{{q'
            'L<-h>J@1*MC+4?JdHUJu<KD8y{@BnHHqk9!f`}g`VU85X$vxhP0xP5|aClms55%z)__4_xA;Wzu_d`m(xD3RzAa#~z7SIc7Zv!$f'
            'srFkZ@-8)|eV~V4000915^p8m-~njx;zPTy3p=?>YK?Rsu~NWNWRBo)m&V8$L#KP*d3`TU%G8!ObG~qfaM^3Afp5};<i=F1(6oCG'
            '0H2+9e2{qqs`HFkv;CF~Dpps>-kqBUNHiORHbjD2OO<{HZc9@X!3X%y%#=g(I<_DjP!MqCBj1#P&x#!~U|@Pw*Rvjk?X7Ac$`wQ#'
            ')w-jEiu$<6_`y5e8!E+Cqs2dskYw}v$TXYW8~Pvedw<*aUxc63F^PHcr#Out8}$#p2OD^zU?br5j;Lbbl_R;m{W}WM$yr5rKZ2@s'
            'Y8M~qYhi&xW6;Rs%Ne$1hjlKp3-V#UY`)bR*{RaREGH7WlIV`(wDy3h+sHn>Fi_A2*m5MH<;>nwLodX%z8oMy1z@M%zI%T5?%Yb`'
            'XA=@~gkzMDLQ3S`KnF`Q3tI5*))A#m1FxeLn{-DR<r09a-cbTLB~zC=DO@*c-Zk|NNMWEF@~2#rzfV~dy8`WhMRk2m253yUFZy+g'
            'xXA>{Ru-&y!`C?swXwn^WKdZ~^)pD^k(!|0pfI-zphLvy6c(_-l|ca^dl_A+$Lea8C3xjD;MqaYRWP<qk9I?Qm5zrkU$Nm8SWdXS'
            'jp2Agv!bhR^gDX>*By=a$KABjv}x}r9@~=Rl)iDebK`@c9YS4arnS9(?YC}Tn&$&(e$fS^YY6`P#tbMOxG`{g>VIo?y8E#_-g|wu'
            '+roO)a5UKm;%W2Vg~l;QK7wYxV&6R-0F?Wn>uYm>*)7b&E>}d0U`5t>e(ro~bOic<b%C}I!m|DVe<4>weun1MF-SlZ)%6MHJ+8_H'
            '9_twv+QI|60nHd*Yb}Kzd1{KTxoH9DHhTcLFR+OO9Wdn#2?1#|E)Oa;6^i7e$c7^>G`0#1I+}?y8mIR8-WdgQ>$=y^?ew4*puxAi'
            '+Bb9!bH3^NFs#Z9)+Sz9=j<uRkT*z57hW|=Z@Y2C{Isni`z$8=4H$J*93Td(D30X+@gB$1bBkgH9b9Gvcpga<4+WklT)q{w{%{r5'
            'Mk}q&UtHSyFdR-PUr0MJ0EcWrYm`B~Xh<t6jfxcWvx@9fQeo2Ib)57s?FlcPYZra#Xy}VXLnHPKJ0y}lr_j*d;X`*nAGWWC9&Osb'
            'Xgt~ZhBeY|cz#Y{5$FE3nnv8PHd+Z&C%K7K6mCIuIQq=jp&hB{rigE9MXO;`;pO4St~Q32D>+geya!m7+^S&hj;T2$w>f5Jl~l0Y'
            't`pkq)LmNYvnlX)4#H`z8be}Q>*`uFEb7yU&ks!4+Ed(KThW$&Z@9w9&b?l3>I*#f-r2WpxB`_+R~xm6(CDqEBZowVz3Q>q?_4$4'
            'uv6pbcT2N-1M{2F*FGkiy|uCB!#Zqfd0?My9NzRvjJ})9noYTd>GgLvipq#2g4p>e*t5$qqnz2F*pz4F-I#unWYDpH_HanrQ|*0s'
            'BDrDXMcjfy?ORi|zAY-Nl`H-GTO|*)+14zk?Q{wbG2(W2<5ZGr-5qX!#z@{}hI`1edPpB)9JPyw$Pu;6Zu-F?=a6L;Z1T)D@jVdB'
            '=$?_XL5RR=V_}qH+APXZ5k}@lzR7S_RGUpmxnlbl)90x^02w;1IBo}LmWN_@Ok;0FIYx&>MG_VE0o!59bjQ?U()@bUB(!|c5ZPjC'
            '>{zN6`(od&c-Dz$Pnjf)cUv33YAnof;J}Hn3~ZW*%%MQ8?jK^lJaop!I_WxWWy{z;+15ba&da9TVs)idR2|BBSdk}F_}RHxLZWna'
            'J<o@Oa3Ai)jmtGXH9~U_9(E%>mq+V5f}3vOKO?@y3%s|y3SW9(QBf1V@Q*+o88L|!f+M~tGUo5<QH{jw%Hn0TmD>WOL}i(cw6OV@'
            '1A|MBEew}Cqr`QdYHpy0(NWq=Si`2(YS?0o8#c^q-p1mhLfQvI8<qNPy|>3wGpXHB4KP?h?kQ<WUupLf>&YXJoPM@?=6j1V-x(bP'
            'wdRCw&LT`$lsi?Qsq;VN_wE%+I6@;y_{h0bKPms6UC;MHFU#TM7Vx@UjP}{u(r?>{{Zf0@fSXnNe#`OdI}O{1Yu_rC73jF0WxQi*'
            '?!kAWVs--F!Qx%QM+X(O8B?(c{?z}b=OlF!?{Hx@Ia52_mhT!V6qc7qg3(V^KgkwhGNKw)9LPPoJ!$viczj~7Gl!t6byVdjCNHeY'
            'j}{lfT30`E+L)!f3j9L7di5Hm7>;q${xJ>|*;aEnKGyaS+K!O^Y3s>iwQ>>#>L1S3tW<&ehYf?z76ib5wOJV({so&1v~ks=AM%hp'
            'aiCr1YM&y}`0G9D%CIo>B7<lnI2cItapz##$hm*oO67jqg>$4r8}~um*IopF<C*VPFyX`%1~GO76}S3#9=m+%;NaaKTMAb+Iq_qU'
            'Zxi^*XdYq!wFA9nC0aTd@MYbWYZFL!v*Ege7am?cJb&B3;$>SGQd%4@!=u>-Ke}={inbh-u7`kXn8al3fC*q_cy}ZgQ7WS7X6*J8'
            'w7X1f?f0~q$Y#`@rbN{e8r=AZVvxxz(0CPeg#UWUUxwGZwd}EK)^`)S2cBycqy>gN<KvSj_?Uu?o#)3=-m@QVGFlAprOWrw>I5GU'
            '!-3b>RRS9gSn+HblkrnpP5|t^RkhVExc7ziT3ZESzB#2^=<O1d@tWxGIkB$0T5GWhlVndB)!(f?>z9^hZE7!gmGxf4G?W*-MV1iq'
            'K+V){kl`}imazU-Whs3>igx3W#jvSc{1wi#5nVq##Nf;8BpOL5@2Eb}xkjJ~r7-0)2Pv2G<ocJijBtLZi#2G*nDP^x>RL9sP$-b^'
            'b(PD7^+Wq|cNL2Q#kiN7+KZtszM=27rs^9*CD}ce$!BO%WDDCE!Ii*<Qlh`ER{4M?rNd`gN#a1+l*O0HWVXwyvZ5+BY?{R+V#+!l'
            '$J<>xr-1*$y%Yh}o_8PK%zk<K{Os-Q>6@po{(k=Q{6{r8_Vy+!)*{f_!1#XmyC@%Vu~S?!GbTXJ9Ajw*jby5!TUaHK46_2up;>EJ'
            'ueH5?SW<P1;V~2~tkW3^j48n^9E?CuuCqKY*WPX0->551xmR>Tfhf4efnuJOq)BjSzobtGk0sMyuDLUP2{<bCrOUtJQqH4wcq_5V'
            'c<Ejtg5!^x&<ZS_%hiFgj4$4UjOAssUR^`)!%M2>3o002PQeKnI9}j>Dc_2qV{N$RZ=dV_VJ!$uP#^ym^+kWr0;85-*(i%9!VYea'
            'y5!Zqij;N(s&3Vu1yAV#;yV2smm;{qgetj>`wLRuzoHJ7WAZty5LT<C<W|jVEeGXJoI1lPpCi59fTpF^;U@~WZuweHD5@23K)O0d'
            '_gR?xXG(1yUSi8MX)m>PKxr+{^1i3UmhHv<Mm<d>L(59riuvC2Rak}z=o(nHC0>kZTRw&@Hj+dxsb@?3bRBB`A?b~LtENuM1U5ok'
            'afDJUtHyFrn9Qxo@exMB#Mh|83M^gVnV2Iq0dW_24QDIxsHMy_)wNMC>7p6F9mZ657EnfYAGD>pP36p{`+Vw}mx{l$taX}6Qjj09'
            'v?$_KM&$H>=?$j6F-C5Yxx&#DCfRirp|DQXo-x3Fysw}k*(#h$J%?xMZFBTa9ew!w(|0dLVZWg@#jJ&8mM$wbxXIEAq&`4ev5iYb'
            '&Jrw10|&xi>#`-qO_nEIz4*x6)JH|QjL_JF6}<p6|G+)?EpYdBib=Bt#zMyFj%u*SJD3w!G+l%@KT0(YXniBdZ9-=({?8*UVg=~i'
            'A^y}RDpSr<*dd_tQR{B!qnDV29!y-~DA?Ry9nKLus-@XnZ*RG@gW&%bWz?MNIZpj3=Le@J(h?m|P*j@T%bF7MmQ}w4=cb0txlIsO'
            'fH9F$9v_JMFD?l8wT0mJ7l3P=PQmc-x^BHXF4}N7BkuuOMq81ML689SEkKbe9v@E%!}9dCkO)G6QH~mr$1@SRw8#{F#ETIMtJDBz'
            'tM*Y=!+dj04qIn*kQk{fE(>$78CB&X#YS5cKo+Pd&%WkMB#fxH7-M~hcH&T-Of=6o`U)rNQ|Y)T#2z<z{S}2&aN0hl*|CI&Vj9d^'
            'Mlt@1D(FTJ2f2wo3S{j8t?-H5dW6QdPKAv#!5cL2lkLK+bxxrTu3Xb>^|+&ck!7fFo!n1$=#{IpVzgYdqmeZxyz6Pu3U;K9_gx_6'
            'bb{Em%(czQ1y;oR3)}%npnRXkGLKhyr3NcaAy#+WjzhZTR|#|5ff7SkY0&9~#-7lrB`u>@D}c0it~CN;T!AD^d!Uf3D+&=f*v4%E'
            '@~#m##Eql@!dXT21nR5j%uO0Z*B$H4_xwK5``$vlbv3e&O1nk<(rVF3w$l38u8c$%(*98xid$N1PnF28uMXL1CgxpD|8JXOrZ(kR'
            '1)bWg`>=mzSZy8j47>p^cjiBBM~f}3>Wi=2e4Q=z+WH67V&elHr~#OAT!e6F?Lw4mKq|*nGfF55`v0nB$o>^eko^l*8+({Ue~-#z'
            '*6XhNV-G6C(>*?ze}NKYc-cvQ9hD=JH%9uX#i~U6kKQhNA_1zuia$n6xj|@T*aAWgzNoet^Wf%j_E3>CyS2shTbelzO}m-J_*<I&'
            '9wpI!=MprXg|5H49L+&RuK&eK(>U26Ez!PS(UI;fj*#A=wNqD{kV3f(NT+BFc!;s|D*1osXcVTaO~Q0%n=9BXF_0<NR*Qg56hN8F'
            '!WSgiB#)vnsR4Pk0)>GtOv|%H#AcKbaFAe96Nza_cP0sHuQ$<-<BcRP!b#(>y^U^gAvTi;Rl?%lJP90W0jJXr?c$(jP*%jLI{FMy'
            '?vLCsVbtMG7LwC{NAim~hj57~_|q=Cj2m>)8SQsub#O#R7m_F_4AlwVab<EH2cnN&idiDOa$xCD9Ui?!34=qEBj~s_tqSPmomhTs'
            'zsXWhwpo9;k@MmX`LwO<zLN>VpSRZz_Ltd0=EY|i?EPgkA%?2cI%zR(Am#r9Q4JHV'
        ),
    },
    'render_utils': {
        'sha256': 'b35f4b1004b927d6b69d715ed563a41f6c14381fd82ed2fdd02e10630b3e04c9',
        'encoding': 'zlib+b85',
        'data': (
            'c-rl~+jiSXmLU4BuZU8fX^@#9P05#Z(Z#H#wB<@qIV~?snVxE;h5|{DL<t1gxJafoTx<SdUgmS=Yvu>eFPwde2t)(`O1@OrsqV_|'
            '%A|mZjkxUCvG03-fB#)r=D{Qm#mkqgb+`~&n9Rbgw~p3f93`PxhVeSga*;2CES!nSMhu>exzSiP>PPEMG7+7dWi(yxMF~F3MUaT_'
            '-^%DZh{L4t#q%@{CcOwcuCaGgh-tdsh;%NBWt59JT}0C^bef6lFpK7!ahy(rxWC@)F<cR?)@fGc+<cYJN*MGk%ED<8Z^S&!;6*5?'
            'lbnReSLszKiZCy_{k^?6DfCPhFrzFK<vLD-SuUX4x}3yOzN8Uvs7YVEDnyz9*et|G40ImF)Mh4bvIv0gnU72QU?AX049l07>vbH#'
            'dZlHtE(@^^ie-Oqe}8{(o~0`>9?#37%);?l&I-ULX%Q4rn&f+X^4T)TmvJ;vfBrR36ZJa_)vtV67E!GKDktkKorZa?o)k-*15Ap)'
            'JPV4jh*qKcuKL>-`0tNt5^_5nFnn7zdkcT=?VX+-o&9n;{`KhfFUO~1ByRTv{Ief^01ScNNfE^RzSy6I*I}Hl@xgc<2h(tw#;`j0'
            'Dhscp@CNV)5ChO@-)9KH8b*DMe~bMn$;<gXnnu_ih8N(&c>t?Suc6_roZ>=_abg<)v4xTS^5fa7v)9Mt_dmTleS7?F{PW3+<JUAs'
            'uZWV3=L;`i!QSAHRXB^v75<8YY!UXZANF7~CaDL2o*%tEdv)?=eD>4v&&T7p?@nI6dTq?rn}zeBj0<llAJ9V>6Jo&_5I2W`!OH``'
            'dOJ^HB1y0c;koxB{A=)QDNchV_uMvl^by;9e`vf2V5-;Qc$&s(HV*O}Vf@Qa1Ni49{9}Tc1bIWdmoMFRtKh@<Ce3Ep<dO4y3Wpq@'
            'KO9&uvPG1Pui^ROq4@%EJj|x>S$Oj@i1W~R7l%aw6e3>-Q@}lFAUkgPKi&e?W1P;%fS!PJ`CaRI*tVWO2iY`?<1!bg&s!IC@ONL('
            '!>6CRpl4aRdp-Lm$RY!Tnr6|O4hF2>YK=JB(JiReiy&O3i8zDr2iYb}eDQ7>gvllg%xAwwd6;c-^U=?nJjkK|PK*ukr?kvNv*CN7'
            'wPBX#zIc%Z3m|3%?5;2V3}gPy%f)M;p765<=T(NoB2<9y2k{0zh9e2#fNlT*bnC9W#fT?0TsVE+dlxQB=-e-gc^xg5!Qta4*kkZ;'
            '@aR!6dGh%1@abgm`0=CZ?}NwFgYe12+5GVLvq$0Mr@{9JPal6D9v&P%9UKI~{QD<QXY<*?;K^h*AJjmXS&VR(MX}C@4<2N}O@9#;'
            '%W?wilue-p5Vig^T|HQ&=^_pvV5j`SDu|K?={$xXat8VI!98XZN7FC?BsCY|<mKyr@u6sH_7w-$@RbQ%fk2~yD#rmGF}y<XdvJI{'
            'fTN&EAieudQ9)I6_yp+o7$xEPM*KNk!*$E-$p^Tkrx9`@t2H1vT(A$aG>#|1^h%_3mipqTEYcNxb1MFHawamk(2<Eq(i;g67B45('
            'W0eA50T*x)6>-?Z%Zd0ZlR^ne*+ejb>#pBO3deB3KZfyV-!RX)dkZam^7QXd3%~!`wD6#RxQj0Eiu8U(5(YbR2Thninmv6ud-!N}'
            'Fn==r{?Wm~?;nPbg6aIx1c<`?_m3YQgpY$K-%p+d0nv@wU=Vyin|yy5JPHT%hmXECO<;6*z?1kdkAJ*bEQ|m0_{HCs7RW(;HW{!c'
            ')w(mEcK92|v?ccNPe;JaoxB9D?%m0o<5S7q{rUJW?@!*nK<=(n9YvN%m@9#cYDrfdJp*xrE;W2ys5=k;xRJM>{H7P-b;dC8^{l!I'
            '=})?a@mD5qUxE?AnrHJ6S=7Dm-rgQ?n!x7gX^bZ{;ENr`unIfKJ>wT={P$4gMFw;50-#g?igxcqF~)!CcV1WYegJW4S`3-xM#+zW'
            'gU~Vy;nS-qiE<=!@?)A-kvyPDC4CabCWRy@)F4DXD)=0H)Ey;u;SqNG@H$#|y8Sr43A0X@x<_*Z!tc~<h8h{H_aG~YC=t!f`q1Jn'
            'vH;i_<SEMVILp$^W{L2h`Tma)NUW<6c2iGYuI#P;=I^rYGqK;$c@<tyQNp9u0@)8b<sdSJvrhByx?3AcSN|n<d*-9q33m}Jr2af$'
            'h}r^PPv(h2;JT~%CTRvT4iNg;SR#Sk3lk-1BRqTY@gKP3rc2tr1X|h4)VT^bNY+WrR=;5b?l@PlXGY`t=nQTS|9OM#-(G!P0E`@*'
            'v$gg0s>v0;n0s;z!&?buTk(_j`n3U7LLY2GlJfG8CD1X!^yCQuF)}f)ZRZRC$*MNgY{j268p$y@ZAvBf=EMT<55LAGz$-lHE%<k?'
            'pkK)4-%4U5??Adi#`7|cJ4j}UZU7GPD$Sw=j2_lR>L&W!N21XU+#tX=g$eaU0yhD=1H9HUKtf(EG@{XTSteJA7ZDJjP8_TzvtTG&'
            '_K|z)JbC>1;ghZ}CV;N?Ch;)(<r>u&9fH&~w?MvMh9CG>M-f!GMo{fy(bF+rA07S)YXck2>j#uH0CXA8qO49le-5sYa%vh0AVhE-'
            'L^1Jvq#24ghBgaknTvIlByhq`e>&>HqV@L(*pGb43BJo<4T0x;m53Zz8(<>|I;qZ3<rUH`6n+544Xz|qk_55%6W!1R>m^=<q>&O)'
            'E@tUXf?6(ALnTvqloTlec#**++CwfEUP)aU%@w#^rhGHGn)C{lFQ^17XQ&Z_oefdf7T_3BwFeJ#*!d|b*We3qpG5O{n4yXf*QKx4'
            'jQ*=uY$VJTKi9Z>L9G-Q1I^@%{vs?o4sWM#4!jT4)1g66`8;wj$ukg<Va_bat*svAX$hP<vK9FTPUwp80(C+NOQQme2$f--3ODTc'
            '{nZux8x8=@KOYh63Ofc@>o~olKe@vU&@xI$;|Cxs@DKDvn81Hnhf^dVe9)@(IJfxL3->NH!1tq^maN0t)%r0v+UjdTj7ILa?^sy8'
            'XG}=WBGWab*UVaOIw28W3;15GILz~}@cD6;riEJcF^_uO?MosCOa!mA{15e1Y$Fbg7pwyGpjy9XPW3{qpb6#9B{Un~Zw*;*+eaPv'
            'q1D|t_pZ7G9bUL~UVKh=bIv|80Os}TG_R&>likKK%Rat^!`5JGqKRLLCf#Zrj1YL=D$p7+h&DEHJW@Rk(hOck@^anNui+HKd*IXE'
            'Bj)vQ)o8YTMJFq^pvX4%%;{#cztV?4`L44LBnnm+tW8hLV)W=SU4VeDYY+h(Uc(w<pR5Yw$)*VNj%v>rp|MDE!|E8Q;Z*4NyB+9f'
            'v&Zks1P9CvvKJ+=O1L_Ft;y>Ou9HPp!aW`2k(dOi5X_^Go;|c`Hd#oj|4t(^H?Hx$*V&X|jio*u+_BWe$$lo5-hX$(4^z}I9n*i1'
            'y2F?hY<O26);fcWxij~E`ODF}S3_R&ILd=07guo7aJkivpydMO3j^HN*?{U0DgI!r&|4oTgZ*t7Yi*2xZq>HZO_sZa!QT234S-jj'
            '?%o}U)*wusg{y#8paf3vW0b`KY(+4OMCaB%X0$^W+b*mIu0p>-G~0g}WLI%S-~y3{IiRapgj3**Ho-cI5pLg+#gwo+-P>zDX^q$A'
            'uCvxK^|@!v6r~Jx$gbc?2An=xNO7=g%Z-n!E&sZ=F#p&(iii0ejl$yzfXkv#?>lJ^@awtgR82tEn!_$lLj0rKH~#_wZX+fkv09p8'
            'OJSDc|NhVa%}c!s7GadiFS7MDUq)+E)XIl6jE&E|8MqGv7l^p<1>`_OA|ZP$4GU)ERQD*!3t-EwaRD++)76SThlA^j=VZan#i|5B'
            'r_4jej6et6ivTuT8l>mYG8d#Q2DTR%5aij#rTkhiy6jrqGF)Qi3JtM@>h>7e2CsD~@>;XkWKVWf%Z;)<U&u5ZV2<e(NL(OY*<F#s'
            'sl5z8gz{*bW2nQTcQ&fEY_*cDauBPS!vNMf!xMv&U0sNTSsqhc*fp}A^DYs)hLT~pnuIg76XA|UAoNriz%UnCuwEiOwVEBE;eD-p'
            '&qseiSU&&?K2H;%dIj%N9tULRhs}b|+yvQ7s7J`v@%VEB7>M^kbJH6TqDWR^m9sz%cp^H_hd7_-zx_7s>Mb+0EzAU}3())RH@r$7'
            'RMHCEFC9{Uh#$lOeH#BdKe#}K(Hj~{FA0=h4=%)a0)P!Y_%A&<9A22agU{qvOv)%GdS65!h>nd_>!1`#bijv`GAout6bpT^45P)e'
            'kbg|#G8^a7VinK>7=fvpZ~5nAny$vMgdpdOIovnc?VvvZOi8C@KK@WW9rPdH35`2#>`vlaAI38VVK|JHW^;cqyxzYP9PkHDw|NFP'
            '$n5T()N4#G2_JOl{Tf2Q3kje=`gRRCo8sFrSh3Iz==9+y=?l^qyagWI;b_K~X6f34Bg?8|T>1>|5X9F-h7{71dICjo1M&Ri=eMtq'
            '&yL{}0UR9iu0fIrmmF#c!yf4%U_1+a2U}POye@OnK>#zrc$lDH3L>9$QK5<EeL#FThNSFftyf<4zdVP<zPO}bm*>2K7ni(*(nlwU'
            'AFy8`;NFJCUOdO_6(es*=|h9f$dzYaCgjh7Iwd}Q@lCMF1$%mgGtuh_<hzJA;19VVmkJdweQ~KbSdPxzwo4ue+&|L*mHE>&nZm;1'
            '4GCj=!}5EKR}BzC+(4!;Nf^#@@dn+L;Of(kO4DV4^p7Y8&5}OFiYz?^*mJn;$|M37wl6O0y9#4k;dZ=y_3N=*dfr6RMdYb=C1P;6'
            'SkfU0w_-*cRNKOvw2XnK0U3{xRK-X;VO=jLi*cS!<wjl3<zM(*uGV~U*>~3rrW&ltRcl6C9lSV!N3gXyc<A9kz_r#n#IwE%K6Lod'
            '<FV~jGy(4<c+r*2a*#}ya7BEOeGgRMLDg514X4NE&K?`pgFeorPg{7H%k*hTqap*$>qIx--eS{X|IvK=;IRAc;ZVL<x-a%knVF{k'
            '15Ro%I3Tf9$VHJFAvo9Y-T6NL7~K-F-41V=&~AtD-Clar{q|NZ1$_n{FMH{-dc2{>3^$+SdYQYVZWK}t;^kq|l$yTAX1M43=eOzv'
            'Z7+C5YNLcBHx0*W7A=7Fia`!vlg0sg%Tbg}Q3D<N`wrY&W99oX8+$w-_!79kd{3>N+8SCp&)eSOptt9@`Xp~J_8nt@InxG0SxxvG'
            '@nZ^9Y=%y0XfFp;&O$N4R)FU`*Z0HT;cyo694`5EP0>sT?0P~LeFGe=i(e7BWst3a(a4D=1p>J>=mrE09m)!U+GOs7jcrhqh<so~'
            'I9niRvjJMyg$o<bH&7Em<QpXMAot9eMFRNwzm+(ApbSG17?AW=rO-$7qQlCcKKT9N0Eu3TwEN+sgNK6HU&U03gTd2*0gmW-UvXNq'
            '6gnY63Q=GYWD)}+1hg1Gs1(0A<`{`}kQk6sa#Q#ilD#P8-4Ov^P)WETJrruM^RO^hlvXUwfJ|RSNYUoz$btxVJJbXm+bS1NB6j<E'
            'Ih`W8-tT`B(!+IGj1JG4KHkFl+FqQ`jX$2bsBJa<nrEhwNcaoH@)2=jquYur+rlu~1~~t+0jdMB>r8LqBkk*>myv%+3>kIa_Zq0('
            'PrVjeXKvUIicque&i5ykF<0UU^INz0pWkI$(V2u-fWBce6b89uuGgk0-Tkh`PzU7<MJj`vhWp-FNPIm6GLy!>0M-Q%8N{swxv`mi'
            'yqnv^KD-Z-@AU;a-dv|~xl(NP9WX#bCJ2es3wh@Ws%stQIoQ89K)dU~-CgTkcL0y-%=6FD2cQiwI?+*!Ner2w6-5#V!Z><|lX_4O'
            'h<R&d*d%AoeBs$LBv=vZ0?7_FB?}mktD7ZY#btfdfctq(JeXp7frv7*1RQU0fxOcB0sfGzFHQrW0-xsaxVb?3k6damLML~WzxL!>'
            'z!0CoJW&fouJt#|6p+`LPA*xxHy{h=dBw2XJeO3TG@zbK`|$8M+qb=7v>4sGh|+ZRq9Mt_<Gbmr#<J>L1{Ndjf?o}!<5PwTPnv<q'
            '7l9(}32J3#;RXI!MIRa|7Ix4L@n6K^qfcN4SNsS)=yP;o9|5oVc*u0}wmPfZVNeu6Ge+<(jFEVb9u5ZQ*Z{<%`Um9;HKG9iHKKjb'
            'N5GJjJkwE}CTO`H&VV-IzL1A+M6V73`c9(g?#yTH$`|j5wYIb=UumY=JE}~iQp-?6O5FzVVfhf%=*2nweL>_81j+^NsO@N?bS-It'
            '%5piXaUM>U^5K85+HyqyVdAC3Ec(v@P2#W!XK(pIr4cBBi<Hb1(XHM_^p=2KaKR)ATuZ$e?e^CoJ2H#)9SyoK-r=2=Tl(V4y8>As'
            '^j;`27$+~_ddq;@M0o`y+F2L*v?NV>AH$4DlcI_5k?{R^a`uzBRI6|)Isu5glRV7eie~8jDO`I+r^{x}%leXBA|p)@7r@1l8jy63'
            'M5e@LuL7>rA34R)u(%9o3Z%&%z)69<?d5529%Rr&V(psTrbMtn2An+2#8rZDrFg3b(gzX`+kyfrP}`(LTDpRB2Wvvg5UKqw$i)|k'
            '^9q(`sJ4Q1Ccwa_H*l!{>nUNg>`swAMJUC48~`nWLBLUYw_vuO$2JL($CdjM$7;P{HK7As`Y1NwOMS6P#}1$<92UgtUzj)S;WB`u'
            'Fyng#bqi`HAutbD1RuH=c)cxy>o8|e5cD(2<v40Lh1wAO3SluI8ZJ$JYw=CP(?5*Fpd$Ly6*`EWn~GqsN6|GVYSZ5eyYUY~aQN{p'
            'V)(^gW%){{s57lj6CkpEp}+@yh45qE0_jtM>dukv8B8WEL^4?7kyjS;-c$G}eLNp|QXy+wzVX9-CYXqvXvY~0?=oD?J!OzkW>C?='
            '!zOMWaAcpNXM9x8dvG%hh8L#QPEu=ZC0$LvXpPOWd-^<L1B})i9#Fp|#bc1$J3^a}8VgdU)KP(lyhdBBy=iaBJiBLY8O#@LvSzDS'
            'Z;yRRvTDyQjkj`|xpOChqb=tsXpRoL(i>lrMD%__Ii46l#YV|kxS=G+4*89UqOg-<ypiC-)w<aDB%TK(h9aj2ND3&KkOmJ%nU;$s'
            '*%VL`#iNc3O$1~HUyD;T4@nKcH*#HwCIZogOD(<WO9vt*>SGv<ad9CgT<l4Yp}82-tW5K3&5UjW>azx~Lz^aXP#UeJ8)&O3c!;A-'
            '(titv;>Xv=gTaCI5#&$hsA!op^5>F~hW#`8)nR4Gh{p^P5&1ovE>PSR#ua3Rv`JahLFEc@%26Wp6kz$B2`@unsBi@EHDJZEj>3BM'
            'K*ZBKj^<<7Fbu;8XMi<ykGnQ*38rq?mM<K8_EiXYG4{nZ%ATl}!#G&w*t~`b5eL)pMqcep&{cUbcRb>FfU=BBI;8r~Rqizy+0k(}'
            'O3LtVPl{CpQf;n}GYzxRQK1r^+do%5`9|HNtiJZ%JUH^qJ8&MM3Wrol7PN0zMK7mD+GQRg^8&aQO7RrtYrNwS8tS^A(Ix5tgC)!X'
            ')3a<t?Q%4|lu1-bb1>=nWvmSR#9!oY*tfk%nX5x}GEqBUE!qQdFa|n}|LYIzMN_CHIVDmPbY`TqKj`{`|CZ430S;>9<%zdbR9*qZ'
            'U=fbyo{Yl2MUcb(V7`5DOVH@I$P;mu&SE%awF`u9uaP_^H$kKhl-%VXf!D9jc69?&bN2p6SK4knT`?dlkDG@z@?p^RGj4SJVU3Z#'
            'O|Y(jU^=I&Hhg^pOSjMjiL*O?rV9OO4CrL^Q}pGpDekZw^JBH6@}?MB1QV^SNDGWdN<m#kDYP&PpTsCQJVG;j(1>gXa(yDU)v>HX'
            't=(uQd6XSuPls;(YF6lCsBU{DuPK)S77g-_;X)v#Htj3m1gfy*6Fn?Kv{rFHfjsn$f+qi(<NaM+ivN0g@`mmYj=U!WYF)0v{d7Dr'
            '=0Tiqgv=&D95=f^|KI=lzqN`7HQ_9h1?cvz0|e0_M7e|T#`jRho@w6@7$REeP={@Klt_zgUZALy&gnzsO^pUemE)lAD1gl6+l7S6'
            'mjk$%xMX*#euZGW81QrB6Ua7Ro4mYdEr0cZcYyv*=lCDYTR^xVJtHnE<h+BizYhzzXPJj~v-2${RIZLXqAtsRXq`!U6mU{~N+Hov'
            'jbRcB(|Ogdqd+;`D9orUDw<LLN03{4;e3u>K!hVg4sz4rXC&C&<x(CupYUF~L*<qI7)3^P$+E)?4A5~DVoMZyPKEHD0FCZydC2!l'
            'ncnN)X4Pz}#AIp1M`D*(8VNi^g_PZuyBq<eoE*q9qnLVfCJ?De7mzFCPhXs%23Je`@ViWOkS&E2bP`tDdM~8ZGW3{zM8Jvm$dK7d'
            'aEx)6aU!qzSm178&XY9^B918519-a?n%>ol5Q(w9O>aGQ9ERfj)<YDWj*HEj-Q3YFghTyi<f|7Y#s8482W*-#o^5^6@AuVF@#LWb'
            ')bx}{#@>9~cq<j(5y|;vm|SP*=@_!>8H`E0y~(|&K|J8^D$0uZkcEqo?aJf|xhz*fQUx=!+Y;gke5cCOq0_G{gS<!ce1N}5EB#<k'
            'hgDJh(gQUCW~`<10}vZn)Cl8wWOf-~qnp&;#sRv6=<&s?Y2LGnJo7rt0`>z9R?#NS?oSd;vD_8NqZUH<jY_XW&b~P@)+{%zKxV-#'
            'CwON_0Zsh;=U-)7nK?N@lc|-sT?3%<b`$LLCdkP$ffEw1m%$_~@@FXeDmS*vOHEcUFGY%8i8qR?a^l!IDqI}BIejIFh^~P`hz@z?'
            'eF*T{7t{5@;p0cFCBy^H;;aFTWkHgXXHm!wnymPuQ;UjqOc#SFg|-qOa5lK68<M#)lDiFf_@PyRh6n((1Z*yr8|h~iqiT$g9U^{T'
            't+ZZPz|>#PUiO|oGvLYV1J#9u$-&&VF4KX?*l5IlKu4#~U%i4cWFQLW{lC!-rg>810%Ad33foE@Ls5qmAKK5uAe$~bSw;1J%fIdX'
            'Hv6tStp45k<M6jW{`JSM!ImLNTEs?pr%W!tfo?<i-}!(4ExGvikH2F3A^+ikV)N+zK^>9Nss^gEJXw=f3QUTrGX`xwH%B0!-;Y0c'
            'Nkf`%f=wjSt8zoc!(h=59xHZ0ZW*(mq^FT1H-=~0%ksvx>HP5zzx!?0g<ZNm^dE1#AM~%zAMpjOi~+BYmg#k8U<4hov&RUpj*k9_'
            'Y<JM@vdp5sO~MSWi|#J3fad;(clVbIpj1tF-g?rNrn>T^TY~1{j7=Qg_bb%L+jgB}!d7QJ+!|6<@PRSUa#R2$Wf8&;rI#2vJ$PMm'
            'Q_8eIhSVQdk^pwax8mWG!9aXRt1F@7(<k_hRWtHaT{(E|)?cv_@|WMn2be?HZY>^^1&kWkxkL0SN}%xP8~}?NNX?VJ#U5~zV9QIm'
            'E|3^2o`MYPI{wwKE*RTMo^RN?qkLa;Ua)rabj)9;0M%sHMd;}L!+$D!k#<oh0_pJCUZWX}={aBlS;CZ3R?Q(OFodN*?|ySKU2O}}'
            'L;_h-;$;Ism^^oU`X#Fj7cr703L{9+nj;Ai&c#P<Gra+(a<K?A3%)7nH41q(-Hi6nZxyiR2K&{rwQGKBnqKz7BJFmjwB0mdQ^wV+'
            'LrU#E+q7|~G-Qs?-bsnhE_ZlOl=iEz9Mj!hKQ#T`n>BQBXdA7<(j^@;+cdw%gWn%Cs545EF!B&%xPkRZqGF?XVP*Rar0){0D@Lff'
            'k_ky$><j3TugJ`ken)}u7?#H^;S=a^2xPcKuNr)XBa)2)v%MKGN>Q9MLcsZ(#w9r@vTL3aj`{4}1kA4w?>K&4?Tdpp6~cb)GsuQs'
            '6d$I@Uo-7sAr}un@SLb4L0%tkVwsvm%87XBFkThCJN3C$$J4pW>68Md5+};KX(UaRVU>@`gI3W*N~x4|1IA4$?s&=6l0yS}YD+4-'
            '{+vVrGY4*^`G%IcdhAJj<)xKz=vzr5=|-oRXF3;q2H<m#V$nDf&qE859im)s3i3BG)D4YkKJ*AJf*s7wLaCy}4`Q%q%3Y1$HVnh`'
            'T4iLowbQ$ys}l1tyyQAxAiv@Z-+5(2Fw$R4(+u@t1`Ol)C3!T7L*pH<v65fw(6`-sRJ+{W3Wz*VLpe*oIx|DHtukocqRfcR`W9oR'
            'zTrl;D3>II&a$4n)4+X1`J!S%cogRjPGLLwd+%QHo;$uri&1M>+uGett*vfv+OXa2jqBTAZ|kfN>o6}Qy9O)pb+Mq52@nN1ytvYn'
            'o?n=fmmxo}H#tf2>M0z)ockO6E+rO_jP->fd3`o0=AJM1?%Y_6T(gGVB|35EE@MAyj9rVqSRgB8yzvE6LF?`L;DRT?;zZYPc`c=j'
            'ZXE+E+@hh)z1y4Zhufu*6zC2t?A%z19a#KAGP<;$^9Z^PNek|U<(4303NiClVIh{*x8GJG6tv-&T1eRCF<^W}2GXpG>R5RfH7j4|'
            '3~+}yYTqy)cUz(AvFlARIVPI4da!r*$<}})4p7k2zssqi?~-l|zg<>M<{6=L4sZj^;{j|@=ekR$=NeCsyw}Klo~uy*I*s5T_a+j1'
            '--zDV`v;(%DrMjzOhP&1&OK6gE8b}pU<SLrZ?v<S2a?k-Z^rMA-@G_}C)RMPL*VJY?y%V?Na#DxGj<LDE@ZV<BC+J9ZGZ7fGIzqi'
            'DmmH^=(7-W7S6~$1i0W;I#ca@V$iv<DfZ`8OWoNAd0n-P<0x0!60*N`s8f%g;Lk3~_XmT)z{VCJG?iXpI5?bbhqtCuz<5MZ=x_g)'
            'KR6sb#I$8*Fv1iNa*R59KxGQ_%sc9b^dCY-^MWN5%Dt(Ac^@6pidJp3Xi@<<NBu*P0bTKsk?=j#X?oHsAv$xymmY`+`I@3|HXtbX'
            '#VLAS%OK8VBP2LM9=<XPUXO;o_g7_HL~GJwuR?TJL)ieZ7iATgpcaMkk4nczmm5X&3q52vXc}Ie^QbSBUykYcM^2EKA#rb~e~lHc'
            'EolfZrri(6s&!Mmazh>h|8aqqQ{wXJAu(r|MiR$2q=9U^yGKH4Rb?=EpWNGx2?5ZtL2&V8RNn#Iv_}GhtI1-P$(7Il8OP+?caClB'
            '24m{1mQ_O9vwf#|TlwpH(a<qr&zMwRt}xS+Y&x)fy;P<^nyTyM1%ps4jPJYk*EIU?nA@Nx;yZERE~@$coo&e_OhNsjb?4bq7NE_@'
            'f&Rc;mIkjfgw2A1xu!4<Ycszk==JW62xn}S#y9<;3f(;6u~Q6Z&T}xYzULZ5bGu#0qgL@g$E9iXO{b-K_>#;bXh|nIWe?me9eijL'
            'MXm0vYy@l<wrET=``{x)M9MIMz0D}!U`ba6j&pQia6JS-O^fuvvk!)z-g-UXziqtv0JcVj?iQ}wAH<G^v@qT6e#=Lw4U@o@7Rm#('
            '(lp3KN7$?5{g%j;rcT|uIZHV_sQmMjkWzo>OdOpmP=xt^*2yUyjgzC+xU0r%jLR#->dv|=Q-i#V#u(-olMX`zR+S&(AX${bLcXxV'
            'I_`oVB~gLNlsL7%Ipk$pFukf$?dtpVOfO3=npT4Kc+O=?^r)wITBbyZ77508U}6Eh{wwn@oh5OS(2c7iWjj(YEmXCY>4s;Bf<}o6'
            '&~Xf~pT)G_brRXo-9jO4);7_7txV+lmhKrUGdPU72+)Hvm)@Il#$L|4k6Fcz1N%p@4V7XXj+As7$&@WX9yB@AF^4%DDOW6CVf0qu'
            'l^m;;lxcL{^PfK>VpO@|2v1<Y%(tC75@9;N>u4H|yy?26?8>Wki3bEkppge_^3=1gVPZzcnA6l3W721i_*4DrjMyO#NGY0-1CASI'
            'nAMn|vPoVCGZ}>15dVO@Ex8_#sSpH=JuP$Ez>!pBs8cXfKQJ&fSkY`pPG%Mx?MEH^Z&dw~FSYNu=f36Cgkli`E}+{G5EzoOf?Qmx'
            '19NF*N19`em|2Bn6)@+U%1j@ScQCmaq|<4cQ7k<filT&__b7tYxwO8~=ao_dWMMB#{z?M7SW^@fx=TEBzsZuO?LDmEj5-=(eL3o='
            'fftOyA@-f?u5Vq`d7n5#Lz!bD*oDi_)_LJp?--lpOwyc5_!X8ALC0t~sI#&jrE;g}(Lln<sAGMvN8}9Kd-qdIx+(JP(J!Nqft#05'
            'bSN)e_nA$tv~l37i~;?lIG6@kePf<2dRemBx1>aCSFBX0npMH$Pt*lBQiYZ;)RZ{($LqQxS3yJ>S}cF}-G;%&Pa}Yu#L}_v5t)Bn'
            'iZJ-+x!2+1HR!aHL}HU(bq(o4@`Y%X7>Oqq__4fo;84-m&ZA)j9CX#=g84W+I~fMobq2oX`H`WiT6T@NG(=oVPd#dy<^wI^7(;Lb'
            '9VxC_D^?x*2-tf45q<bOIP`<A^R9*IwWEr0j((L=yW&;N5$Y}%?y=^J)@wW}sCD>eD2`Y<d#|omEO~|zmsrY#-2s{SlBX)RNYXM9'
            'WnNMu>;+~MxyjOGfv`IJs_xJ}vf%sg!X$#<s6Z&Qc`)U~A%csAHD{cjz_97M=mS86c^U%U3S+okao*5k87-E5JzlQc5*p@55Q2My'
            '{!@yWmD#aim<7V3fyyjQs5+Py^nDoyF4HWb$6BZ(FexuLd3LeWHjf>Fv8@alt0{TVl*v2}7K?D!r%VO-2m2+=vfw6{xlOQhm;v7u'
            'ZbF%JWr1<42r!X#?BFp5y8r{#BZ!oQAjNM5aPeZe6Y|95aW`vdgP@R0TvJpOU#5ChrGcMx>;1jgC-2A4Pu{$I_2T%=^W!ll_s5j~'
            '$EQD?ynaE3@b4cJlCb^<ePvUia|{O#3Ppv9HRky$=?Ebkg<cStz9x3wb!puZnpBDtb4gCD<S?f)#mtC#M%)WIkyli3F2|t1$3uL3'
            '^zQ7Gy{(XILwjfD{rApq@4xMK`rrPsOTYdf50!-dU*5cWe)8fNgaY{E#mU*x>(@%Q8Oslk`7FV}J2XA%Wv~KK3O>(v-O7VT%K%5~'
            'HTf+YvyoXV2nvR!hZuvcreG|E<CWZz9prdsa9(~6WZrd5OcvzG&5*8~bL=G5TH&ytNnh~Lb$=ueT)?9Ve0Aj;9F}pEJ^a8rDgsPR'
            'E7kAHou^RsQOr4_H!wO63*?k%wuM~z!5RD1r)MNK(=@wQa`&1q9cMh`^{7mJh-wyYkv6L4wq;I#q&AT5L+skeg+z9=(=|$ySPN!G'
            'Qe&69M^-6;ag5tr1K{?ShPnkrB%3Z{U4UpauIjpttF_HLQs6{|2I(iN-&LaqRD2nIz$3p5G4&<i&hKBH{dDrn84S3T3H+B~l%X~j'
            '{>V))4wa(`NMNXHV;G+!cA6wt4yv_y`z#E5m9n*RGVT#>BBhSW2{DrSHWLbTKo$?*)2;ac`6d#DL^>hg=~?BaP8p46LbH|VFN1mv'
            'IU!LkBeIrj3_Fxm96g$D;NPj-rvM~Sj@`xdZ7SOWBDBu)YU-#LWk_;1eh@d5{AOUxO4eq-Y^GaRVYtp^&Oj~GnIZhNE+pt@Rq{if'
            '!a-93x#(zm#4XgD9x`L@6Y4~v_M5wue_yJrPjAAKN}qvXK(dc<c;RZX2^Pj|r~)zS{&Dj*kS>G44Vo|QV2YfiOAOXQlsS4gXby{n'
            '@v$a`A*Xm@=-YIzYa|*S?gMQ!Z;Pnmbv3`~*Z^dzw$|QQsi>_l+S2*sWI!WGuG<_;N@>|T+}M{bZPG?9YXk(3b4N>+Fm0b75o4*I'
            'sK{_h!O*FQzhscGp_(Ykkk&W|HSz0PiLNEfr%gZxR{yclRjZW@oV0gSsr9Y-7;Z9{{CC79Yi!NFbKl{GQNN3^yWBD4l&Q9XzLmRn'
            '{V0axntZab^(w;8xWKzz?Hp|^kpmN#@*F{5kYND-)EA5|1`Xy0n1)i;qLl}y-q1iVblnHY*#m`q*stsEQr(;;wcI9xHE+?>j3cLE'
            '=F084VJEsZQ<XDOE!Q-&S=CNrXfA~rr6l6;m%9zKH;oI%<Rukk{R@$Xp~h!C?CPMP5g+#p4HDJ9)s6&i;&!i=wYc?+S*e=uOTrZg'
            'YOn#apb;h*07<DZbj^5sP#t9u-cVRj#hQRO3yd4kS86JGue@4G`W*)!BU!UZs*!sI-1X407;1-9>CX&w3*&k;rzrKY`j%bN`g1Z$'
            'iaShFH8_m<QMKcm0=%M}^0b5aq9^GyZ(Spj+OMRp3GVB{+Hy(of*q?rTtNCoUyf{y^{vp9(J@NvdYD#9F8kiSa%PAgI@(_0%}-@-'
            'S>26cqrC|*VajpboswzhU!_z>+rR)dhITISTVG-!izs)cxYn7jl{#2{6FzyR|H#9b?_pWq;J#2>%rd^CVjZ(r@ja3x3g^TlYrq-3'
            'V2d;P3nl2hdUH>R;Vh#14Xn!?N3&1suC(2%yxmCW3$y&c&$NWkA(hpsfs=aj=2)&c=G0O-qe%TH&+7|Q$$_!56qP<hRRuvo1%Whf'
            '++-;w(m>W7y#}zXy~@z7q8}6=6H_&Hkp~+x#E`j3`}tF9bIkjS=9vgnQCAtVNOwV!-kw@RY{I-Ai4F7U0|)LABf#?KG+iWEXIjf6'
            'jtliY2_(90?Glu1U8FA+U8a;I^YTdbR$$7k6ml6+$V-+^0!%DgY}ifv5?SPetqo`!FlHnH0uIbE10*UefcC+;$0$m=zr-vpCj}NK'
            '2s4rx-mo@64Eh6hZO4kC3(oQ_!6^V(%tb~+W>+!~xLzBcL!939s;`q83#;n&cpDTvR`{#}SyRryLJFt^Zs_#5FY9%XHa;%VK10M6'
            '$Jp0(F?eAjW++(qe8X&jv24HzFfUkR1e*^RTn5HP&;-xNScO75e|1~HaO8J!H!LJPQUeU@H#T<9)Z*SMZS0M~=TI3H6j8~UQNLqd'
            'uW1^yzNy<;1KHaCFzt)=`Rns(OJa4~wYh%~JGF#+mEKwUXKBWd`0N_rSS^4<3v6ku(UYs`3)AK=OzkfqeXu|9jX1LimP~cX@xFeQ'
            'i}3{J%xlXg?NFu?)CFZEG>WI95MybZ)jS12zJwd>Qb$>V;LX~CER;6?l<M3hrQ~3~VQP!Y(Mc$am>UTcQHoEnz#|p{VQLPKi27q3'
            'Pk?GhRL&dpHw$i%q~@7}VDLmTVH^Q!)Hqe;zRy{JYdhvfFuaGWtb0t2jWDGTvx92`xeH~iFW1F)ou4?nKaQ9~(-6(V+LC*MVQNOo'
            'Sfm;kgan}OY*6)Z3Nx9jr;-y<ASlM$pf;(Wq&Jl0=p(ZBWijpcqdaAUpEXcKWz2G4k`!qL#4z=NQ57kb%p;A1W~EFLR4;X#PnY2;'
            'FbdS0CD|2<>v-K7SL>O%65h}e!n`^?X3UW_G-VjWs4@vTz)R*J<1Bt`$JjG{2&f8+v}aU<E`p+@a`w!SjWOK<rmHfmkaMG|>~6;c'
            'zU;4@%M(laCZ_2eHl~218lek89-s+i8e-ixv`tgmam+R0?O-V3+-c+`lnc!`mij;5q?8OJuzFW%QsygIDaxUS)FDO_^D|)RiVcW&'
            'a(FPFkLJzf0SO(KYpxXwGqk-*cA(;Jq&LI}XkZEzwKlGFQMGTEJ3ix3HOL$u3mu(?SLO;yr3C8*(DduxN;Y|GX-e0N(Gr7YV6@?M'
            'sfOVy((uep+ioiIt~5E33lq3x8CjoVvmVlOJug=si-YTm2Q*^)p)VeF?<WFr1}1|e)ae2&QfV<*V|9VpkI#pX2N&&~%=NWjSvyfN'
            '`p{O&41WFYa>b3>Rpp2uwVMh5V=jG;#rEJbwdk&-qnPr5No>Pg+#LD#ymxSMUcI2ibc#09x(>{#)@4^ZuvCaX2MnpdSC^SddEeKW'
            'gFsaPvIRG39ye$*vmpjBjGg5itT9|t*-%ClV?ZyE=!l}mF=Zo{gThxgC{Us0Wz8-C1~8d-SxTK5Q~OyoLlM}Z=&lS7i{L&YEx#RJ'
            'ibK*UHSqaCuIR)0g?=K(#t*9?Qu9co{HXOwl&L}8VMZ*WdNiJFsshydAzkcuY95ek6LPxUl@3djQcfsk6?=Cac4r$92cxMfjGlMy'
            '<E?uyn(zf4z4!I&-TM^n)@S(seu)P#TkWLgLpyScbzI{4n{lQHqUA{A<u+)WBkNqzzF3vEU>P6lnUSe9Z#JcpP4dM00Q)-@6>Ahw'
            'IsqAAk5w#;42Bex&ngjRRArwZofpGa*!&0b#r>xT?tgp_+>4r%qVY&Zmod&eKDSOmbor_t%c4RO={-C7cD19Q{Ss5$+(Cl0Kd|i5'
            '5HDsKT2;GdgR2`Tr|g{6y~X0)5IHC07kPH3TMRu7Wj04E2iKjstoaa<OEf#mN}ZGbPS-ad;o3Eg5&QmO#E_c5E+emK=-;8{^=Z8r'
            'xoJkdZG@utTQbx<@`u${^K)|Qe%P|h77ytv3Aw*9W8*b0H9G7fe_jW@?fyHuhf90%XSs*F!)%EDlj;U!#H6dL<4`W)8&0cES4;oz'
            '_9nNbFZ4;rn|2jliK{S%b-cqTeO7)Hr|-_ho$lx|0!BHyw1!z(tJTW8HLSjkH}xxbv`d~g`6}ZsbFM7=&x&0dsUJ<my0+uI2Ff4~'
            'ciBC|#bpd8mf~_E$9*G)+m0aT`0ehUk>qvQ98_KlD{ts)#g?lM82IfAXR!4S2daJLRY-%}4sYcrWiUCyqo4g-6DU3M%(dj&*B#B^'
            '+iR%Of1__E#`L<it^eQu^S?af;VnMvm`~8R(d(K|e#4Z|p3nh=<WZx<50=7B`?eX&E+0!NqW$Kq#l1fq>`K>l*nip%Ym$Y*)p!xC'
            '-3(w)9#^sS)j+F=`aj{kv^|yYV^T)u%QT}5dz>(Dy@Y$R%CoBkfN6kNkYcjgi4{*oma*?Do{t7kD~E)lnuxo&r9yyHrKD=E#pK-b'
            '(xLDIbOAyQ6a&IIWeT}ajT}Q>EgXg0a=@zS$%hiy`QV}odbPm7iaQWgvl<N2mN0y$VuR$s;a!fIL<(uvvpUK|0hy}cfsN6jvD5MT'
            'h^fEBfBg~vHN<~C^BjvKXLcVGo><j}HJwC7Ab+td!^t}bOoB90cN<L<)YIB2%$|e&pH}?|(VP40t@uG^%y_I=DEBZl_s2c>L1o)G'
            'P<^cv(cH0l<*k;r+9>8a))_ZG09AN1FK0t=e04tdoQL%s0!^o5$A=^_<Q7(>ISb;fZv~>-M8Eh_##(f7uo+`o)enj~VKv@eVu0?k'
            '(LH{Dh{;cYjUJ;QrqvHG!oLQ;mf{q6g`LU~>d6Wf2KfCU63A>3CF5)L?C@~ss2T`zBjMcvLN41hje(_sb<^|sm!I$|eF^{UoCcLK'
            'Z0`N?<(<7t*y0S``d9(4@-OY`5{Ct9Hu5$3dz-gGL$=5Fh#KPP^b~EmD1EYoOkOi2BQtKnU-_DXW|)TOViU$x9Et!P2bg;o0iYLa'
            '7+^swkhSO%u`UZzka8WlTo#F^6!^G4Afd~$%F5{+VUSe8F8;xFih5z}wy(|8d}jgK-BjA8aJDbP`5dHnE{!*@bXiwH7Qj425iD{V'
            '%z~9v;(4%sUKTw{roT!l+(l*@rx94%BtBeN(W#9sFj#?+{$-%vs3MA{p8x}Pb3z;SgS{7$kH;O0V1E@u|1m(npjoh5AWy$ZuT%o)'
            'C71Ft-9)_eQ&x$;O6KW>X-nkim_BPLt{?RWgYS&DbMGyj@pobV--d$n098byyKQRs7q{KUY6oA#g5W>G(uCPjGCN%cv-D;Nm}Sxh'
            '?|l}mPcebdP#l`i&EMz0KKf2jquiR$OR_Hy#T%eMzIX`)CDFfFWA`?TfTV1m6aM+)lsHi1!AV&lf~r3&!Oi;-#1{4V$20~B`V~eW'
            'O?~kTvccF3;K6-;6`r6g2F>=r$^ZTZ{>$gn7e|<P-UdxeLi5ohfDW`i&|XP`zWAA_;cNBdo%->sFOFp@3~OWLIvx>XzlX2*x0*h0'
            'ecCKdb-VDurd{yBC5BtyGr%)N#DDyM`RV(o@@MbBAJG4STtNT3<ty5zQ{CZWi*5YJ@V`R~EL=SB|D26Vl7AT7K`SyG6D4azgxA3a'
            'C6rTEgdC&ZLoPF;$gTw?g5rxaa=$Pfh9u#%+|`*dpgi=!=RfSYS3c-nyOx}6Zrsxiog77(%7Md6YIJSkSfe?A1|9sxABcHt0?WxS'
            'J``h`+}QpKbvV~x+L&V?OmlM~+)w78{LTE_0e8{Pn%g-Icfhx@#VcCuHZyYh)qyIH5I|^A+@!X21Xmk*gH*A;a$^Daui+|hfVP6W'
            'jO)^`MNSu5ym=;&-{%;(zoL)3QPd7c%UIX|4CM0kT3)vE02v;2b@sUU%ye3=%9vU3JfyH#0dy^x^!J*Tl30+g0o|nR#L7y)zJxt|'
            'aqyYy&_-o!^A?+W+4ccC`a4c$J#Nspo3g7q(Dyr<IthTI$ksWJ`o!FIsDy#$^VoJ?Yg^Ys2XjdyMwX65f3b0;t65+&Q2Nol;P#Rz'
            'Nu!nPoO`#ozx_6?-w*%t+i#PD?e<m=ckA5=#JP1Oh~|FvSvF%u`<(R19Qr}1>na{>5Dh)rmukNMA`0SkfhK0S&0HNJ84tF#=V(FL'
            '`uh#gj7wbJul{W{=<R-kAbVFDEy*i>C#QmEg*9RIyjsm(c9I`9oL?lRM}q^6&3q@nVqDYu6excq{Qx1@41+dQIedieNWe}8oK0LO'
            'RnwY74D!ZMnmC&DxuvuzQLrf04Q7$p%O+lz%6-{awoYk*#<xx!tR}Mn#>GW!pFDp2@CkY<dv2}NrOHIk*)q)4T=tjY2N^}xJs%!C'
            'xlrZgWr7!(DU}t@Xj+spIQbaM32~f1W(tkjA&6h<$J9$^PR6N?@uJD#&nEAHw-F}&XWthpVTOvqFuY<Fp(?6{$CYfUL#k*lGtvtM'
            '*w-nB(c{SYw9gNNB#1Y8l#daF`GOL^>R6MTWKXNh!W(L5l;xOWgmfZ%8;OfoRO=DPtP=OpZ>3f8W!hhM5yI>ax-iE}nD3jkc!~OV'
            'U5qgy3!+RWpmbh2yL$?m9e5VyR~&wUMP&J^#&G303^Hc!(u9<#A!jINhwMOxA)7@SnVSqISu~|;bCsQ6t6b%NVlgR1Wd|9gr$75g'
            'b~$`l`B57ifp=2{hdbFV<bkhmRa4h`tEw203`3x#k`{uRLmgM)3Nnx_ncbt8Y^BD^I_>eD->fxs0Nn9$r<v?brJ;jK7^@1*Kq4vu'
            'ykVk#wKy0W;m*ldYhX~zrlI@wc@sEyl)gJX3HM^?nmY|Idzy(8x6x#*HS`tpeUPZ5I6_rRAiP^zldXr5T2UUvaP$^YJQTMC`ghs3'
            'Z{lztt60K4@UgrS$dCe@tiNw5Ogd_JelXT=hyG-~Z!av`Jo(qQD{7m9&eNS+{r4723vOUAG-`JwF24NGuoXK;<^t&4I#1yH^&RIi'
            ')}GhLx@TnuXu1MOA4WVvdz)@mo5Rc)-%foi3ef<KDxlv;i#Cdj4q?&*vY&s>Al$ph`EUbi(;u(D`<wwZ-E%grG=|eiYRtRO>l(Xd'
            'H$Jx;UVPS;?FLDHAoRd%C?LSNUAsVSme_k!=LeEzZXm%rJIGt4N%Rq;1H=_DY!ThGg{m7o!LWwDu%!fV2eRYA$gjdAN+N+drbMs_'
            'K;-UwhPl7Wuj4Gxi1Eir_R+>-C@&s7l$wfT%4_w|C;_3_5Gwqj!#J*jO%hzmdXnx@0OD!v<yM?aZw!NM^)!vkFU+&nFew`!?>D^~'
            '?WtWZFx^_CmTB8a0z?r86n&H%-f6;)j{%av+Zr1}O^i!@n|m{8cok*qC|*T5teh<)f%s{!q%w;)P}g3KHmjwrq%HeU>{>yp67czJ'
            'HuvghqhlUewMf7_;sLAH01)zuIr_hx{Mkh%&no!$xg54R)`ol$WsbaXr+U&=<-e*7Wu_ND>c+^ixJvmzwkj@)C&QzatGPJ)>FA7O'
            '%XuzG3AzZYx2Y`TYQo1%Qi~+TI7QR>w8(&w$1pDnjRaX)Tb8PXcyOe{HRquU3Y~@sB*E!Y>fbo-tA}n1Qbj0p#7SO$6;9F_#xz{Z'
            'o}6E4cNPjWbw+a@iYm>Q;#I0XvK);a70|UZ4e9uRU654*RutM@=p4pEK;2gqztUA+Hd&d}6H34`b&NWg&El>bjisJv>^C@`MMj<p'
            'ow4~IIk|OP+d-eI5*4Y}0CZPGf!g;mV8f~%)IEfBU%O7rloGH<3pkc}TUH0{nn##;di?tM`PuP{wqy<@!0=&W<Zei{M(8e6Z(U&;'
            '0$5~NeWuNMh^YxKjZ6}K@eAxl3<!9m69*Uv?}js<$*f{Y9?8gmdB~C3k?9|E%tuG**km5as$7+f`q4?`%v_&sSv}<mt+E-+G1>qh'
            'oRaxkRgR}qk9FBuAEIatmjmS7nK~KTD<|3Kv@CK%pTRI`AC7)LW&&)?l}cSJzx2Y$(QbF+s01pICtVj|@+x_TPSUA}VXhLQ87=5*'
            'H&Gl@a!*uRa4$}WF^lH&kQ`O4!zc-&Tr5frfe_(1=T%im&R)mJ78IFE=JT8~xhIsR4mT{TB}>k*`edA9(vXc!`8bgq=O!jhm*IT!'
            'w^3|~v`6DNWgNeiuccPcPQAnDOR*GEL!rYtWwz9j)2y!xeYzy4V}y&?Z~PkB!W)z#dB^garZ^2PPr9SnB{q6LGfDw_5UnEEhAD<P'
            'f5eK=4wR}Uv|{ct*p*^V?D0yiiY)?goUC*ll~b8)6o^|TCs3;k6&_1uME-<Z&s~Od&v@cpZ0-500n&Y!xF6lQCZu!-DW~$)=+*+a'
            't!o(2j{>O8Ce^;p&fN7H>93z#FlC-Q<FZlJF^xxEw@rU_1^@0~QUrjk3bLarahzTm2*HSt;RTD3gh1%QNwGl;)4QKXz&Cu1GW65N'
            '7;cI2pP4V*;`Z+5@#KwVv!%@n>(;BhrnBCqtv1vd?U%X2oYP-^JbQKa`gr{Qr&p(MkKc`dK6!Ec`g8{dIdZ`a<LEIueid#2t8{G('
            'WznTiQW5s{!Tc`UYO~3|l?ZUAc?xM9tnnHY?Ly3fj(F+E=?&TpE6KDDOmP%Lv$`OMF!6_)^~45Ot^?=Y6qDBHW!ZS#WNR+^Yckx)'
            '6gm5IAj_P<F!RoX-p7mE!>#8x%){a~>mR9iXw^g*8@pd$V_qw99c}^<UEwLp<}oT4K=1~3M5Hj>I8q8UV%&77uIZ9Y89J(C)Ae~z'
            'YbWEbU@Gp`r2jF}QK)^9hG96^WS^-HoyBN&tiODF5?!h~irr;Sd+WAW0d9;qvY|Vis$D%NDU%b!USLwm2g1ZUY}H?_A9*I(!zkj8'
            'Cg_qL1G^c;2AM;_feD#at-x@3>*cv8cSMshv?=x?`Pkk%uUDAO6{Rz6mAzH4MhZL_UJnQOqk0ew2Y)oZk9yPLHNF9k@WbJgN7R_o'
            'YEvE;%%Ou^5Ivq|bNuh&Q*IP}7|&PqW>a22IvlJmu)&nta>rqS{|$nvnvhmjO4Yt0Bk<b#GWx{yu^!p;-D|jkByEGw%-u+(T?BB8'
            'nczTft#<7j@h2(6$e{a0isbso<CiDzj#Vr_F(l%-%#<J#CUbft5zL|A4SLzAtY)ML;Vk+A;FsogC9f>A$Sp983zO(Sqd0#Bbrxdj'
            '*-SCY-*ET5NY^SQQYB4HQ_Mh(17S!irgR=<dBGW#`wcv&657y|tONgHaAh3sDL)q`EY910kZ0<|cGNqcD;ugvrltc~`H*wpb2FL1'
            'g-K$o(>C9~ZEU@T3;APQUg2&buStuF_TNPutk6P}p_UTf#j{8nN36KEwm7j`bbqC#uGD<@CW-arNA03{)uth|YzHn^{d9a;?T)Gi'
            'nl){YzgLSE)=RIRco%YJjfWBaIPl<~RRNHgO2?#*D!q}8fZUO^vpFLn@7ze`j@>b7R|p~w++vpfRyj##gcvyzm&vWzxg$C*gzlGb'
            'XhWi%k)zczPFK~hmR<AKPTSFVqIR(Yc9IXPyL@_|?o~z*2ZDOB8;HWeabB)D53kI>B*S;DybAiX_Dl5Yzfx7+Y;i_LuAZC{M`i@9'
            '2DzEBy-!P|bL^g*DUL3xOt<dy61H#Tu~U3gV8sfqKg3)la?LQd(xu)P`@4cu%<hfJe*n;|%z;`i%Igr;V}Ey6tqPivb?Jh28J<d<'
            'MSlI3yO8Q;Xi$|VPt`eBSg&d;f4(-=Y)@3V{U|Ft0f~O7eO15l%xN{|B*yTP85N#dP^hc1XPlv$3z%C`Na_ujFt@^u_N=hF@J4=4'
            'hR(2vtrui2qX2X)(V+3pumg^MKk)CT>-`*cbG^@9YG=jS=j_?98X2Kp9H`UKrAY^^Upir1EeUC?*40wp(YiUJ>W)rLVN`c?ae}|>'
            ')>bPc9PCb*COSAEYxgQ2oHm&E)ALhy<MGEdT~(o62YmxoxxqZ0mK3g6g?l~rt#=#yopDrlS`OZ^nwT{H@*Y>5oT<&m+CV4oUj6CS'
            'o1@p#hJdPRodcl3fB@WjnnWNK2n-$XDf&cFh@?5xQCl<VIZIbej)|h#)eg&cQIQ)sJ?J3t`t{GqE1jL4Y7H}&v2sUFsb8=t5%BNC'
            'vinN9PiQ8T#+W7%<p#S-IZn0u$4HpOsbbKJ{yG$fRYEJoNw%e-IEEJ3=n%MP4xg89`nA_zRTKV?pbF>h_$-C^{{0o=`~OK4;*L+n'
            'XKTk@hl%~4!)rW2*b}5bTw>1X=M1fDc%y%jf?cx{TEDKJbe$GoMZ5l|2q;ht)Ncl-r~DhJ)<4CO;@@4#ZXUvRz2}~(xBNr}=%=VU'
            '@6k)<z_oiw6{*GbU^HqO7oVpuwZK)H*QO7+s?~oid&M)yn{MuLj$o`Us7`vzSi4)-8>-UR*H!Re!~D?zwcQTle)?}=5NVob{rjeg'
            'q>=F-Ypt2%KdF7;nudvMQ~wwXlWMPVZN}V)D(rrmnWi?-CVNfO8tedZpH=WwfQfe>dylnb_c#7b;D%9$#W4ic?!Zo(CE?W&Gpga0'
            'QNaCx39W(XC!^ci7vWgBci|3ui$i5T#qMJ5gO=tDI#oTxI41i;=bnD+T{M7_#f~u|yCwp*gQFDK7tYX3VbB<zD<d{SbsM>3tz<?6'
            'vMoq9!nK>YP_<a;tU&d=38sEqTLCSAHo)WOZSd%A13VT{YvAPLW@!Af2CRxp9mx#fO(<E#?a88|T?dREqo?C@8-Y1tem*5Gv2R^M'
            'CrlWM2PLFJ&-M*YEjmqCvZ<6p`y(lVyvm=zQXEya#xVerM3g)%eD&uvo9o{>rgcF21mDO4Y0txyULb2XP^wP=n3MWO96a?=?l;+7'
            'QjN)cafh62$yAz&Z!K6|@!)|tgolR~ugqGsMkw-CJD9yH!HsF*wH8>v{-<e|O}m4;O}pQSQ#b-)ie)7p_YaiTA`5%G6O{1$=;*mD'
            '+L}inD5*{YI3^|_ki9-RJw7$-OBA_?gAH0M(LtUi0aYQ3(gz?ghOM*XE(89RP`$m}07nWR@0Uay1=QEoNA{oS@$7qjafDXEf;2ts'
            'Q;ErZ40x1XPE_UeP{~5`*P0?W|Ei*eR&6i-q?y;9iqc>uDyduHd129;I0`1Z&}-grGNoHv>zLGSO#dRgx;{9V_t_VGE1?>k(-*Kf'
            'iN!I3PK%)FI|ol@gA8)V06?Kfn~qc@vDY;$$CQhmbS0Y%4L!~*%xe}U)$mloD13oO4YTr_?&S4vx*VCEzEz!g8P{?JGbm6+2S{-h'
            'arUBTLPfvy=((&9g0^sWBM4T($G#%lRu)m@lt0y0+@J~2ic3U+6Q`nz%BCvErTrO>5^?nQ?Yom-k6&26%~3*1OWVnrRB)tLMJ9}#'
            'fR%%<1V=AOB=fbuv5X4ff>+T8S!0d)j+WYWzACCl)s}i%oryOwW&2`pU0ny+C^~JI(8l7^y`BMW4G$1$0>P;}HPg&xq_C8nj$qLF'
            '|BZzi)m7$*R1E<lwRYXAc!x48QVF=RE^mT*Zu?bb=F{(@dDFlu`~}R?%pTPnYG_@Haxg2QApn_vcz1seg>bt!j~y$cSYimVW>qy{'
            '-HX#4YgAT6sAT>UJ-LSOnakMxIcRaOugTKAY9YGy+~zc%#)0jLmJHwbnCpd`{`nk7(lDa7EaG=h@S-lUR^s}6U+s0zz^vO=1Tf?D'
            'y)wB<;0E)y``s2Fos%TkfH-O-zE)Y9k$V!ITTHRRyL*>Od%jPN+fmb~Q=>2xt=k{NxLAeVeJe+H9fRCt*_NkKz4;dCUJ+%oLKP<L'
            '%c4yjC}KOqpo<6|8*WB6Z*9A-)Hdec-bZ!jwwAE^KCVSy;*HYmVpP*A?5xSK+wVvvJBTa1CT9=iXUJig&Uyn`P=y++GhH2<T)%At'
            'yqiOC?^|{M<H6N6!e&ulU%dYsW~pVTW7@I`1G1@MCJq)aRL#@`SWNP<V56L}J*tbhz@QkGW2mAm)+!{JF`C#oi3b+NU{+R;B~0?R'
            '&=gtl>K2DvJDP`B_?VmsgG5H{7?Fe1WjMW}qA8Rx1g)7lT&;QP>uMGiJlsKCG73JVna!vO>b_%HeEa&BQ?gMSR@i;|GS|cn;<<!K'
            'kYr^g)C^yv$&zpFB$d7>W5b7Br@`0TY;$0DZT_C^77=6#D+TmrPMG9=c-Y`kp}RWD(I4+qb&mJz==IrOj{f!4>8pR$)$wX!<fni!'
            '*8%W3047g??1^X^X21Xft13FH@?&)MAh<%`)&w?gnP4=A%2?5@K|5k>mU>1d5fw<bk>W}F<?)AgoMvJ6n>P#e9PqPn9vpxO286l|'
            'a$#nAev{O?Yu2?k0d^-Vn&tP(f)7pj!<;+{{_NN^wt~{FdvbAxX{y&|{A7PE#YgXS*z#_5zpbVSbX7!{rNgH4+wL7z3=u!;IJTot'
            '%@d0vw9UGOh*cEKO7VzN;vFxtV7=6?m-;c+=b@LKJv&30GzFSU9^cm1_yYYwSRXU097GJC5cUQcS-v5@e7vz9Z(K~k1Z2B0LEV{s'
            ')Q++zhJA^ughb|nzWXN7Td0<7|C|OoyGVeuIll%m|6;%1nu~~{rVP_FRokw{M4YqE4XiRKSJn>8g^I=Dy89GYfwS}0w@kV@+xA|q'
            'bc^4x=Jo-o66}sly`GVtdo!|Bb^cy0fi~*!M|DGhU!~U`)ph$e9Hkr{)l|%8n&Y~D+=ZfYAJ;YCc1u^SKWa1<_<QYHwnq&n0K2``'
            'SdZ$eZioG{)X}3Bm3CuAr*gu(Dswa9)ycp3DRQ4o20C?2=<b=@m`kUJx&oXzblt{l=r>N=vTF>f4Kx{fw=E;nP`X=hAJ1_$9Hxj~'
            '^UUp&Jl97VcgMS}EzQ=ub;r9M=F4D>YmP>`dmMjFjdq@Xbp2?`&+$2%WnXt1jHD#)a(kF|`V>#xZOsRD>sy(qM^$=w-%3F}GIA`t'
            ')_Egs^ys1LHahfW3S;A|>ilj$ar5-do3{|qi4$ny2xJTTW9!{x`!suTPvXtHF~%x;G3GPHvKP`P#lC!fXG#}JZAc1BjwNco@wf%0'
            'v9DP#q@IwqvlI;VBDxAO!~84@{wluH%s~-ep%)r(cp;@5GYmh#rAX0<{qT@m$Iu{309vKYqnewT=anjH<G(uhHjYFsx@&l4FCb3A'
            'deC4H;a#j*MDA@6ajag-CW7OuEMVPFiF|8Q*TVqQ>v<e3NPqhs8euq1G=HtO^<S*amxzO<WRvMeW(+hpOyzu{&1?$gk?xez_3U|c'
            'RS@nZ(Upp=YVbmnpRM|$o2K&1<d+o{YPBTPmSZ>fS-E!t5wR8q9g}NCjIh{&ZEQp9p)U>|_f?JHmdTu}Md7Nm7wAHBuY&Sz;ABO0'
            '1XQc3&i+KTE^8`GsY5Z)76yOr{(!2+eqmh#p4?54A&M-G(XViUYPDxFwHGEek~E{SI=fw){T`#;vm``va+w9o0HpPgaiDOEzWo5L'
            'EOK0>Q4%hVXQj7iBtx@NR;e}+VWpqAdeoSXKxG&j>msD6m%ZIlZ`~+P+~SqXJ1Xs1s<l-Dd%7eqb7P&xa+!v^mQW=UW(yPUx;^Z3'
            'a-KnbGBeaEB?4}kVjqiB<(%gm#m@BV!nl8U0$&Fqom%P<tG%o9U)5=p2?eWKj*e--*1luafGXou3jilGs8J!It<<}9I;xuQ^;xLG'
            '?+tIP>hCJUc}tadgVZz>c^?yPY8jI`>OY`Q$75P9^RZktXMB!E;cL#LB?-4Y5km{VeD}Q~QA3AI?NoUOnI-^_F)N>mqFmWZ7r2t5'
            'nvQ;fN$3FYeMdzlyhe%^CN@nIcjyXis<vaiAg)j0!kidtz(1e7`Q`NIBg}NCRDW_lNa$Q`k*9QlzPSI$oAs6rgBHz$t}XyrTqevw'
            'vKQ&9jByKg4WwHu^KtLX!5WWcZkP4F)lDJUk5zys7>uhRg;ym(7&r!G4&-Sd1T!5mYP^jS;4=_IojUh!e4#h2dFL9uTKlD8^lg0S'
            '0%(lv7>bs#Wizp;<G)WAitW+g*v;^U$M4i0R=u{yvg!PMwS}wh8hF=m139Cp-DXHE+-J>WAey<Tk3bnv)G~N2QK)XoH_uRcR@ZUE'
            '%2irUo%p|c)NO1m)t*j!`E*y8O=Fjh+r@MhgdXOfPD@jJhozKFWnN=TX0)rOP3k?*WYMd&Yl3RA?^V-As5|VEh^zOtT6%RKZteL^'
            'BWU#kzXu#v!Ku@B{aH|?xqaB`Y~BW;Awz&FS8+#<06sycx*6m0&A(gPDhJkT{hFgqs#T@uZg77Mo?gc`aB_9;4UGw(W-WGrwOa0m'
            'EWLx#c9KUHT<yQ?HfaM`xwQ`)yyJIulK6j@uH5b<FQQ#qTba#!=cg5>w!u-JBP~E`Zt_EM4~@Fl>@aV3Zd0y%HJ5m^SBrhXtNFdV'
            'of=F9p6lIi_mNhL<}PNjnY)Ytx>KzN&7JlD#a@}WAt}JJ>jvJ4FM&em{b;E6=~-GdXk%;D8r!0tZD@L#H(u`5C0)DU>(sJ|NPZ*U'
            '|McqlPp0o9UWz%V4UKZC=s-UMG=~=C1KAhvISVqmHSW3P%Q*e3%GoT-o&`FCGO(b9^w)^f^eSiV*N~pWJ~-ge=fW7=^Vi2m?@q}*'
            'T5(NQ+UF=y>@#L~BG2RofVwNYOy{BWFDcN0zg$a?THUHh1!-8JtphBW;3`zUAvAUEmdU=}GCIBQV2qJP%}S9|lkZ4os(W3U37>5U'
            '<Dig=M6a=ovgxcT>lW3WHC@tV^ic?cBi%hN-LgjLlv9!{EXBSBb{iwb%TQFVaqe-c`8naSsXc7#WqqXKsnf4b%ta=?<(12nWQ8g1'
            'G+3tGPv&`Z2W@vWgH)@>RSk(s4dHUoq<ba8Mb<KjG(*?c<8*7b^gy@f7w&uvGdtZxD0?WIJqAHG+mYJl&}7m#hGIT;2hyVB!o6&H'
            '<v?gvNg=|vy{k!f$`7k@b{{%Uiw@B_s$ApHr42q+a+zi(sz?lws%eKgbZ}dhITpH|1Vs_ou7CiuNiQ&KA9~i&h1Uy{Fk5VZmMsz@'
            'u?Z0ByreYRH&HT6Z@75#hGIXWqIL;z4t*`f6RAg<Oy!z`WFt}{+PWr}?@-Y?)lQTnKKkz&nZDng7&%K|2Fk@R4i|8t0Q!d9+JV@?'
            'vIB-&U=oHix&@S5X_veLBZBtk<V>X|l>U1Tv?*K;D|Vn~cRaZ+*csJ#oFcuzP=Is}Lz@@68XIT*cITU7v6LwW_B$6xU0ntEQh(Sg'
            'BeFA@B^dIf8e>3N)5$%sxfH^bwCNe9xskCBJ$385voR_jQ1HPVXyejM7zs0=N(vY((RbzibTw}9bXFo<!>bw%U9aPabS0HrG+bUy'
            'U7dTi?nd3rh_NNE&V{i?*Qm=j+-c2h>?d#a17r=%>WFn9<n7@;y~jmh7&03SEX}G3!)}V5Nbg>M<%`3X6>}<B)K<>0BQZ-|JsMAt'
            'NOK0K>S0i*hwgzXh(C7Q*Vrz*ucbo+v@Fyo?q|=mi&~=DUoMVL>UL)#NHZgq@%TE8rXdmr`KH|ivX=eLh3!7opc`D<8Va+1TAH}_'
            '^l<IpY0n$CD_<O35_s2)Ln_-iv){IV%n|uWR=!T-5>u*GdAJ@v_C*q8(pc`kxAJ_|{=tDU7f|(hx;ww!&b)T*U%Fp$lY8YeYyU`F'
            'qO)BI&hCQNnEC9kjAuUqdi3J(8pG}|e`Or@;KV6;4DdhO$Z8;P*fF`+WJ6sCPux@4kJg)HLcGXeFvd95auQPFja4Ak78vNt%}`>1'
            'n`2t!D&tO-woI34G4hxBVu5TGRUp+YOBGcJL?&hzeuZAli{TNY#UcJe2j+f9SF#D|JF$vBAbZ4l&=OtmgWX6d#t)96GOvS#s+=`)'
            'BT2I_!w$JntDP5Ztegcn(-e|WoImY`%ooR%)!CFpi4fN1B*wT#j6K+JiKUKdZKBjpAU@p$o4l*aK4}46eqqR**St)aWqW?~26-~c'
            '5el}Dj;c!odEN-akFR937%+BPo`-`}Qj{sM(!_~S{%M)_jTIiqLDGooMt%a<!H_R$$mdd4dRc|x+Cy>J-6|$TayiQ&7dM!z1=&Jo'
            'OgAA6=PjT@$|YM;;|7HGSfV=TFpw&THjI_EFPBt*r9xsgQ~4vTvITc9eOV4i&WCsY^rxd9T!O%xYE0&J;3C5Cr>xU8ElNTh^k;^_'
            'AH8`2+#KQ+rQ~i{0-Z&$NwKodLc>9$BRMcNZ(r#9sCve(`s#TPYe5T2dn0F}Vofo>AIn1_mMSF<obh^!PM0%LYANGVFIP2~OYy*X'
            'UNf~_5_``Fp3;wnmbVO7TEY+{Dbw)k!2=Z8@-PI#0ZU0$&{$l7qldm4ShCwr9r%qnA#NLhrV~;@W-RtdkDl%W8LiRn0|Qfp6H=*B'
            'G_NXXhJ%1xRDx4q=KSb@+i95teqsqTpj6Lf^nQ}3aY$fD+`=<>g{rO82y0U6F5r&Bgl0zmHC$;5bM+V!4`FV;Ol75_d>ti;5^cUl'
            'cEo@9M04y^ie`4r6Dz@tT!JoYnFXy#ROM6kJJYAiC=J{UxKDIthDCNAs6^fsRlVK$&8;W2AjQRNyZhbRX!Vi(+5@ftw~V>v%gw{K'
            '?_Z_-mvysSAXbe$E3C2+3_72|J3`T>2932}+v>AoX2_h}cQQD5vozlX=ww3WRae0b6UH@t1IsqoGHFUpE~bw#=W6Y^-PS5l-sF){'
            'FH37=r!bOn*30K|cT~K<vENx8rU`Fl0!(+jeflKj#oTYt0%{}x<=mg2mG;w|#+PRQY@DN$3e@=C-zp<${|i!r8pD&BL8+|hD#|%!'
            '-8>5889HtQMMaOr(V#y#Fb+TfEYvP+qW)dKu?jQ4Gd=YJg;|KdlRBC;*2^?gxKHv9dz%F4B+fr4-LCkdn!%odV0xbH-8m<`nj=m$'
            ')|Gcl$&*^gSgWqS3u@qZ7gBG)BX1U5MU;7$O1_Ja0WvDHV6_OyU9gTyXd&B_`KpBbZm6>68bhc^%RgS;)PJTF`#Yr+`*W38NTrJS'
            'XGXDqW)%BpMzOD(QS2Vc#O}@`#&<vKkI_0rrelm8wDNdad}Gy^dB(4u?dvGPME)!Ea<&8f4I*;S<T4s=BX{mMPJCBZnSZpb4WFE}'
            ';U719!k0!5w?q#A4Wov?AY$0Y!@pUiFdr2g2l$zZ0CfsgGurruvBckDnyDLOx?dJ+?3;7=cMd%Mj)F_RDEzn>eEqOv^{P={VcwQs'
            'BR=^&;3xk;0m@dLgTGIpaxwU4fbxIh0A;!K4|qFz%6-fTT#h($S#<d0qXO03&oJKZ)tl3!H{#^Y(O+JP7sr2k_4>uDcV;x8M|wrf'
            'J{AZH76chB#Z-ntg6Z`!g~?YBalT_df5U)h*F%B~Oe5r}mq35|vW(?D`CMdSnF@8hvY7Xtdfje+8Gevnx83vM!IKMVP5DJx$84h*'
            'Cu@Cubawnso0t092}e7#AYJmRI=JYBHX%zhiaHYzeG8N$rNxF!HK~ZxWKo&cP(TdhIb6G$DzA$+;yvYm7Zg<@+^8T|!90{@w4_PR'
            'F^M_mu#ObjB*ZE?uKJwXm#(&wDeD*;@@h7+JA>0zt<=huXt(vGEh4@0i)jMYX4<$L8e-L0>dnU5tIgK`-UHJN6PcZs34KD_Mk)1<'
            '6_)deoAAFeVD$f=QTnF+WycsQ9h@7|ZxUnoy`;g|%nh)u2+}hmj=<ZkF*Eiogd@fl8kugcK@WIWT7f$T(`oY069tTF72|QiDCOFC'
            'WgZJA{;%FVKYo4u^YQ5`@#FEIU%&dx$vZb=VTXg8OS%GPz)4`4qBO|@o^G(oI5{F5XOP2v(dvRg8f2<D9cA67Y)0tA=%R(Kn{Qf%'
            '??9IE`T}$8!ic6AY@K77Yq+B}81SCvm}?g`)-~WDA$_I5{Dy)3y+W*;JpEJ%LBp%o?zT&y3&DQ-sZrW?#l62%ymqztb#t>fY%@gv'
            '<HdWohi!jKT=X45-Tz*J+`GcFzg|f8r-flV3JmlVc2pWWBMc3l26Ww_@Nm4F9qk7O12~U9DH!of0ub9{4%;HW?h_C7AIHu93q0pP'
            '$xZ)1rJMaUMxQs!IZXQpze#)PTKmsw-@Qco%W*Yg+(Zi2*$6<>v_$XBKDyy)e+3pPQA+Z~Rk-oB(j4SG^hqW!0@3gHbq`j_a}QR6'
            'VQ3EPeSGn)(#w1I0L3|>36g9}IPSFm{{Z&^<8%'
        ),
    },
    'google_ai': {
        'sha256': '218f70e619fcc6e8f197aaf8138b7c74e17e7510b1ffa1b6f0bb84d3a479d8e5',
        'encoding': 'zlib+b85',
        'data': (
            'c-ri}dw<(Fb};<EKLx6Jo?4udWIM@ZXPwo~tJq4S#+R%uPj=Sh(v(EmtSC}TQg)QA``O=f0YHEkS;-`Qw*7T}?M5a65I8tEH=IkO'
            '(Kzs;I10vI*b<qS&f4C+m-ym!9N&b#=nX{kwLcG{K<o(kG4~vi#LLW2#57Jsb{nK(k;MPukF(C!*84ZVis75Vxi~&Oyg2HM{$B^@'
            '!}G0n<$qf*eHg;$G3N1P37_-t{A4AfIP*twJQJg35N0CCQW4)r;`+vqyr6Rs27Z(^JDrYmExc&5b?YUQaXj%S;u?nW!|wJC&A9!v'
            '^SnI`z4UhbS_DxTM1DsMZ@o;+y@?OQieSEo(=-^tm$O(jVQsT0zTdjP4aT>E#!6v&zL+Myua+ssUL?G6w)DbqB@*8o-?l_^1YLc9'
            'p<B1Jz)!Z$-W~4@;s4Lx9l}~BH+}{$<GC35L39I8k_aX}S_v9)986$cUMQws)CLf@aB-qJcHlt}!UBZ1wMhI)FwQ)fF}z+*0{SWL'
            'y%nx3hhxC=K0WD+Md*zY$X>R;wI#$}M~q?8nV$+zj0u#YnfmZ7YX{N9U-%JDqb1bO@d}>f!u^Q@{hxGjFL-!@5P;w&NHf^BAH+D0'
            'roqh;CYb|4{Cyc`9t=12mSL8{5KlW|5~Sl8_BsV<F1%3?2GBGWcY!Cac~Tt)e<y^oo4nBg0>C?%Yet`+i~AtEh4J7&0HFV25e8$}'
            'RM?mJ9=pM-XC1K!#xq>&#D~s^7HJlY8HdFJriVBWEr0CL24YKinnEjrLjQa?IPMMmhjJOA54)H^`{x~4c$VO-cf`oUZ^Y9n{Dx;V'
            '$~Z{IOPJ0Gpgp55f9QxiFAOFGe1O;zkO)BqQ&}uCk*;9(#~GePK&;-$p-BC?7s&_E|EG=!<C|0-5<tg0z?ejMS?13d8A2zfZSj&U'
            'n8HZV<&PbadJzEkqp$k;NdgCV#AgrB03Kk3EKDTPYoBb1z1<E1HBJdtQh(tkF!PoMG)yw`=Fn{%16hFWL1;>{p?7t)wRIv-Ga%<O'
            '&G2vo<l1obLVu3f=S3?39nX9ONP(E_ujp(hnMoah!2Hrb8@w98ZuDOF1}Eo3ar)}j;9$@@60c8BUjxMfdUbeuaB<u}8TN*Q(-XvP'
            'U!3)OhsS-<c<ZeqZzd-3O|lHVxz})NXw#Sfx78NMbSwskDMH5Q6Mvg!i*$cy2k6%gfv7X_?{*}e@)p6)Bp#<bjPo!=;Sd1f^AY$i'
            '95BSoCet|}fh{2)rTQ;8fn6ewEkJ<6K>mE>PxL67gw`H9BF{dI1JhgLZozi*{M$r~MY>D?)qFhG%?;NW+~8824t$g*dky4x1qg{J'
            'FZ6)SA%%nS02_fCWUGarcD_H5P2&-gq8Ap^Sa>Po=;7dCcrMZykvh%1RVq;kDdd)*WzOUAj0q-C4mfs%bmxa};p{+<5Ds?-fe(X6'
            'Goa9J2uCKoM%wHzfao-EC37!qh<}I%5E=RxPC&4n)6d)B=C&c4a1Po8NC$UdGpMLf!sic8&(8V>guHiYho-1l&j{$vbRQ-N2Obdu'
            '$j5aH{)`vyLj9OAQ^c)@e~H?_hA*RxxAWw52*+;@=WN25iR1U)4R!zw9dSCH!qEmEbhCt>5ikxQ6EKz(0iF_LiGd1wcYqoBEPx0R'
            'o0e1_xXiB2kl-r`ksKH-#k0Q>Cw(~0;=F%yc+TV;1q$_{&IiSR=>T85C<;tm^B^ET5QLds10N1xyO=W(pte)u7(imwb|azyaRlD$'
            '0aFcAAGuuOkG(92qt0fqmpk{Q^CQa^ZJ=`gH39}f8xrw^4*|0liW;8#UIH_`0j2{x0$*pwx#5A4=Pji{Xu*u*@{4d_aGE^0KWQgE'
            'w`3Ogd5a?QfHQ1AxrQU}^B2<P0?!em6f#Q}kL2|WVw`kvkdXoWQlJcKHZVT2q|E=2ZB7~V*Hedh0Dl4YXBvEf)xtoqRdIrXp8SmB'
            'bTkd$qtp9J3m&HQ6)j<K5I58jSS&}dRk!(2ID;4ZmsrGrI)Li%QD7<O8Hidjkr1F0AUK4<9MK&|Yj_Lbhd#4E3D*{JIeY*o3*?Gb'
            'gZiQn1_L&5LL^S40(UFJ(=!hy5s;5eyz|BZ972%=bHA}U@bFZ$d0GIyKC*hGDL1e_+&B=(`$z6s|1}8YXTM%c0g;a+9EO9_lUIY+'
            '7iYbfz(WuFuX-0pLs$e#3?!T`;rt}P&4C5evE3KTMHqXNw51`5KT(cgVFBgI7W|t4F(Kh2e+Z{Mi4ltF*2vH9eLqs?1n4Xhs&q$!'
            'k!Z~%28j#06TmuKjYeaO=$7kFmq>bCSBVN<6ydmtid$Rq*~m-%=g-t%w{Tv=V5I*1k2H?d?>JSz64<JEuK)7YFGh6r7j6$v1Su^3'
            'rKT|<DM>wTQA;@Pna4jr#*yr}@G_i@>h})*+}iq1d`)5GJMjjTBfw^m*j6G1Iyi=%qSKZ7guXDdU_>+(NHP=aHnBYvB>>rBI|!R$'
            'sOd7IgAV;}NEm=OX-rrxreUyHAiBZo5V<^%<yo;x<7EPTWPIyIH>`({Nbw!&4S@YA4w!<V?2}>z#J=;DVX*bxzBnVqK%L9RxZc6R'
            '>BY(LeCOTS>A&?+egKS<XR@Tiz_REE4oK4d{0@MCoI?zV9D@SDO4~Iv8aIkviPJ9ibQ6i7C|tqeTcX?oL<R;-mXQdiC=4S65uCui'
            'vW_@FxZ%v{1fL_N1F4_D9D?En7=`pm(eoy7YZxr_P(kPI#=fv9P=JEz6iz&lHCXH~gTwxrgOi?uTFly23Y?U8N2&P80y<m0qodRJ'
            't^}3)my6S3&pkVp@^x3dLPknZ&Pi{&9L)pT+f+$+uvXG_CFEgBfcK!okZceT>6|R_%#)zzt#Cl-2HF#KyIq2g0e{^E{yh@%$H^k9'
            'dzu#FXqsD?!s9)VH6PW`gmgK21?V0C%0j;p?AE-q_3EtOcZY-inS0oS#RCHw9Q0x7P0~IaQfNV1zeF0B1|P-D%V5I$z!QNhCqW1}'
            '2V*9|42BgzwlaFOLgg&<wBFR<0W(mwU|&E2gQm+Ugnr`2N7*7s=00%UF>tNfG7O=6NQzd1!vbzVkDK@-sO@T~8y`^B$039<YCKy;'
            'vS0I201n~HQ#IbLHwyzGq;mmOZRM?k;*veN<hj2JqE2JW;RER1i{?09tP*5${=`vq0d?4T$0PoV5)BE3%v=~>q)7P!P(Gw>3}e=i'
            '5v(Z1W<aphF=*W==K<-yM~yrML4j2Bxu1DJi#;NCDe5>9pU-;l-Q)h@pyv)xU1Nip`Q><h4v2nn?*7s{y6B(7<4+6+(1PLnbF$bp'
            'TB0#Qni((fflDTr+c<<7;wwg4H}U_zgsqq~T08>C4T$~|YyvyCB+(nY<8<*Mntk}SJR5MFTs#je7~%nDcJQWu+;ah7=X?_Pwzjs$'
            'Ad{vNoqB_JMELut@;A?b48ery4`-j}&}f`_0UpfzTR<64+?KcnZpvK{BSo2Xkoj(H!H$T>37lyES)2fw(qDQ7%SmDPmJ`@KBz4?z'
            '2gD<xx`}&_?GysE%Ms~g)ct^zGa;!Hs5xjcxb{EzW1v$mqpwRu3n(jbr6|sA5n)KY1*i!c{%An7rjdD(mX4?asyY(YjE!=7LZ}zW'
            '1#riySVs?ewfJ9xfUVpJ-Gs(nH(Ga?EPWt@L4qfG>U|77wJKoXaDZ7-N&|U1^`LOlQ#|QFOubt;_dv;`2~eE`hzCfYJ5n@eITpza'
            'kS{NtdowXZPFEN$xDC;6#YASbWNrgYXyv8e3`kN3_$%>?l@~_A$FV1Rzn%=Z8PTeVUmTLS864poqG>*nqE6lhh{ji!gw9F+1mWiw'
            'FI@U8yx%8q3SqlRi~&3xlMIQNB5h(V)Jo@QVw<4k3d5k#*w%tAn{TD<_FG6U{SU~o4&VTe<O;~@#%eC`RFKa@q6r|MBFO_OJY1$^'
            'k-Ue)+#Z7>2AGUC5M+U200J>I3Xqqkakyk82mDGpTj%|Qv;NTS9SwTtCasg_3wT0fUo<7Vdkb0(kS4afxxo9}xBXumjwEg?K#1kY'
            'Ct9XE{>l%r9rXP5^kR7Nvd=8F*dNr{zw^h=PXD6cw$$qg*hb({gA)*C2EB5VF;W`P&_P2`Br2FtkAtJri^Eq(y|cbPc7wy}j-dCr'
            'Sv>QjQpbzo8(<u7`zLuv?<1-s>B9HNw=ODu*r)fm3uu1+uHQd+V^7d(jz;u2D)&H*?UcU(6kjqY7q;oJe=;2Oj$DxCUJZ^=7->jO'
            '+1OGK-ILyNABKRJiQhp?0EW^`8vlOTYyZt_f81^V@hbo6cG_2;c3aP%KK!2soWFz9quxtbqQsm2(Yrc)(1*6cSPEDLs!ikr`<n%t'
            '05#?DbI|;A10)M+Z7>6Z27F@s2l1;9B<Lpi7@D-KA$xCQquyj+Y-`-x7R@_<<q1E<k4)f=g^1fuOC0()ezI>4m&=SPY4l3)0nqHg'
            'YLF9olLRM+^wtX}!3{DBo<zqSX<J#f1yBN5QLwF=ktBF<-1`-T!DtB*5s(BBjTUh9R;fov77s<|#?OFB8bdOGVBA+|$sty;qrTD`'
            'k3m(UXDF6xTDk7H757DMfCgoL8cVROI+_SkW402ws3gV+K|C}DIPoj+fgyZA9HRJz!tvcx4WvhbIR;5N4(9;1e5nswzBvFn3Ux0p'
            'JoBeo!sszgL!pq34Zz!DV8VAu@RbO_ddM<U)(w5{22x3S8{bD#<hV}!MFP^q&NV3h^e=M)R^3ocAc>Udu)1TB=Ao9f_W06V&4L5z'
            '72@B{Pfs-a;Cs9jwX?Bakzzuj@6Gs92Q4sCcWIGbQQ$?8zza<>r_k*eny8VM18+z@(zHPyy17N#9z<TUQj84r!Ca!DAFi8*ahAF+'
            'YBY@*AU~34VihhzOaJAkF5Y-4k#q{w1b?~&U!)m08cK(*3#TA+-Dc{Cpc3Pd`)EIDiF{OgApSuzVoO-#w|K`|mdHT=$**$XuBfMn'
            'PA-Vy96C4!=z#y&FY*b&>->+3MrwA@E&~O{Z(2_syN8@8v{hGrZob2=eA|@&7Wy(gxhwn&8;mew=>C7u1r(2LrD2`lxe~NV?uN4z'
            '8j>huD?V%*A8_hMq__EC`8MT40`8!JHUX`UPg+Nu;$gU!W_o52e5r&71`gH|L~T;$QN`{kaoU)lF|{0%=sa0-DXpuKL!g1;IgMl7'
            'MEhvxZ;&}6x@qJ?XR%AR%dv|WPWd<RFKeae>U6jCF>I&a6m4g$cajO{AO<1=dYm{pRn!MfGkDOeR{P18hw0Sq=eTO)euKcE-ZM;`'
            'LKAf$_bmi0G%^nLei8qxngVu9>p%6Nd%^OFFsAZg5&@*3k_hPC6XH8K#oo=_+eaOZyZ}TSZ_V!jQbi~=r;R^Zw4P3I_mhUx17nDM'
            'A|dlo&^GYDsREE}D*t-mGcO<Jz<BtxqGv!IvxI1)T9CO!CaRj!fiyS%!le?sAy7#b+m-U1%h=?CzyRpfMCqUCpCoaL7o{B^q~Zin'
            '5hV+3r3Tu70`Y*>2?Lz{MSClFY~~-MDHydMLT^RmK)CQj=BvaJ@g8sPMN431upXc67WxP%+J;P34FK&M4F`4)4pY-nYBut~q9ogy'
            'E<+Fsvhi(`X;@L;Cf%+|%{KU}`s`l&`7@Br63}5`V_4Wg4OeTiw_XB!3h%vnOKf+x;eXt1L88)jdz<Web3Y7Bwbj@WNy%snq+FuF'
            'F1NI4Fh+AD^k3;dQKJe*nEz*CM;FO7?PIlsbjI^YL0$kVrlc3142b3+S_L;~{s5r^jUl{?C>YT`;w2YPOv*@f!>g|aSfJwDN5mP}'
            'z+1i(BXFZk#7*S&kMO1lczLx$1Cnz}M_jA*qi?_>mB?o!T&gCS=3227hDe>2Je*iHT8k<)VkLEeSQy4^WzXb&#yQ|yzF5-$q(Si+'
            'tX*O=$hLH)GS;s_6UPm9aN>GkY8vG@pm*Q9^U&N&Vns?cxrxVU=FpKtVNvLTal}s#j#Ga=rVX^ZM00-yx=#6c?2AuoM*osLya2iK'
            'ZoJIksh-6{gGV=bZCB6wQgyzfHVV>Rv;wQ<Ee9cEK*e~)1c`VzOI!In3#Mvd+WA$p0A^2)18=FZqA>(8kF0>)e;py2HcUv!6$WgC'
            'M>X*y8tBB0599zIOLX)mXtGcI8PErwFrQG|I`hQh)T${xb)tFV6pgv&Q_YgqzC{}pb(Oq=I^-z8N`mMsU7@-!37hI+AjLtusibfg'
            'jUi$t1^HHPXktlTp+#2#(xQ}A<5+5e`m;QWO<v1n^~<Ou-LAzq=y|91Q>jyQ?N1vrz6EhlDT`$u?a(E=a-9@SR>H4M{U2R+Y`%R>'
            'Mn*a?0w`oACNYpc-(dq~erv^alrN4`ajo@k_{23%5rrAi69ys?RU_0+8L1+%j8Z?_3FGlBSD4W+fZRjLhoymsW{T#BhIIG!vivw}'
            '2LW<-OuY2grj+@aw+)LVpx_==9pwED36|QHWwHdCgrnivY>wc-`H>$X^<oNXf7^L*{`WY3$>CE+O-nFi&>AN${zhzXsvi~NOi8Zn'
            'O(x_JqOB&1vC}2k2XyE{w?4dwPwt2--_mWjH`dpI<UG;&ojf`3w$7ALQaII8={e2=K8wy98U_}8fqT$g)6AbgbTlV<kWXpF59(Tt'
            'HOi4Sdsb@l2mAJxm7u$A@Z+Yompa>GM{G+i*m&L&w&4_}0LNz%INOr6ozNK{EA1Ayf{>_mEtb#F(7wySYnapfWOOo=FQ;KcP*3|4'
            '%HW;<h=ZtE?5UN}2fYLZRRfr}ez<VaEiGpfHUQPBC|bWY(P|a#sbuSBi*XR%pk$zooC1kar|~T=vp5KXUHMV=ZdSPKzd)_?Hkjbd'
            '*c3x#YqUh;dzvzbbH4hem3af5W9l()Vbv`(`tQn}sxlwdVtz}m%0-JzoUE9=>vQlD$3i+|IV{VN2Xer|@CXBqq)y|_4E+ctfotol'
            '*J#g|jE@W=nhlCY9EUQ(g#GK7I7v@`4?`uq6iu{1z=F2Cm6Fay`ocNCg7iNi36yR#$o*+JDFLus>|@vP*NzfUY*015MSxdS>?m#t'
            'D(DXhl1C~p(P7FVx=8Otes&)xGa;uDM0aFqFOh+Q1fIjURiIl%PCr~NDmC{CoPZh5Na(h0L&W?O(Zsv)1!(Vg%a6+)1Vy?6zyfW}'
            'NIIDNADvE7X_qs9jEEVR`eoeSvJFnk$L!r8rwFTf))Aj736k+?5{@1!rfBb?x>K-ivyy_<Y`F0iLl&Upz#vxch<secck*w*jL`cS'
            'pe-Pl2om|EtRi-pq|=Jhs{gDsaw*YW9d`Ps%K}Kn@opJAWS(~on+;%6W?1BTWOvwnqZ=o_`Zf>?V^F-P3n#$~d9A5#LeckX?x7bJ'
            '`ijrTftW2h9OYxD#+0<4ES3Oe|M@N0&#@Kx1uO)fT_=nA_0Dxh|C=l(9B)wNS$s}%GzAJu$HncGW9-m*P`R5fqW~ksC?IZ(ejDc?'
            '4WjkPPZx1SQArUXB{VWzw54PAJ!od2|FEAug^7J9U`pLx@;+Y7;RoUgdm4qo4OxSvTI4OWc<$*Sv9Hb_d*q7m4*P#aFSAB5wx^*Q'
            'oDa_~4u%&Xk7BbtUd-iqG2QO<+3Cf}A-9q!!ZtWp2OQ|O!y&qLG_ow!kNU?ixvv!)q+j=5gEWk<y&J@~v8&<>lyHR}4CChaEsTjo'
            'f6o@ew<R+C9{_nv*wHj;x9PNCN9RCFx()U=Y?ua4)3L5gK6r!>e%fs`;3JEmTLzhJ3jbvI5Ad8C`|8S$|28fx3~|(ji9<`Ae&y?o'
            'UT1Z$VX@TPxJYRf)~@GKcy7*zK?5!Mhw;X8Io#*_JoA}33PG%~&yX{ovu^<WuknmU2KL8<mkoI)U1M#^wS;@&J-i)IpnXH=tcc#e'
            'z*wmE-<KZ707^0S&X)oZit>*g6Y0Ea$MM4K7~OJ{C~s)DX`HH-D4ihvNF5dfR!9zhB!ulg`El3X-Q6X`?LOJf+mvuvhk~^@60D^I'
            '!CEp7OnyN-G^wTh_}_ggyu~l&7K0SHocm@tD@lo)P{K;MP&m~$1I}I*bbj3ZHaordHCFUrzX0CfgME3E(^ipXLPwUVi~vtMKN>-8'
            'L`ED@y}wWX3Gsd{Fz`RgcIu+|=mJ2-5=V13&SF?!MbuuWQHdi@z7EzGab)*ffubPz9$x&de>6Pq?;~o=gHnJa1w)cEo%r?iZ3|t8'
            'S4$%}K@X%*%r$}+GxT8g&<Kgra$t(a73g}nFHp8=F4)-~O*(k@I1Xl6APNrnj@X1AsnZLxQd^F?!3GV_TMkJ;-KV?>d&VB@8GB%f'
            'Iq2qHBipzX8qp{rrCKOK`h{F5F8QOE2g>LrKZ<{HI9%=jm{l@n0CwhqrC%n=TQD<XWFLnV3D*i%d%N;rJ~g35Kl1?)I<_SK$RX4r'
            'xu9L05BqPA`X|Tzzey@&gw%o>8Uwrp#+N1NfUAOZ3ukfeXAwH*ZsWlFqSNV!f8hUaqb+O}tU`2<_*nGUuK+w{ixJIu0q0%Da>pEN'
            'o5xEzVl%+cz4o*A)4fqU$Tp;s3O^;t_nsRdf95{J--Z%0_sY1Z^h`G$-}-aUvLC%5KTZX~fHJPYW^sgsw@N6gfniiSc(j%$&qr<V'
            '*?;@>@J_!1jRJJTo5Z_AY2Q*xS|VOxW*Kq~s^4zu5}%KvDbntb?I*iWo<9yI(o_|ko+~)*T9nB+X$o4#Hyf=Af>i_~UM{Rs;BMZ3'
            'coMOL7jPsN0h%h9Lfv=?4hv_D_}-nDq~0n3DM7sLz3UwfDxvr{KZ)CH45o7uq*mUceQ9n%&lVl8f)<uMAB!9e|Lgzt;OGL40!`(L'
            '?0*=CONzF|@LZKT$Z2VxoFzthiFUpG1{3*a(wp2axly|%sDtRB;|x{!(Jlomy$~_mYi1-gtUS>>KkA7ajFkLX!}pN8!ziD@*XsnY'
            '^^>n)e_ybkN0-7Z9Ax|(|MRHHV^9Z<v5c}M$E$*X_w{B?;Q#M5<6KnwXP@;0#f=Sb{fd-Wow`gvG6xlqhQZ0Oe|FwKz?@fXIo&5>'
            '^g?)2tY@D=vc;<WP$SBv<7w+Vg;Z(_E8gjLq{WEB9gha*L+wUyTawhpz^#|MQ5<~)qR4kzob$*Hj7!_x%^kM_2DFEwD_hmey5+&<'
            'rON@1_ca~Tq19#4K6oYm3|m2&K+3Nt{8}y{NL_79=HOIU*_zibh9X%Q)EGODS(mB|)9h@E4s--|y~n^FFnQRm%t3=lMqpf%olQGZ'
            '$d#+v;bmaK(1$Cp-<xl7*D%B^Ftp-SG~dCueeu-kh*Na(REh4UuzGaxWY<qb)pSWAQ`2QAra_Ws6b6h@pu^L5=Xs6;PBb(3kgtkG'
            'u)v#q<z6-Qq7)CtSHA%O7C7v?J@hr{4F|taCV{f8vsGa<Y}FaeS|u^rk)L_#zPvbkYk#7mYlX?GRVl7I01Xt#+Pgos_kP0b>JU_k'
            'AdIu+z9<q)nUOK4Gcp?Z_37Eg$;GdyXD}FOaxltG5{PHJ&oEGVxr!6KWN$>VK)+nXe(m|1#jrc~6h@bJ#JkW(d{{2f<p;)E#Y^;6'
            'nNT}uBNdOG5^yoA!HnO+^zkYg$teh{{0xTj=jgk+V)2(E0TK5-;iWL)RUk6>2ROTk!^twiDU*V=jFw5boI!tDl1$DJ0=b^xfE4i@'
            'cEn3<3CXQ%lqJy<D<Jmp;f~zC1M_}oLo88Se<<&Jn(M6^tw-zuYPi0>D{59L!!6bBmA*7;u9fJa!`85O{??^#M!fNDoXum(q@d-q'
            ';*pWend5I%S}=~Ft!avpilo$xSC0-W?=8qGk<~XFEyI~h?k<?I$^JecJK69qNIW2qNXP*Q*_xIrT}9QBtcH2k3z&iLMaxBXyOL>5'
            'H_I_g4N0~i19jx{+=!rGOH^o;)VA_xBsM6y<ei_in`l~S@kw*EHByi@vLkvUOw%E{e@5Q|+f<3<o^yB*-VEOiPTmeqUb{#AUobb$'
            'It2Pub(v%sJS*;J^xOLIi`wf8vZy|+!P0LtpA9Eec^1CWj5eH7@yLCH*=#nE>O=jhzO!+?Qg`*E+dQl%2w(Xt)x_j0f25KSedTXd'
            '5wU6nkrx?N*daGrsz)ePg4B(IV4kd&|HPQ_ND1JWvQ}q%ma&kM{G9dwaxpmTAG)mZttCIEQdmzE%B?C-YPnhQ^psmvqjkAm4Q7@b'
            'RwGxrT_t{$n^kRb!DM8yDdQ@lf=gNE_q9yQG9DI?L;0Gsa@?jwAY$xe5W<a2Z%V1|K$h*j@BJ!X;ygL`qlY=UWoozdma?T{Z%z`8'
            'SQcU6Xh`nk`)KczHgQUCq!^-18IDYEp`2D(9mkXdb04H@?}Ky);xr}cHS2-sdMOhNWRz7BaGVo*lKPADna7d85~yMAqXwCQ=sKpT'
            'JRF^@!Si6!<^%>UN<p3~=X%&`PH`}fQ_A;v-aj}!Iixgp{@<5A+8HS8nhau4)_^?Yq?L7$5GY<^HbII8;562p#}02QGBeYivSQ14'
            '6qO_tbKV4$7u*&mC~;Jhs)TZB7K9-=99t>F%W&+<jN_D3ER_fTxc65VWSVyuLr(p2Zn(Wr1|-odx;u55C+*9SA`p}@Vi;qG7N~JH'
            '|8wuh-CZ#}L$8TRs+?=T6BKL&uz5O@h~w}Y(<LZm$!dr4*r8IT68=;mPN$X{?LBv&{`71OA+j1&(jcovk^Go16=+_TLBmuY4@jP0'
            'lgtwfZ$;Tf$;8oFN5Fn^_x|`~J?S!=R1+<;MJ2g18mQ$Q?B}W*^q`!Cn01;L!<l3OwKyFqdnpEy&PLa%CwlT*57LNK!b}V~Z}Q!o'
            'G96*6+JD%|*3hb9*n1?=l3|bNZY)>^m&qkK|C>HeXgi8x73NMfunCmbxK0FOLZsv-){$|wIhu>srRuXp);jz;+K^N<4z`l<Dbjp6'
            '9QKdjkyiUmLdeD{8$2yhHnbJ<Sy12~PK*zc0X2K%7Fju!(Tyxfs`5dK*~U!|eY_r0LW9a44z#4nccKj5{J8QbF}odf$ic-bPghjw'
            'IvQFnjPk9wbkZ0lQ>Tt8+rVzhfyCziNmXpBez2vKdX*d=TcYX6FkVh#^7SKOa_LP~mB3OIb1w>}$UCIT@X6N?bjgwQWbmqgK6Ei!'
            '3=<U?p18Se>8crZFoRs9+<Q1Ya^IX@oFQ&MdDc-@WOo|);iL(SxJ5DUEp;Rd;Vy^4IBT=1lX?KRiD%0N9eGS~&zw9}sfmKb3@}Y)'
            'm<SFcUJcI9hsu%<EFOqfFy6P)^XsH@h|1UO(w|=MTz5c#pVN<-ze=w;3^dD<V1(vdl1Q#^kThpWQ{P8ZiUYCDgKRt3zzZ>1Op4KT'
            '8P!-Jv}0rT2q`Ag(c5^U(tfM*5t=-SJ0D%&OR*^cg9=xF9$rFKH8nz52=o-DU+M&}bh-__F^(M#!K*FrpALawQ4mBQmk%AqQzqVh'
            'DYCC<Oid)&;)H-<F!RNBjlpwUQx(dS&{b#!p8<6VpEWoM0d5R@+PVkHWJGf(1$hY$`A);RZ0}v^gdMyNoRzB$wJKVH=VQlpe7|96'
            'hn<ZF7I4rzIXxL1^p3>A=`kFf!+lO!Op<NQ%@$u#u~3;8EBcx7T@n6I+wlxT3%T$1zPJn~=@mSngcRHAhl_zZ+YcQ$CyLT|A+k@L'
            '07pw=Zh6XTtZvJg*z*kx-RJWaf4i+O%<gZc;z_b9shl#zN|w8!dpJ&buP%8+vyf`3OlNfL*8!Ze<(y=Hk;^?ikNvWR9tC=xH5cb('
            'A0sQsS?62p)<p6YciP-!Mlu-%T@z}9!R64Rbnrek;u#1_GCrWeYk&uruxSn2s|IoWCUkbxBUffwGLpf#DlgPSd2x!J(DGz7`KRjB'
            'I_HcMbDdWcBjYT<J~%Z?V7GvL1^HKvmGJqnSAowSwyFZjYk_8adeuVEZl<9r!?hL4(Q|~awWn2`)r_U|zABAiCy=U$1TACt7g3b8'
            'c1q*Fd=Di3lEZ<Hj8bjHIgMOopd5#x(nW9jfx;jzN^<CjTh_%|)tZm9>6fDTesf|v;*eQ8rJ-RS(;|$L78Dg^CB&pw*M;l;Yx;f?'
            'FT}^(&q=xkw*M^i$jle0T2TQJKWS&i#V^rH^mfu>^)f{^qmwn3cZM8pu5lz(xsj?J4^9Thu-<eIniS`S_HHn$Jrk!VN5ATb-9-|h'
            'k}U~12p3r7P1QT0vPI?1Gs1##{`a?W$b-R#0gt#*X&!T6j&J?(Oo#j8`K0P2-fBC=9L=%>3d(3SteP1{abpl0l|tdY0$IzZ(Xi=&'
            'k@&yRQdOepz~4<(fugg_##&J*p?g_K%0=t$CXD`(ki<=q*E%x!Q7*9%IC<4mHEC@1H*><bIfXC5>!I#AZa~1dO00$$hF-;`n&Fir'
            'v#R8N9$j%cubLKa>b&f{aizFLxJ07^;GMhqXfi9++7WY)kiuZ*XQXCY;Y8x_aD1}Yk5JuPT4)AS0|_{bzPLvrO%_bLqq1|1z;qVb'
            'q^ea$<7~+haS$qV1-b|Nns9<`X{pq9fNic*sRK%}j0l{1Bw?iHdyyQD&5$FX6VC2{UKL^1wRk$^oUS0U-`sEto@In##2~E8NPE37'
            'S3L(nQ7E8+F@W?}mKmVjD?cxl;4V6Atp_<{s5`=3LU`&Pbg~@NlSrj31AWQZO-K9#0oCMbIA-G3D&xeZ8u3cEGryj{*0$L^iKfbA'
            '>E>afoY9gzxzxj}q6|}t5Uq9$tM8Rn_sZ7y;&{?(FSA0gLIfxGu^twClmk-@NEBa|AfjVbxxM_eBskjfs@&FkSnN^fE(g6gai%Ly'
            'Gbdp%fQgBQjBWQ&VY56}{_Cog-BijX7NWB)jvey?Y2#$S7Pg5rE#m>lt0>!E;Ue)8$GS3e0#jxVkRWCORV(O{nsb>qDV{W%MlnYe'
            'r5-^&je)1!!Cii%f!Na1Ye~}3svsYZHFz*p<Cg@7;wm~=a3Pu$td-hf+T!H8FRxp5{lpy6*i$O9g5SpgZ2DoUTeMpY)G|?EXvk};'
            'JjpFRF5(lnWuS2WT;7-hnc>-D1O8PUMS@pSp~ij=@N&}MC1NAeb1PpJr!<aYITw~f8YMi{49#41gTp^?CsY2sT}Q?U3|A#)aSANu'
            '8Q(BYm+NH|YJT4d(pY6lH>2RQ(7gpZglnlcdsYmc<C~!v=MO7;D2dBphy3WOzPXafihcOG*)bPAF~z=-)ufP3x|G?nHVsalC`^0&'
            'NhO9hYc90%e9(<$gcYl!hADwIG<Q|lT;S$T^)6PKAk6Fo)nX&GuX|P1I~O0!j`>rgkBQudfG!0EqoHnLN&+VSVTc@)0X%K=!_MJg'
            '>T^n3H?v4o<yJ!Q1Ei=-k$EoA$mgPOHCJQcVq$1%sVX97^vjRCU0bTy!qud%5DB>QGe1JFR(b3Lv4TAe4Wr=?6sHtoUDzZDF=V-_'
            '+>Z=bnul1-0)0}kN?Gzky80G!tx#>Uk#`e9k1^H=Dn|uMYJ^Op!hVvAadO(1r42!{ja`*B$bM}@VKe=+TAQ@#ubNL+{b&g`(^TsJ'
            't?JA@s;^FWv?=qI++i&buDZ=cT}?>>l`2~~f4;^KAYG}<X1$5*O<doat@>g?psq*)sBm=^&qVW6VHlPFuU=DB!b5p6l?c8on#Hgg'
            'jw}C|f2jyHhI{z~mQ|PI{kYWDN^<E64UPHr%un&POc>x{RJ(OAE-y&=)}PVF1+f?CNqXB<Ph>`LZPx>&RNQqR^v2jzt^hK1Xd$*l'
            'f>cWeGBL6??9nBz&DotbkuvaYjN?%=po{VRk316>L%{w@E)}%aNCnO#c6n{xLg%Pl{K$!ky_hR!Yklg~(h@EESSR}!MANw5KE`v('
            '+b2F118j1?ZrS|yIi@;Q6aibejjuT3Pu;xCY?FZ6C0u9%kR5$N7kd*Igc>%jSJ~77)0SbD0v$*c1zm4o-}-#x9ghdwG)A$Tko(JQ'
            '=o?t$^HZ|vnClPI7p&K{O4gK*rfrh<##oX%KkJE4C1S7T#_T-eRd6j+=LWju5ZdBV9UUG?19akJrP+82KO<FjY3zq#?j4w59qBZa'
            'SCVAB)|bbdm9^Qb2Loccx)L{KVxykWnxZ~iIe#_t?iy(8h@ILL#kKw#I$g14GqX^XEPs@-7b!9>(pi)DdCdFRrDA~{a%F2$XLOq2'
            'Y>$34j?q=2+gN7P_D>DRsTje{;oWTVBc`2(8X_HZ*at|1(Rl<@q0eJ*HkA8jaXMYIszc?c${dWx;4=yTZ!oT|{$-l~Y*2tyi2sr-'
            '*tiX{a)lR_<B^(gm}HwgLB)4s94E<gk#T1{mWfYPvZzKsdMmm0b>+~iYpsfBIZ%xgV#w1u<nX$iK}Us;T|`jei@-X^grP0nZJijq'
            'Mi~D#*Rom`mm*L7b*WRPs>keSsxwjW^&j1=6|eu+T>Ys#Pw5W{wDt7WE780@yEsuUiU+4BCw=AFey(c>UsFz9^7&?h>vTA)wt1x|'
            '@9|zx_1hH}LLtv@bq{Ob@G;4=4LgY!<^BX2nX_EmF~W2>kVLa^<4-nE)IoI-oYWr&93mM+@Jq%_`qF*OpKyxPY&pmJ&&Vt|{WkhN'
            'cXYnMe<sHf29xr8sE1RHFk%DlOPx(DyD{MhqV&FPv?Q``it3uon--1S1C+dHt#+2h(lN^jNK18<xmJDKybm(%*}3Z}`gR771iCv{'
            '=4VO3sIpEYOfNNTuX0;2ehKqt(g(HWSKg<t+o-y5G__%kLIHI?@~Fi<IF*ybTQyx5yx`4|1<}&CZ=_I{?DAqScdxdJquGdeRc{wB'
            'Nf$Z<qFeGcD(2N{mY_lnyScqVHpE<8h?!+D^3_xKQ={aGj?U^2b_Zv}T`ixg>TNa*{A(`q^VCKWpv8QVtyFqcSHB^lOa0|IMc*oG'
            '%!m3K9VN1|5rTYuK-3YZ-{Y+HWe!^gz;q=$T;(@#_2&AwwkMSML9JC5LST3AKYV?6@BZ1@>6v@h8}{9!!SP_|p1nIpk&4b6*+S_a'
            'xT@V@sog}idpS5dLbrY@{Ep3l7-6UeD(Y-+R(HQF;K*)Rf10u~FSA?q?Bo<w8Vo@rL62QmVql<Dzelz?@1OnBKcj@wl)*(_m&mun'
            '!EqmGJf(;Z=6;M>re(k0tN!p;>gP>;3Za_<TM5<s!O1VZqrssJ>A-eDL@J_Nk+_<>7<$L?;QSnbfZZy%rEGt3^7iEPJ%(B8TMw>~'
            'jw_HJVq8%ni?Ps~48cjp%ODdY(7a}TvYk@JVt>5Ebi{4t)<NB{0{ZZ*cXB>}S-Jepw}X>I%AjCEuO%#KIp!?5q#bDC<J&yR2=W#z'
            'D<$1UA`HKw{poDI>Kz@u>>a!<evi&m6~?uyo*t^F8U<R$0F|>=f7%<@+Je=nvL{D_w|(~=M%<j8<dd>0iT4H{z02M9AGwIQ`&;|!'
            ')820DLAfVdZJR#_&Tk{E2ylrFe+FsyABetjBG2wWM!uK$$v-}{k=(SQVH?^&pWo8&FZa7w@Xv4OKh%!hy!`oJubMya)3}Sk<quF6'
            '7-m>b*SY-nfBh}}t+9Re!<s3kv$jzc-CUES3s9eHtUe{A=E4<uv8p^S;|wbaOgO;G%Xzmnj?x*SaLZP1fuRDI?1f2pwG^f$gNxEt'
            'l%bV;Oiqy89pjSD`^P;vOb38Zzk+TTW3?P&g}Ju@dHlC9nzbEI+aLzt$RHrlenFVRQXn1Ru))m@2D7OOC{!c`)MHfgD1$c!Jjq0k'
            '6okr{Ox2_naEL^lF@<2rRFr*@QEGEpN?<Arhzce!GP*7WRjH2@D^v4Qm(sdoiu^Pc`l;nt#ZZc4EGuc%&2Gv<+QefpvLU*0@Wf)G'
            'L^#42PLX2nC2oKWQfh?n<7B2&TgvL&JA75QL#b%eWdceJ$KwJiQv3_kX2sd9uCS69z?4BcMtBa{@nTHSMa4K&O)ThDig7V6or&F-'
            'H=}IOH)(wBI8_*5H<V3P>V~XQ-RcG{GmI_iH$0I=*2t_#LpShYLzXTxjgI_Q-lUq1SVu+D3=BpzxX4uQgoEX3y(D8VQ2a&JHZ3X0'
            '{)EH?HGHdb2Kig9KIZp`2r~SVVbclYdwIuzGwslyO-J9a5+rb5&vy4(;@R#~T6$xF=$q1gBS1Q!I@ysP3tKEBq{LL<0c-zR?|<7W'
            '7!I+B0NN^Y0#+JNmVI%h5MaKq*<2K8tYvh&ln)9=%cYp&ckaDZ7&Qvl&RSC3yq@*rlhW+8G_xmvq%l$dLjO9~|8VOq)2srt(Dads'
            'u{T7qABu~7#a=y8n2n1eeyNLrU#=1x`GQoob}H)xRtTPTqfS6TT5Ue2EtG6M-sZT3xtPmd{J8rB|K}<G&ojnh3NMI&QnJ9h9-@HG'
            '%00aVSG+Pqi+sga!523`3Kpl-#I>MY9E|ScMAe&#?@iFp7u^7Sf3guKN~ukMm6>E|=n+@*EIkLNc^iy^%t$M}9)iH`W$^kY5Xl*b'
            '^;}Bq%J*SCUR7m6RFR+2syX+P8Rq-TC(UKX)zhZow&vqaDrWP=pYoUV@JrnVIp(Z5OAK2})H~JM@Q%WH%2S9PmQlEz(Qpgp^P{g<'
            'A3cq=xgM$2j-}@b_N7*l;t9}}d!SmkD>DnEj}GQ^U#!Y;!;@S<k{l?7A6~@z(Y+Ut@ZbrMXVA)rQV)t|A!hrCFja`7Bu!3ZlK6ne'
            'E0hbdgbT|ZbF_k>_B-}SXruC+C>&B6Kopu-N>n1E85Pgb$K)b{2}H=39+vA%UwxUzQEErF#runJ0h2D{N9B%CJQe>$xel|Mn2(gK'
            'Y~p|)k$=1~z`_Heo3dx-?IjtP<ypmUqgY^vvB-EYwxx?UJ!<SXb{pGyC-ssI2Bjt%>p(zA@s<+c?UM3|Rcm7qoLSDOjun3sM<iv+'
            'G_GxlG82r4O;+t5DX}}ei-&&btyq1bimZ`}{yIW*$D{?#K$cv0P=J0j9KI9R#z5EN76=~J_44o{8z(x!F|kUR{y-sAoSY#r&Ky36'
            'wttGd2qMm>fQMQ2Lp$!%y|f{wiFY%n`frhr+mHvKqr-gdR2MRbZR)Yp&n;&4u~3fZof@*GEyF2gE|=mOvP-?>S!hrp8p{@?kaMc>'
            '4pJ8)c*B4%Q#bt6Q-6Z?e)em-_X;%kMuA_$x8xx5JLL=4lEJw^^yl7(=57ZQ&XbFY9On{-y>iO;&eDR~)a4ypqNFAl|D%eay52GX'
            'j~8tqk6tumfmKZMzNie~FB9JC3Itlp3lL8Blkb7OXE~ve+g#Y0{Ik%$BU{ro2@X>A&2oi8_K~NPi+5t<*hjHaon;-8Zw#ec*A8cf'
            '@N-*vFO1d&ygTS{(wSx=A^S6d_P55*hQX!rbMv>!56z$Vf9t@%&d*NzbHmV9aFl_@Vj8;a?MG-r?>eZ}FPeLfWiVE$7bG98So?m('
            'Iib`AKzlNYh)!8=EMO{*<kXFXyxDX{)j*=30Hc2ga1>wxPf`Td{{fI#TU3U{M)1fHKqtwd{0#DWk!Y;NfQoCQDwxPIoYIxYMhCW-'
            'Pp0h2xY0#~yw=AXrVwklV?v`Y$o{(}yUmd3I0yl`<z|PJcIbk(8p^(84&A1l3vAEnsH6`QEGQ|Oo+B$GU@|Qu?~kl4&&kU$Z3uD='
            'FD2D6#JB_P_0k+6&s!1iU<qrEF4rq@bb9*soKtdmNN{&7OQ)p0JNY!F$I8Xo5pXM@5xHGT=WR(;Ug|W96UsWZy2pfSFE}R=Ww+32'
            '+zUAy*sKAAR+(W)DkTTaY2y=ClA)vBaVhZCbszS{CwTBsWnrohYvBBF>#OO?X^|;IIZ74_#RB$!|NZx8{lgw+q=%mk74us^6Izh{'
            '{Y^hw_c^xuWT=B8FRC<=m%br8bkgOh8G7^41jO!r(VXj3FMcS0j%WwBNSktBijCG@cH_sZty#2Y&f#JcV0>}8FOT2VH@LZb=Z*2k'
            'u8qnH>SmnA2OVv&PSFwtH<Ui#9kNn`fRRG8Kf(Op@>cKp*T2;xj<ewDx$$%gUuRderc9c*%*OlLF;}Hq)AZ=#T&+!93=aekurJzw'
            '5R)a69C3I)Y>D<yBF5y0_d%+3wF~s?ME73uCuW)SqoABCn1Mx^FItQMB(9>xkjg$MQ~(@C=8TYB(#;o70OQXQtR&tYQysOv-TmEN'
            'nwq?P+DkK|(Mf!VNs|(>`$yT%VahR>FG62#OI~0`)}0Zg7PqDTiDqh;vTY^Dn{x3!h$iuUD)Tmd&_?X*>?1Dpn({Yc(woVe6f;zs'
            'S_4WJr~VuKJ%GP;D+W9ud%kNOFI6toP1U~H=mCKa#@^0RoVvZ}#t;3p;W+4H%|kRTCN77q6|v;`0gPyJ(@q9h8*16b5j#8L>2qLY'
            'kGP#{iFXAenY|2rA_cdEWD-QiCO&P6r!BE(iVkkdTicb5e=yv_;d$C^J{N6%*SYuo89YNDp8qTU08Rg(JkljjSYO)P)1RUwWBHeK'
            '`|6LW*M;x#P!k*|Lu-2O$=|e=(&5kB&*pK2u8Q_U`2RI@5sM;X8J0Rfe4GhdB#e2O>y*>59?4l)&(=-ikLzb3aS2EVh8$?X{K-TH'
            '@h3e|q6_Vxv>AXY7<?ZM@0G->!u#A3t>9p9%k)^3h6J^FB+2r2T#S57V2an+EzGLIlt5E&Mp0=*YAd2hG(x+N8w=?QV~*=%A9D2~'
            'KNNLw7}ghz_K(J4)O9q7NV!uOkG;@pZyf@5Zhnlr*sPd_?9jn;%y3${&*(d#2PEKby4C7c&bojpu=tdz1bXb5P~1b!mj65Kqh<t;'
            'f_?KH<Pjuw#0{Wct^$-9g*1-mqX2Nl+{Q9t+;cuF1CAaY*lUMKlk+$oshViJc4>@Vbw1(9tqF;&tf@U}!kzX(4wx?lg#-kKbHF4>'
            'C>0~7<cw3beJUAJ;-$)0A04~mXuM1kG~!{5%Gisf!V~%)CDgn1!f6{nm-&&D40m+=iw;pS0z>e39xl>ZcnKv4oWZP`{6{LwviTn{'
            '{xP1>pHg~;^>n!-3NAq5r_<mA$8qp)Sktu8fp(u_GWRTkNz-{~=<XecC%GO{B*<hr2caIJ315s-(1u(u9S4D`bC0!gqOALb6ocO|'
            '`By1Sgn^aj2VuTw30|jizi?l{E*!<<S<wR?03N4U_a#QB7(q7-cY!BPxv<GUPF*Y;*y03U9txbH-=<dbl1s|2uj3f_PefUNRT^&I'
            'j%`qqVYbkXmY%dRkC%=!TB0?U6f>+5&hkz$;ag|4?1IoJMMi3L6hzBhXapc8=!=2(1^|`7H*{Q;saUu!tTc07kRie;!=<Gr-`yj#'
            '$082HF8unj#S3JsdG|RyGIP&B|Bi`D>;M2@#;xffPz--+YArw0l6~Pb@~80`(G|LxY>~gThjXVBNxseHQ86&NK4ZG-N>Pq4i3Iva'
            '8Z#>|?lVsWg^f^v89uWm;Yy>iu@FF6tOQGN)ZYg(2G1q>0neg85C_%o_h%TH;nbTo!|7Iqu^A6QALpNv(dsiy+!(-W0I+Sq3iU7G'
            '$pDl&vDiPCc6Yy)sOO_BBiN@|Q~EZR;alN;ZTprNmU?<#vqVd)ShK`GcQy7@5Aq!ctH>Kk3X)NE(lD1lz9MZ8U%wDUT>MGRNU(ak'
            'pcYL^l7}<`)Aj>&DAtA2Vln5Ddg!JbPn(LXN<-gYH1!d5ti$ucpbg&x09|CEj^$IKQMOj+1H>GMaq2f!Q)jCN=e1e%^H9JFLC(`m'
            'AN3Dk_s^=5O!G}4QCFE<8a+}EY}Z$1Wlv2P>4U-9!Np+czJvq#76u=_IqRRlIXyxb;-{sz2dD7n^!*81zo5b1kK%jrd{^_Gzu+{E'
            'h!1Ga^A5AjVC?`Pc>VL?;JAl%C6O!eeK1>jWP**{M4Q*vI6B1)m8z?%R_mb`T`(8k2w8))t)joeoxavnX?9uhrt4+S>xSco-Ux%3'
            'CgK|NWME;52?rNkcf=rL+LyAm11~s1_@TJji0;nS?RZu2ESv}|*>TU8?WxX_O$9vo9usZGhHJ3O3dbIhXr`HaWHqMzl=OY#uOz6d'
            '+3ImA*D!$!EhB(Xc}~nj@?DV1TMD0p;bH<_F5@fXRg14HlwYA?lVibuBTg%=_#pSY)2J@6uP!pWyDlDe4PR!SOrCP^Fi$jZA4eT!'
            '4>LniQ<`J<AJta$5EY@!>w}=nj|wA_t*<L5NUplJ?I~X>RU6ghqapxVkBN>=${!F7rFhb0?GFD-?WXyb^CU_1u!Ah?a|~~sY>Hhi'
            ')M2a)^`TS_c<d=7RXMb&@oWFAe<X&zH^+mM4q7>vWBC6_to&?-F^rT81gka9mt2T(lBcyP#}lLHp1{;M879KPLg=57`o>6qV>=VE'
            '((Wo$L;r;elO!nBg%j#Din;PWCn-@Oca_3OPGVMt+=Z@OU{fagDxd5;1JkF6;u|ko!mzaJVzB8$PEH7MHD)yoR?=8=rOAGLcAR~F'
            'P&N*>=V8oXdwx=kwRbm9Qc(Y&eQm`bk{$&)`6@@?*Y-i;py1CgQ6AT%sg}WtYwgxyfKwyROR~k|)HmtCeoKj?SwI3zUFd8WpkGTV'
            '&8U5ONksNcG?`N29wjP}kzuP!e$>c^^{{f}vn4)NF<N0&mHod=*DuWzbEW5wOvbkAX|)PfZmqZ*8j}6JvVlgoY~^1sFo3oSvEZz>'
            'U(Nt{1gfyZT)$jg$L0x-mzSa7Wn*_pK<k=-)B&>P8aFx#l@jZ!E@<S46_J+fxsA@%NM57TFy*q_By`xVXw5I(d$M7$`M#1|+Fu7j'
            '%Bb$N(T7UYnvf@4Ra1E(P(P2nP_|3nm^I^Y8s)W3KncjGK^>dOTLEib+h`oeA;2(l$v3sW9kW#JeE(Q;BlgdwLSHh&07U`TElY+F'
            'x;p0~X@8SO-P&YeC8=YsDx|Sy36fjqY5gh!lB;QAemJbO$BgrZLse18zH8KND!g>o_0HDy&K~Pc`HSnnJ*ntzXEL_>TCX^b5Z+(8'
            '#+Z@|70q=nDn#<ErcM4-`P2C+O+bzfPyy5{Ku;=bV|H7jS-WD;I?$a-k6d|%ueQpO^g61Vb~TsAt@$%Oma8apG1h+!CX`za=?=Ge'
            'eF$wTNUY2bCn<^}{sk{SR#gP|Qn$LI+mTgen`#h1huxYXs}B2B$<QM2lg&9-QD$4&!bh7}G_l>f#B<cZsWQ6BuC-TQ)t^zl9=-S~'
            '3XGg2PIh-nehGi>is!p^ZoUlfDxh7l8eJ*FY_!Faua$SX{^YIWkL7XJ%adPYeP6b=-?OfATI|nVgCP(metR17b>Q!{2#`sbNFt{('
            't&j>^qI$u~$#b_di>Wj>y~lVLHiR8toDZpd!^lT}Zqx$MIz{WCY+|yKr8Iuwf2WZy9+y%i3uP$R?*9dg8mMo-^v`u6E?Jl*rz$8W'
            'q2U?{rlvudKZPZg9hGSSZ#h~jyhc{=CF`GI(N=erWtrCyvCFb<Jr&1WI2qnercLsfZ`M#u9mvE6k!%V3F$o?mDby@VW#k!umtfwi'
            'G#Cd0c*v|#%;3hsMHV%tiIy~6YWNv9HGE@VPl#zer|y!6dv8RGt&hompGA$e8hJ1NBnPPflFpF(i8k^<G)F-ez;XFR3sR|ZKab*-'
            '?t+Rey&!{DdwFX<Txf0NHd;&yjJGJ;kr-r6V?nlx$7BLvSJ1NBoxZ_ot~2izPTjtcxXs~KlQ>*bf?*ncyK409tB0LWF!%#Hx)|UE'
            'Ie6V-=9P<?uV0L&?zEoyM(de)vtTB}$AU)}#9xqWn&X;|jWuQK)|6GQDU)leCgb$#x8Kd|eyAJ%&-ne%0TUN#w}ADZpE88R(1L^&'
            'sh|x(k+(>1<7`9o#%GZ3YU_9@yVGZx!|RO`kK$k5haRy-eCD=3Hea?{tUvj+Gqv@x{l_)7$Ey0&>B*1P^{La4AFJ$BrzsZu70eI1'
            'oAtc>wSoa9|C;sUs&q%8%ta+du4K1oaWYP`-ziTP?TchNsg@<b2RU!Xx!qbKPgPctT)9S=c|;g(3EOP5E<I@>=2qT*%G-7~yTzop'
            '0PAk;9ed5VY6QJgIQY7&_}@pF>V?Y3x!4a~vA6ZbG@mUfyDf5`uc7~BHQ8WFEG1OV$1+Q<)9?O?Y(6LtTZmY#{jIS#@<LT1L;GdZ'
            '{?pmALLO$--zwRO3*au8`_|c6%L=lVM@w`KZ>SpR`01iz)u`9zOZ~K=wq}Yp9V}XrYAkQ&5SB^201a4BPM$~`+dP?@vu3lU;d~t$'
            '+n)yTRei($#CPJ5Ya^2P6ls$(2|gxO5Jox`sn69riEM7Pt8isUv*QM9!hEz<iTd`x3>iykY+~bMok5T<k^Ij>QbOzc4rMeY%bw{{'
            'E{%HKD#qbd!;_*XtFT08JgTwk7F9o;YEq1r;gXA*VEU<N&I@>gmK2N!gRf6Kq4Mp{y^oE@tlQ{b<tk|D<6kS6+uHkQ$>p^hlM|h-'
            'O^Jt|zIowYHEAiUw4_l=N;m8pGFmB*s79lLwNvss99`yr0U3<4m$TD$q@(rZWa)I`Hw?p=_O6Qz&pVyuFOIZRH#^@cen_ry%G0f7'
            'c^I&<Os+s>6-k4Mtai_iZ!jmUZ7=WTM?gzbu5s#B6pU!6_uhj1#63y`SZzF08Kv2@CyUw0^<vETqFGAQ5Tw2l&tBn0gnI}0Otvx1'
            ')0c#R)dD~oi5t*3XJcP{j1x36OqPkz`MI}!G4oavOo$#NPSF)8KY5tYq+Euus0mUMredIPvB;rXVyP9Q5w+u=$~s!vk}hIv926?*'
            'V_Mna+2x9L@IZkpc_C7Az9Jx|_+~;Sa)A)|A3Ii|LT#Dw&~Y-QwB!0N(=5wGzA;>+w0!?*TS9p(&Z<tM?mu>a+F8VlCO0l73fb7B'
            'Mdfs+9WH79^i-8kV2UKYcUj$GAC`U7iFQ-%=aCu;Us2^jGa@?_>Owo@y1Y?sc}f>VxmMVGuZu5G4hA(zEAG3KCx4O@T|T!o#ZuaJ'
            '1XDMjSuo0f=4-2lOt<=R2~PH(X5|g^MwMXF-;xt2SEE9<Z039%-nUj_*KoG(N&rUZOmGvQGaE`H81-opffb9;qq?`yx0zR|KR<jc'
            '^Gi4#BZVl(>6DD(#bh70C!73QAvF%)%)`df>n$TyCbDOGiFU|}GVa_&rcIdnD?69fzRr*$ziYN2=ddXLySFd;e|WklgO>e}GOJLH'
            '96&$bR+rcxUS*82XdK=pq{TOUb!t7_Dkg2e#bBr`Ee}d{b7amv>#Yj5mK#n!N;FAYn!F^*FLscc@fsdX?D+)r;Eq#6wctk(86&E~'
            '!?;m*W<0}#Vh0lI6vA8hPT3+#`3??}B{22>eAXV{dPz#*U5spMj23fs5=nGy7~QlD-yMsarI$=hRO8$&HJG+*mSZwd4LcPR_Q+fp'
            'MwlJ9Du>w>3fp{W?zP}_MsmG7JK~x14KiSyQBe)9;LB<2{STyA=$!Sbcq;z&TO_V0%Qk&ju^b9Lpc)=IXF=kr1y4*e2>6TP0trZQ'
            'sZP*imf@-85lri3A5NdgbuL_$;Ll<o^#fr>67DE@hGtuNEw8_->N~Q??QqyXem6Y-Vt%+0Gb8=iOn3ogb~y<$QjultAk2tIM*w-A'
            'Q9%e_cefG3+dLuo9>s%@h=gG(xI36vAj~MaA&76f2+R+Pkr+cMiGQ1hqLX(mVM1kF7%->5fQC0X?4NTb?CTdQ=bg&;Sm}p?wbLl^'
            '$8rSA%$8h-Jhl{#@ogN8DL=_76}Pb3R0WE@N1f!l5(B!jG?I>lK`M}eSWO9tF$@M1k5eE1AZNLt;8G#&XcQvM(K47&A8X_Qs5z@N'
            'UE-ILV^g&qS>V$sX=7%OE9eJP%ncbtbAJUyP7Zy8VQ;~{_=Lm#OM-W_{w&0*NAOHmDK<uT9-M|_mKjwuXsUD8P_VQU*_Ebp<Bw6U'
            '_$cJ|()hH$@L?ReFd2zu{}4Hj;8Zw<Se<PQ&eXrDc#!qpe4SSrst^EFt0HAK2D((gueL0Bpp8$1z+?lmDmw}SDEaH5Foef(%4(o&'
            'UwL7;QLPoJg0-p&!&A8@dR&NO$bD&b>33?IS9FeISFO)gi>PuSU0>l@>o|?l{+m<))zx+9f1B|3Vf(5g=tY7h3g#gzZQ%={OXo6S'
            '!-8P*fGbJQy^o#pAWoI{sx95E-CE=9FtL&-TdX$Rl`Z>!xfq=F58WQsUgIuS2|QUU>oe+UnQx7<S~Mz`)1u1&YxvrdTe_W9(VXm4'
            'S<Qs(+UNB3OAQ9*4-(n%SY4q4s`Hd-l{tv2j>KnFh$fQDpUDeyRg@&s_*$$bU8tm03TCb^AzfE_y0WbFXV3HT0@NGKeB;{GpAW|B'
            'D%P7!ss`84{#M-eHBHMN9ZiY#Dq*ULXI91R-?eTvB@hP+)!B#^Z2|xp474U?7ugE6DWT}Yn9r+PP;|;y1!Tf4y9$5K%K68Gv-N2F'
            'Kb`(^wK!8%susCRwFS8}-TvOyR>f1L8R6Y*SqyiatF5g{-fHpGEz7>Cc<Yws5n>!Wf`j$MY+K^T`ggGvz9SmBRl%9%FY}#d)3Sm)'
            'gMu?J0_Qf5N2Am)XV#5kXPbLBZiDei*BPurZ_&haz9UJKPHiJ*KuReH)p=21*=o$OcavzAjD<J13cUD1s`_l#l|>pcheK1Vq^h!f'
            'DJ|>wl-$<Eik_ZbV7oGhn66k#uEx2RJMyAP+MVWJilOg1hMsGnN~PfdFdhwq5>SIfCP|AJ#G4gmLG(G}X}hhn0dU6FLc(~7&<%<x'
            '<qVj5u?DZ!+?8zZ&AO_P(s#@Vl;o*^OaYSdI0;5p7&yIb7HgJ19uT@Fqbeh$s(ogcgwO5YjUlP^pY&2MxA81)Up@S0++<A~WRQS%'
            '1cSo6{KL?oxTr$g9^WA8a!iOud`VdNo_8+@g~W1!!J{zpHWRAJw#<ypD8}pDR-<go#Iw50LQP4>A}aPVUl1nFCCd<Hkp!_=R(v>M'
            'gWQ>_#zn=S%!*67JyaT&Tr$4t&r#*r4_U6);a?>K7CvOGxrJ?`%Z*%2(*{RMC|epnw?`@jwNt^X8bq?E{73hq5=dO>!=aJ|Nmels'
            'S@Z*+EVOb2Y-z8~KvZg5QKFlcr<f8ove2_3j~}-y?<Cvt^;TNNKgy6Pc)IA}&>%C;T)RMF2|<ds^D`qrfO?by1?2s<%v{5!8sa<}'
            'N7TD+VK<CPOQZAD_b14WX;Y}1_PJJ=BnK;}Q7?|sG5dEv$$s*<{7wwxL0yUbn6#+^!KP^Zy44FH`q^+gOF)C9S%@FYheKj+SMf04'
            'T9rH=OOn@YaOoK|p}adgeKUAD81@hGua`%s2XBu+ci?ge2ZLb`b;DA4f`+ZNmhLBGN)L^i)aI_N25;2dod!3iS+#tBr%|30V{B0V'
            'JY4~=oV&;%umBrSSR-4yRgZUr!MLa%NV_&4Z*g|DZk{aCw12+DWpah`YhL3u-z4E)UDs3|zSi~9L*R_~1~@jC^1~}rN*9QavHa$a'
            '$l`^&kpIr$UyWsaS<(^jWO}rCIt|9aihz>Sjq_7+vxJ72-ozX+y^WXQgiMlX8w|2k-3^I7OaiBT7jR9KnOUIojxW54E(Ibmw>rug'
            '@jS@#$~QqOlN*v`b`RrcF&aJNL=Fg_(^K*&;hrQQQ*RCu1?KjExuZuxG!B;&jLoyemd9h}O<M%-2a9fagY=M>Vklroj`!T>vZA0f'
            'I<hE%-+Ggm94KKUXQ$BtxwJ|d9NM_Sg&GFE?*uPh7E6?S*n2lng^p4m*bfpc{thNB@?TI}^NWVT`a3|v5S?5o|34)sBO-}La}~-('
            'qy&cZ3?~u=gYtSfz_|37vxS1ViH8OegmgxFK(0k>L$n=?NE?L)Wzg#Z_yK+MQ^Ot>Gz$7xZ)t&cm?|!90-L{Fz?X}I;l)}15NT_^'
            'Gs<p4W!uYB(xcCVImlM4g>-}Ru;u~RTHfL0VzoVaUJZ^xx1Omb<n+hn4sznJ&(QG<rY-FWk50YDO8!ah!h<r{pHrQa{OdprV1H*J'
            ';NL1vI)<F1HwdUBpNGp&D$N>KOQ9q;4_9hML^zAk>~QvvCcU&IA%3qI2)SFjlwA}VIuN&})UaJl1^S_7I+?fFJJX#;EzqS>ze|*C'
            '>@B)K;g2M`DO7nfi#gq+LRGAMq{}`*w-<Sadc4_zCnWN-c*ApJ7qkDs^s6u#%HYqpQ`azwYx7pun02)wh6>G~1r>xsS(_#A0-_OR'
            '<E%0p!`^l2e}M!hfb=r`eQ`{f5CchkFPcxp_umT(u8xDxcRN3dA8@b^kA&IU#S?vWdAvUhBT8CrK$G4TYS_RNb)QZ;uU~uHO3vuG'
            'jUW(=L-Z(>ES>*wryixuRSygLcs+yHDxZ=NzTTDY*SpjGHkZ1OdpYS`ZS}e7%o*m+zKEH?w$)Hi@^dJmp-$eK5z8*Za<tD44yg+e'
            'sT<FX6InB)bXRhf7ux4KcO}ax)%V;eOHpoM6ua08LzDw4&vY5e5}wkaBgMpPc_x~C1X_lX6xjex0xblJuUzAl(U`^J7#x_TQAHUC'
            'dm*lkEpM8g4L*(IQVG4uHEjEe*$L06vf%MGmn18c;XXy77EY4-Tb8{=E(jtgOC2w<)-x9I2_kg3!h3Ws6Cb3UE)p-YQ<<r91lWfq'
            'zyYht>aLO?%iNPb7K5A__A<0|C^FR+J$bH#wCICGFgO?<{VLA;f4SgnW$*`z3}<~16i&{SKt`nugpBmag{7VdJ+5MpaxBajWqc3U'
            'z=k*`^elY&cR{>lTmY3|8AK@ebEHJe@CxH}^W>_;nH$-XH}xVaVx%Nz7U424jBh%6d02T8bMxWek>p({Tc|h_b1Gy}%)eS(d6JXd'
            '=LIHt$U{9Oh@n{LfKT}gxoA>JCl!$pDo<bK+0xlHYd<WjO75`Nqg?_NCqy=jcp-v0*~U<dA~$R)FOw;-CXI$F4%(97PGr$}S)3EM'
            'AO29f3DfX=V$XK>cAo7%)!r#N7ouV>(m1DGyzYUh`A7y{(0o&Kpl906r4E)Yea)fD)DW!nWg?3x`ItX`_@KvByG^TtfeKk~88=Ua'
            '5y&AlZ8xT=j%=z=y(*>!-=GR%c+OAy+D-n>mw=RK=e&2^w>?qiwwGZmE8Hu46MW!&c>3<VBNHu?CDoUwcSo0V1;~t67{xF9=i?F0'
            'V5L<D+Q@8p8Jxv!K7QLRxgug#>SI%Hlu*&{Ldh`rj-O<16uZ@lCZ&Bxu|m|D)K5eSor-xY7asJTTlF;^+kFlHXY~o43&%)Xy_MI%'
            'Qyz6CFT(s)x6=R3&i14Aw=O$?<bsm^Wk^fvM@tDt{Ey9Is~w*zpSCK*qHg(7Vf@eXUpB%FrQnLp$qGfNxx7;-h<98J!A9X~ubgwO'
            ';=WEf@0V_1j8^I{rZf;1@{QIHQ{Oqi{j2J%-d1#1E7u!P@ce1;P?YD98kFIncGyL~Sa{6_w~F^Q`QI9RXe$70NHpcQkEEY7f?WAr'
            'k9<t#{&+jCF<WYe0zx!ip(u)J^g6qh_p3~Z*4?2Nx5j!cEzIWZv{%11rH3u+2Jmr=F>lZ5jcEzTV{)lhK{gRkrk7-h<QQInZ%uPE'
            'PVVug74C}*H<8koLB}n?8_5^q@!Q|hyv&Wd;_Los(k#?GV2axkPb;#}rBtc4l$r-##Ni$Kx|0BpS0VC|e4-_LTgf%`TdPPKFE_u5'
            'Jmz%y7|sAKe)s+un1fw7`q9YddptNlADq12WM4O_aQaO*g02JOG$4-&%l!)^@HrXZn}pTQBdJ+)H!IyvxyO7BhnXPH7*@Q6VVIQ3'
            '?Ui<)DLYG6u_aOEWfRZJbhny>G$>RdT)FZxjeK*l0lZUFr<=1dOkFr@6$x7FT$IYusdg+=b$n_eQIj2}>X74XMO?fbxh9A<SANBJ'
            'q^JQPNBUB8%l@`JE!EuOsxIm7Bfa$6b;6y3BNLOCf(6FP8jy?O)vxsqS&x+!kS!2}kbmo?jEgqndOy4RjWLH(h&h|CG4qBpd=v<1'
            '$pTUlBF^eVxAx|{STtO3D!s2s{z&%pa~kQ48?s$p_9~Ca<`c6T0-IY*D+@*-<!@QiFFJA*XIo5bR-*jsKe<y>gd~`*N_4n9<+-0y'
            'h}&lqZ~F|}5u97~X494(nyBtv^XNo7F7eA{d}YY%+6KMy(8f74FvHFTz<29)IYPgHVt-xAT}{CWk~z_qJ6)vN`5)EeJ~(Sv%`>HZ'
            'FmkSIo13Bu8p!;6SNS<M)^;oiDfnZ951GPAvmR`(FMjEt91Y$I5PA*=``Wsf`8NTKI02%P1s(BvdFMxAFr#=R_z>RMdg<JoaY%fl'
            '_5)J}(@v@?WQao&GX`gaqvQVJ04)#`3Yn3c^iVz4`mLzePu8=Tb>n@hSXP-gCKu7F4qBcSrgt%X)2fw0o&P<8>(7)_EB_xJC05O5'
            '7&S8!llUfCQqqW957QrDJ`^&hI{T%ZD6(h$DQf8BlVU&)rd!3x3ITebH^T}xowY~M^xnMn!v*u&ZOOPmW88KgL~n)ux54i`i5d@e'
            '{h6!0Mp@&3>LJ$L^5x*@2o9_Jmy6S3?|;ULt?&I=Cs`&WEXwQHv-EJy2qk?04VA@b|JCgLpM~_~rK!+Rx+AZZ#GSwLICv<JD4HWl'
            'T0?N81mI&Os*wC+0GZ${c&a;=!lF70d3u*M!%dbS{V)p_m>4_pGElo#euPzDftp4BTnZE*N*IBLYlh4GS>h!)1(sKCe4>oG@oOzP'
            '{9j7<HN=(k{@E}6GbOML2gm)>iy>E{Dl6@Os&S{NoJzTSrRI$PRXI<U$pLGpZXs7E)j&YaaPu<Ux$hOad%*u!!?JAdsbv^;D!~7R'
            '-seGYaPmv<XmIEP=DoWZ67(ueLVpr_`!!!GwXKDfzMB@`ZFCb%h0p2ume*m~m|E2)CoM2natgZG-YTSTlr9@~=6z}6sLd%<iofvg'
            '3x)AEGILa#IAjg!FT0?$R!~T|w3&&+`7bvw*Ax&dUr^;@mi%*LS?bGWs4tP79;#vZh^(ZV$xF*Csx5@}!OB+F6(+GS@4m8OMRHp^'
            'p_C;*74Kx3y^@Kw4iAt|?p2Ov-`~=L<W&p$hL!@PjCW<qquryr;?k3zWI?nJKQH&&dspga(z`@lySmz{lLm@+$t0G%P{EHixg<V$'
            'm)o?L+gA@D7<qKlu$?0=Xo!t`u4)Noiy~dl#?jcH1&kZ$c(3m;Q8sIAJtj`W^0qb;i5rcrZy{fO4Y{gDp89Mm^!qhFoBWiSjX{iT'
            '#<aDOw)ce;>2s;kL;3Rb_iS^GT2_XR=_XUJ^=+cHO&i&AZe8HZ+QF%p1enjK`N^P9+a(OyzH%O<x@3ibH^#v%3&h2*hZkoTN5{Rx'
            'L8oElA^bhaTtMQlo)0beu5C*{J9$AwA;jcf`+@yYr5oLO|G0ND92}f~ll*8tM+<XlRZfXWF2fYoYa}$nlJ-E-Qo~*L-_B1@WERC8'
            'FG;)=Ik3i4iYQg_#nKZujptOWC>8gK$G&kv#$+{4DXE4huCK3g7GjD;yPb|W!yCC&RJtRpBDb>26?^W9n<cvM8u1w9)*@G<v*gOZ'
            '%XxC;6=2o(nvGGMa#3s<G$SKKi=(k$0+k)dV4x1i&NPz-tgiX<J}%Jt`M31@X6J{W9e4(Wjq~+E+jH%XLj#m@{L#}62J<YMdye%*'
            'Rr?QPK$-z@t$-9)n%fi(b+hpaV{eVWFB=YBPzD&X*4}gSM3eA`b~EIGvB8FbA|PcRqsJ(WZ<gq@BcbNApznR)i6uOf2R$?S;0mYq'
            'gA`0K*a*7)saq^oxON)G=V#u+i{{=uNY&oi4m@<=jY`@$lS}><eE7=p&Z)nZT{oL4Psdv&WC53A0SFy10xX`R^o<BxeP^t^>(eGn'
            'ldU{a2q(qtHUV_wTYo&mpN^3kl{*^LMf6?`F$$Fe({?1K$zMya`g6;{pM8UA^62q^net74e|o(yu0J&}5d?VwHX0AtEc&K0|EP@8'
            'gZ{AgFvL8WA)Zrl&GWvdl-w}z-<Nc5;wg1(MjrZbgWNSCKV1m{hkc*p(1EW29V(@|E9A^na4Y0zUB6o@lN_%+@>BS~pe3+GJ>J`4'
            'kf54Q$8~X{uKOU7%gS6@*exsij~mIDKOU_r#IJ6SG8eZ%^Q7=+X@JQuXv}{}9*oGoI}8!j>c}OvxH9L#4UGIoX3cea$(ik>=X}N;'
            '6!~$!c+Nmk%QHsk6*eBh!<b@F#G>CxJcA`7S~N=tQ(tnu(z9ay3cLu;E_$ELF#048#r7wh?!z|Sc?sZ2)#tj9KC4u0<nunN1t|Cw'
            'ecsNBA~jDLY)PzLo^$eNd7OZlG^j3us6;_4h*p(!QFedPPn^vxYlRi4>02OBllcY`TrbH2bmAoY7%ordu@!r}-zbOet@q@|=L{~L'
            'oLHxPc`TVQOzk!dMjaldIl@d8xBiDoa0ALj(^;EpxJhA~qgCdo&KKnz#u%I=!#Y|inZ~N}XRgU=Ofms51;DvxW0_6cKiP<0SQ`Py'
            'TAT&IDJU;Z47#!b$c^D^h0*v+6v0RCgV+;OmyO_H1L3#@av1t{ZW_w-g7hi?wXDfGy!U0x4roY|=JOwaeEJ+t9BzY60(crKf|^FM'
            '18w<zd37=yhrHRA$k|i@2B^lU3|HT`gcXguAP?(|>u#(&dTg$g87``dj>!lc)#TBP9I`n>x^h5PD>pB2CZrgg>!Ovz>TFAHx*TDG'
            '3t+l-N|3c+pv)ES?Ggx_>n%`U#Z*yd{I3j*H`CS2kFN_T^-44a(TQUsS$%G+L?d~0B}YMm>V;N(9b~GM=pDiU4zSxn)ZRf8Vq2yY'
            'lg_xv(m6Q^M_5uec`T2z%x>c(_{f6eEnqHMx;?`;5aw}lcGP0k4Lv%2%~hGuvk*sdl6ht{_CH0@A-YL9X^0o%!B~lV^q!Xyv>b?9'
            '>7#h<$c0lem+3O%NGC4DB7CfUy3IR-`L*C<XjPbSE(@2wwEco|WyHKRvOuw@N^?~)yw~Ir?Jk0akLlt%YPaPN<D@y)h5T#Ep;d;V'
            'byYkXidH2*IGs9Ao>enQLOD9m8jrI@axGmm5E?0^`NlC4YgjYo)XWli4Nl%<fo^S@X4}?PNtKMP*(vXf0OpbTbN}ND*a}Qw1L{&K'
            '?;jv?JM%^$5(NTusQJVb&Dy<jP$0t~l2T}ho{<#>W*8pJpwurJnDqf+<*xk346p*lauEknMutU2hmH67!@3sLA!{|!XXPn<X#iT?'
            'NUGVUv1oPUd}dHu%~<8872{RJuvHwOVk2iwj2kH+<uUALj<YrjZcTR;GgTF1RyfP+Le0#-DW6CU)u<0Y12*6>Od9)n9daz9WK{&O'
            'E~r$9nJa(YeBZ*DI%i#cU%7uVPu?ccf8|kXnmQZA2tGa-w=~DHL@X&7;bs}Kztx0Icjl9FOmSsEW%(PHFh->??FYqSZAx#>(EfsB'
            'jJ0--P9eIpAfV}kReoh1rNjrFGC=nv*mf5_6O3_dU!C>)?r_jQa}RsN9#;Ar9Q2KT)p6cv%F=F5YxZ1K1|w&)iGSlKbj88-!G^sz'
            '$Ago@!C6Zc23z^r3@CXjlgv+$HL2K5r5aUBnR&R5WL=jHy9FoHve~6(=IWX(f5e5-<Xyae=#>L*X&_u?TG76QOm^RgnsiZW4fP45'
            'REuw*z`yg8cFs~!bc`c3XoB!P2Y^met<bknk(*1!9(wi-f!yHAf-1OS)|b=wCuF{w$a;6g<2xX$`YX)pfQKdVQC5e+$X$j=MPjF_'
            'P`)-4q855%%rb$mw09CG%|O{*u?859Qmo7uq0hBRzlkM+PPvOBhs^uNN#{tjtj*WBQR>IFR;k*Yk9lUZZng6E{r3!DcXsb3H>ng~'
            'WsT)-O#vug6kb47zb~)Usm(rEHp`Tobj#0lYCR{ft%)Z0x->ynA~rmbI4ZBhG(B#f#jsnE#jr5^wDGAt-otOvZ_y|I@<TCY9)|2y'
            '=MJq&Nx>^DB2eLPd($h>u>7bBp-NBeadS-5zgm^y=A7=1wy9H-Bb0ANyZH(gRJ-a&OG2#6|41xQMoqk9f5fb78$}DwI?Z(%6;U9x'
            '#}Z`OsveC60$Huj3Vz(L3Urjlc<WGF^(Cua<nWf%h^bbkCF|17ai%#t^_lwJD$xX0oN^hdk#6H`9?#h)1GRNY5mYwY?`6kL5`;~I'
            '2p9_t*ya%hk0_d47p>PS<kak<fU}${BgdIP_Jcb=6*4{>2tfjWOOIK)bHfyellVRrJ0cweq5;|R$RR!*gS?umn{wniD)<&vqgXUk'
            'A0}{gbo$;s>%Dg|G_B_jPhBl96%w>o>{bC$Fi&NJ*9pJnURQKuWWv^4c}8Q|537oxp0$G460H{d)`gx@srfU&98d@3+bMJ=SzPLO'
            'R}Q+PNg2_xaL3?ejpBAU48Hb#Es`+w6(z&WK))AD<bLKOYAngKvTwKv@;df&_du>`9z=kR(M@)1Z303UGh)jfP~X7Z*V-Gw!NTCr'
            '9*21edYb`#v<zpi40x8wS$_xEZ-blL_FWt<=f0W$3UkaU(7!An&>#W|6)I68LEnTVz8Chapj(iI{tlN&AiF3TVKeW>*x;6NjyZu}'
            '=zGy}0aR3KuiA@Qwl}6pchkTh?$94Njy@Ab>bIr>#+z-<U66P{`GJEai^R`<@9w~ZgR{Yq9@>MVlMy+b<DV%xBMADqncqomphJ1%'
            'l)Dhg(`l(Urc5O>Z*Igb!KG?>n*oL;2C5z}{ef(#usD*ZrB_KUfBZ~{Rb~C<`vQI<L2{tD%S9?dVJ4iD$*J?+RCh-VP%xRu%y*tR'
            'KL*YrqUC%9V8!i-7cv8-T7<4ug5}S>6$>=U9CaB=JnmtiU`%Nhmr26q7K2D;!Mm5@*JzoMrx-h*ct$-&n#iy5a=0A%cwCZTOp=NO'
            '!VnK;N~zN@?Qfv+yq$Al-ix!NbY~p!E~cbB$;)b=Z$)P4DIlk#wbjS?p%x7RFY@b7op8*=rV>t6qLS&&s0VkselYuFsw?)=+ook('
            'H66~Kx^e%$BtX@SE8lKBu8q^|I&VF)S}QQX`#ayu3XY_W*9slI1OtrEdx!mlzStgNsr6_HXo0s@x88`%FB3qBU>mQVmuPm;S+5hz'
            'ht$HI8Ch*8Y|uuakMOyOFbD(oBC&#BSZuR@aNPe}!Ryra95vl5D|W~9d%(jT>ay9|*D^J5mDowDTr971C|+K5^3fIBP*n(Cv5%35'
            'o|n@YPdOp<jlNR!0&)FXbw5_U#SY)V={E;>9sS-PT?_PmN=5TJ|0WkVfoa_Pqs}r3b|mME=Uxz9OCeHK8@as7vvF%L(J|d&Qd-VD'
            'Rk>P9$-&EnT&>rGtT!W}4Sdu0BUv#Jiw`>MYlT=IFKg?CXg;k=pqPb1${W)t74KAU_SaW<v)l{hr*sW=^Gm!P)g^CWhm470L+A<*'
            '3k7dr6Z2;98f(uaLAfdpP9NLCm+&7<T@PU1f4r6(XQ}j~jyIV!p{qTz+_W_zN;sQ+{AN|i4`Ls@wWiMxE8L43jHbo*@~Kl98YEFn'
            'PP6u;YkIl%q+{i<Yxt<lvafnw8x?ps$un}tG+m-EV-;_b=#E~{id2FO7tE<v@Q8B1j%7+<%&1D4m+w8GG4xqKFHSBez*lRRAfQ}r'
            'P`+vN1A{~0CsylgI8~h-#?=C#su3NKlRqNeE7hV(`!8Q8T%%6bF3SeW3@wJkb0tR|I{PRI_22a{q*ZF&+5{^FcANn26L~a|V_qVp'
            'MfG+6cyKasVMO<r)4@UiT(i8!-~7exDxvZ|%kd2VJ-GD}k~bS~@fr&LedR|<Ky8lviHHB50>8t5d$F&}>^2Snb%G?hjj2;_nZ*(R'
            '?1MHPdK2z4@D_f?KQHDZSaC?L!<)b#d1|Ob*kaIBsWWgv(3xc;KMF=vaL7wN_zN^9nwAGGCVoI2dVpj}<RxmXUP#eV`05<-&!d&R'
            '_QuKb1Ft*89tr4LvU`w_Pr~1TXGcDMb2$psXun{2gL&WqC#T-$fHLFT5q}#02to1k=@aM6(3^sQ(b{shRN{TT1sLQ)@U>nY^-f-2'
            '^j_23ENKtV8e8A(i^F)lWLvwEnGrYGgt#xBJQK4wA6sJYd3zL)!4@-6d-vkq!Lew<?=hGA^Aq5Z_>uG8@jEPSjHuLw?(XjHuKD8q'
            ';Bfc`-_XN@H@%aS{t=?gUSTpg3$i1AP_`ldz`^Osu>V(B(dpa%2?t<9@25}bL(;yls2Jh0OOAhbA+c-;=zU2qT84j>JbwR7<=uh^'
            '6}nP=H-79Ec(LgONewn+Vm9mNuE;*hqhocMik5lNnbKs+LP6PglDWU_J6SP5RBxX@>x`a1oA~5RiO;LDyAjFbZKozJn)lYD(H4kv'
            '6!-!7<|)uKLA)c;S;l}cnk~k2H;dhS?@po^JlGcssXU~hF51hxg>!*MZS*n;)2?|2YFzf@K=;8UyX_Va5{D7s7d0n-pLmN@A5lO-'
            'Pn->2z1n%-`=y_sRw-B;Ctk)NNiR7JZNLFq%#5!)u*z%9AtsA=j$i{&8efFo3NM%08%Y)tpcn~%0orzck;fjWZ1+LJk%71XD$=!s'
            '9{|-DtyQROOAkI;6R6G8DOJD03Ij2f2%|aJ!ZfxeKvP0M4R-OphBN#$!I&5QbG<OQL5X@@ONI#IUjrf85I@LubQU0nG#h^!06~8G'
            '93GS5c08x3pHTxLKAmR*JLDCa3tiv58NmNHkixd`rdCbF!k|{Z#%e4v7Spm80PFe5vA#dzV_mrjJ%7tPnV0_q;=3;W7)+-uqPuDE'
            'kq@FqeSVHtC+u>KI2uH`tauJRIKYnYm=UN#it$sFm#`?&``*#p!O3ebUO);13UU%{-@o~l3@JVFYVg<op*Vf@>byUsS2g#(Tn8h+'
            'BY*a+jgwQ00}_ymYgQUPR86N?e&OkJDm8}%4oR28t8rW|GYJfeU^IQU+kW<pF}NdME;IHB4W#V|Z=*v020jXBB}S&ouHXsTYvfoy'
            'l}-SH=mX!l#>$7+JJ*AgSEtvi_$WK>$Vu|NrqlVtzY&udBtKA1X_C4w-Zc1tMG$x;6oNVTJ#=hmTiN@wQ-Cf*me;t5feFE;Ok@EI'
            '#(mfhppq#W$0+hKi(`?_(F*{?$P^AETD)3F5WHpT%TIHqab1!szxLx=zyOGpLaI9Ay%)}8bw1i*RiTeSCqt3d2H3}5E)67yCtG=P'
            'SNZe@@e_vHAwm-eFm9IFN(cS$#U;1j*WI4wK}ERLTs;$y$ucqyiqRGO<lTR|axQnRxFr3(ZhkNd=|2uYf?Q~I2E)jdsuZKcJi5Aj'
            'p!qDLF*!~Xc<0MBlbco;4WY+)8Bn2-TzIDiArechC(X(z6k{BEjKIX^WYcZ#wsJ^{wrJ+db&R+qSO}vknjzsdOxi?{P}M^w5{Y;+'
            ';W><@w>P<lcoUj#J`FW`7xwT6?(l=4*@%A?dlqflEk+u$#HT|f*eV@F%%Hn_yHB2>Yx{zIA=mKe3PZJGojZ7V$2&Nm9d%ZAfUdOh'
            'z7HAz<D^?DX<}yoVAwWb2a<ffqqsua5~6)_J78!0aN3d#G4k=)0oR*OIl(iY9OjZzXQs<k8O2L=e}b&Q8L2c+7rvL>a>R(NW{h`c'
            'RI<?j07niTqVR1eNEvb{lIr`*+~rCO5C4CA*V^2+k!8Q@S0Km-k+DcgnTa<e4Ku}z=vWn5a!Gc!yjm6s5)j4Va{)p&t>J&)`|3tF'
            'x<Nhc%*;-0DwT+2qtWlak8{sOk*A8IE~L@vp=^{Lz9bWddfpiBo<Vs4KOj23n)WVKqugV9jyqlHMo=ZttSNU0`$6E)lsl|yr}-zg'
            'Z=*U!ZhC<{W@8Pm7rC5^Z$K%^dP^i$qVn>?#tcW|;l5U5-F1w7*y=2^&~4(@Jd%g8(cDP{s`6hRBT$jb9+y|V$kxaex0vP`(Ravb'
            'u`Ep!j_0b#u61)LugyVI^ect1t$L)_h*c!`32s-boSNTKu~xA&i+r2LuW+a0ny0h&=#O1IL6roizyI;*@O$|M#VrHoqh4;W+O7n+'
            '6c9c>FbLjTDSh?mUz}^teV`>cYS8xwM}K(PEe=Hjamr2N6`S^yL}on0v!GCg_oc;XHO*X;y7krkd+vJ0Zw!i#V%Y3$OpXk5uyD{1'
            'X(D)B;L}UDn$n)9T1*+e)$$(4mU-DMPefr#R#z`CMWCq!TrFep_XEyTS*ah?s2BVDc%|9ZsV!k08wC%C`r6@<QA6OKUih$XrD7rC'
            'GXR-P5Fba+HYr3oD>=E1<AVQq&Us;+6%@N8Zzmj-oF2l0n~lvl-Kr3t7Xccjn5c+vJ`m>bikRmnB2b|x>QKC^Z&Xy{a*7!d7)SvR'
            '_ExK`CLc9@hq#t+;o?+Y<v$)A9JqGqE37RGF3!(Bu!lxr2AUxL3o?bI^BWT8S15=^0cIwdCQ1ik5Ub>8p3NvM3x-7!{28cLGEu>F'
            'X`{2mdz#HxS=CLN3pwaG2=>n{qi>G?2g`JIa4b!a7_nI`*fZo=^a_PV`uGg>Ze${$0RAS8b1e;PfBAFky7|pty7m9gUI=eT-HPbP'
            'wn?pZQEF9PL}vj@R=1cpBo{8ZeyDxQA}vKUzQ!)9yP)2)V*>MzDheY#GJn_v_v*<_SRU%qtIZqY!4@hUB$eE8ArY`v`8;Pgl(}F*'
            ')9Yyw^lmX8>RhglxprIL)@VM;F%D#CB!l%Cb+^<pM_L;?f#Pw>IgE5#+0X?NJ_e$ZEt6=%EQ-pRhaae-nD>MVZ$!dtwBpcGB&E4;'
            '4`5%Ze4K=ku;vVnF(M;bu63TAE*8s-a=l5EDL3cz{Oa_iLwmIEgID%-no+TDVMrk+o)-$p)khG-L3HNY(xgHCHCclaL{Ty9Se+FN'
            'XP3ly9GJ=)j*QCR76X;YoVo|^AnM6zs&co07XJ!$23EZj%u9*>UXYT|Pot+%n->O{lN7Jdd4?>O*qD3{(n%tN|2komzRKcCmO*{m'
            '$xDwEAYZd!Qurw1S@<}NgvF9n4(=UAnsDbNK%huRoyxe{*-@27%SHv-atQAC&njl9Os3-jp^c$6)8uA_vFePVX*4)t<1N`MRAeg?'
            'n+YY;9%;FQ<TgUGyOeXD-f=R~bwGkR*&J}p*_0Bwm&152;k0_F1b%fJ#XLZ$mravd$t)rJUg=%xbKUy2xs2$vnN46%*7!rFXg9Ha'
            'Jz?SG#|DcSb7LE)OvSx|cPke5C>auI8Ln2D1jTyT>561;ZW1n!#!(K+BYr99%`iHa+i{dy{La2gDzfxdMuEO-U;Wy<ym<evb9GI<'
            '>H0k4<*-kx(N4y<HSb*!d_+R8D3&U4h`uXWhqT_O=7A1;<+gP^q+7y|Z$b>lna`3n?0myK4>B1tvQWL>q#KY|C!d>c&Gk?~_RBQ6'
            '8Gphx<2nA-szI?0BOOAL%X}qE%S;5sP|nXKxlqK%PKum(*`FHiAvHA-II0qd3ap9&GOdSV5o|A2RbYaQ$_v^fdu6B!Mk?I<)M~;1'
            'LxnKoD`q^r4%_cfPA|f)Z(A&r1TkP#C6x821qiLn9ap#c*?*yjv;d5nVfQ2$jI!!Ezwj4mV~3CQVY6{-Eq%4|g@YJ=RM3Q#4{)oO'
            'wTurk3O$}Ph#Ycwq@YG39=Lch2i+;o4SiWJ9}(q@cli?UQ3H3L!4RYjI`TS2#iu@BA9SDCHKSY5%g~L#DQez%Zp4W{_zTy6y@c!Z'
            '2BOGy@w7;kQu;Z~z4;B(K)Bf-jWf1e&g*8EG0t31Nr$=h+n~K#%*I2spkoTX#>W&RZs%x?<-0!1(>sbyL0XU~yQSRRQNYIUh}d*k'
            'LXi4crU$8!1x`+_&IaF)i+qYPElEUZU#dssJCp{s2Sa;|(GFL3B9|$c6;I=Asl*B`LlFHt-O``5n9=0Vc(S96QSdy<^GzH8G40j#'
            'TSYj@ngT7a>GFjlV@GeXfyHNcw*{Vx8Ab!`wt}T;E<Rdi=}_YW_*aQF)jV9u^k{x#mGxYF^hCvls281H;Dn)LT(pD`Qzwm>a@ue^'
            'F2}Zvocok#EWZu<o&oP8vmeN-;@+c}p#Y(CX<-FA);>(7t$d5P^?+hQv=^`NRZs|B!QI1pkTO{{t!_AGv6?}~Na$+S2*S7AZ|d@`'
            '`J6p6OJSEO$jY_JR<9C?7Vj|;kb|=>1gC<p_w8lpx6{sl9<zWN5cw{gyolmwcyx)>#)3sFyOr+SwKBtXw<ohLr<kcG95(A#UmMP~'
            'Haq@UL7_^`y-5B*lKvCcmpIs(NNh1Mh|VUXwi$?Q(FEOr;-|WS0n2Vtd&kK(ym9Ja$q^1Ssm(Z%udSav<}5Fr&aNfJWA*z|Bg&VV'
            'PaZIwXd@UI)M?mE<D|FB=NMaz#jGKljEO-tb@6fT#cV1AOrB0)UQzEASs|$Ptzl{H5*To|UsvI}uroog6&*&Y|AqOXP$Pm%1v6qs'
            '<Yj_%p)0hgu(uy?Sz5JXD4AOUz%eMa$~%Maj+3Esyaeps;9Iybv0H9d75~$I`}XqUxAxhQVj3&M7_oW%(&p}K!~$7Spzw~?9N2>`'
            '#|0uQ@J__T6E+GI7~iZFVnGIeu6hJd%E83EVSUZ%L=u7~x#*)&iGyIX0y2TJMqSw>G$#U$7*54L<ERk{&4!5a9QOE_efv5$0To4K'
            '$8OOWv6<MEQOZOWA^aqREtby}A4br(IXU_aFbbB_JX4WPJ6bWIcK-v6AaVq6v9A9!tsdN}|D$FPF<c1LsP&*z#(VhH#qfa9U%hA8'
            'AGQ@V?G6v{UKq`+|LfRehKngPaD4@9iK@ymmsrs?Jef{v@8{T49<ZsH6aGTX#y;S&-@fuZ8_N^yEYG#I{66*;kIlvNmF*z=68p<z'
            'Y%pKR4r4FQCm3T;Ae}}8XtQGJGRcCaf@i+OXrtPttTuyC+|~M*`d7Bzc;2w{b$CPmjSV)>u#FV<kZRM$|646wtA&JEDpVGKtosZb'
            'a-~+7HmnQx%PbuKC<_25-l^97gDY5oMvyhK+;Pf_ahfIjDg`Psz4BLqWX>YSSDgrTlnjy8;xo(xraUz&Tqkid!4pJ;dgF0SwEo*='
            '`|M5ozy4(`5Wb*rroci3gys;&4>nh~tI~g`zcjhATY*C<TzM&b2o#w&&DEwg?Wxv^8bgp)po^&BYE_6WAZ9R3uJq^j^dn+~T=lUg'
            'Ym}E4*|z#T(*6Cuqek=#KYGM9k^&zgt>e|}hgC0)Q>a!7S-545Fe>yxt%2GK8YY@3rfUmbvq3TJr)V$O?sauc(B9HlkzyEZg^rx*'
            '5&T-$yCnAQcI(C>ut-8%rY^GMrDJcEd*?fi3=N7q-|{kB!)P%>SmlVBywv47Wrwv&)Y8UslXHoe()bw91vrs0TXQ03OpL4!h-+z|'
            '!+-zUIqSUXT%86#cYZxP{cv$vPMaKFXy)nXu}ho1_IKJt<Lld)p6~W{Tmoy2b=~p%-OFt-el*EbSp*-)o+tR$wF!LrW`)Ae#M<{T'
            '&9BdFSQl7b*a$K1!&AA^Hy(I_@ucrscc1r_vKW<pH@91jx0q4Sw;1`0dcF~V!igKu1h5+3k|k`FHTi6C&s)9B%sSyLLs}B}HHWpG'
            '?FPF=c{C>VsOlt>cI^fu#OreK1GM56#x609+1B~3t7=w_2EM{K>HfhG8`JgC;mfX)qpAm5S(2>J7V$HTPMqBJG#$3iIaLaND90(('
            'YW_`N2%RMl%Om_QCL8u5TlMlR?kOdtjEE^pFq+GWoe2LY$0K~*BhJ3wVebdGhzB|IaJS4hu+Zv0r3gbTrgc+_EGPMOBuGz7N+oQj'
            'Yr1N)p1NrZ24ia~H(U9DgU)P_#4+WO5aT|^9zBcNRvl2CH$QhyPEOB%>7AUu>6~AkUYx__{U0xX^ptpWand;pm?af9Pa{#yX?!2<'
            'U_`$AJvqqdi=dBpJOzFCj~O)ypM%jN(E&<q8RGOeJH<Q6kO@~F#%^9|vgc6cTk2)w*?1bIoHQVmR4*zKT=8S@_khJp#hx&n=Gh8)'
            'jYh?D#dl6|hCSl!#=<&azK)p2MAQCzpi20jCbEs{N9kvJZsX#I^{{g8p-d;Y5px<PGSw2YZ-llbdV!lCp`qxtVz<<m6|y}a4Q^+='
            '$w>$+Wtw@GPpjy7a5s)<wNRrPMDK9^E>M{zI96PVjSa=gF48$H>Y|sK<>ZR=q!D)<BKev54Sh=WX3%$6c3*rNt3^X+UC*qGPrZr6'
            'ao7VeA35PWEPg`@;aq}Xki^FtI_vmd=v3M(D&>h+z&(nS1$Bi+Jh*z@Zop+EBvjEymWFl4ESjL0p}Q3I&@W9Hc~MFUUMM014bZgm'
            'x}J1anPki(DUf^+^sSw23v1cpcPS?N_jA9D7PB1W4p68uJ?orE8ult3ELA>}?G6{CAKm(%*U#cs{0P5`h1IM$Bk3|Kp9(}+svbw*'
            '8O9D>Yb!jOs&H=VoJAG+-#MYzEPJxU!OI3s1M+nECAHL#yi^*R-Vu=`j)7yKroXIP2*qa@gpH%8Rl)pZOtu4LEPi`}x!4-4Kquk#'
            '8PV@RfL^3o!B(XlJB+l4NvvuCM~fAmq6eTi{JozhhJSmaH$XQ48asi;<|X^L5l6)J*U^w{&*IZw>Ajqc1gC?=8jizzUHwL~oCfzs'
            'BQ|dEn%!m@dgZye=8(ZV-a-`2#x-^8<!~u^lKM#`8nV!z;8NIeyLWF@wFyS!G#&?VI;wujTQ@Qfyqv-cN0Le@$qzl(6R|yoSC-Fe'
            'ey{9`M^Un6OP3zy&#-QgC%H8&n3`G|WazZlqE)bb&QkgEmsL+tVgA!9!b5?2rvy}ATN!bx&S|#_ioDS#dn#rmxwgeL$KXp}^ZV(^'
            'O4BYVddMzSh?zAEQzpTV_Qa7X0YuuPhJRqQBE1g=+cK?%&TE2b4GMhz8MEt7RuLVPi|H(a!%~*`G)V*6^{JdU@q*$8VcmTSz6;hg'
            'il6oZIw(Jnm)vkJiZWU+GD?5q(nOL}hvfq|@()GHfs|b9da2TZRg%Vlt8E1bJQDmYd~|M5xbdxAn9%lp6mY46@8nTMFEwA>B~ztL'
            '7t~=r-oSg>!rRk_k3Bsg=N$$O1Zv29nz~aoicr+kl_2;tA4~}WxL@RGlp9N=?Zf5)x|bF!vieAE%t^}ZG?eBFLzw3rCmX~$NWmyP'
            'H-nP|wkAs4$W=|%jmCFEp-MSq)amQXgkA2dGqQ&=vIEY6ilim(B6i1+7;1Ry(VucU+2d+$6N>pJPL$>WQH#_L@edrD2EDxw(&0-r'
            'hnNRmClc=I(a5$;Co;7o9&0fL#gC>F`g73y)Aw$$^NFetW4y)D7Y)-zhA~*k2tt-6MY^k^?bM6hgyon4V}Lpdx2fLLe7D5n$u>FS'
            'M@oZe0Lzmr)a~t~yoK+Tl|iAB_I#>JEG|y#t>zRI(@dKHQ7ey_RB63MAwZA+)H24_mRPx2nyWRWCJ{`8nig6%30?$LC(d=5Pc?;|'
            'Q<EBz<#gSW<O8##pfS5P>*}h7B6f=2U~Tq|-abWhDlf%~wejN*?%MbzO_CU-zu4S%!sgG6<%Vgb#v!2%1o#^7XdvNUTx4yKClskF'
            '>iFFUX0qIaV%RCpAXZic8^XlzArO;#Wm&j%7lN98dE`1Aq5!-Y$CN;$a_WtaU57IiHd44E)k>iDyA*>$=s|9C=h=F7{Y8^-+`O_>'
            'ZHR0X^nylkh!8mtK==8LG&sVmfR~%6UU&Jcf4a^$dI~H?6ipNd3ihO|{qQ5Dh<ur&V)v+_0JWF~5HX0liK1OgGwscDZ9Y3R#&pAd'
            'H<v_LwvIkkd+_!MZp@n8oU%n2t0=#j+vX2Hc&^O1Tu#|<0tPc<&V&Y>O1R|ll;!gTn>0_Nw4s}zFHE&%q$zSG*>?u37gC~L_uMlz'
            'zE)HbSQtiE<eFhKavLs~Q5xOh-7a?ISM?(b4o}B~35RWY@#>XnBCAf2h8D`Gs8N&q_ssg7><`c^Of&fSMAXhd*t^vN>K3hm?Us+u'
            'oBdVL_GTBYVb%F3m~~CA(-TRqXanpo-=Fu|=kHE`xorP-`tC#TvUAmGU;g#A8kIh$^Pr#&Br=2nSMlbIVb@{ptP^pvuUv<qi>zjS'
            '$8IqjfG+MQvl!);j!+D+^A`8Dg_05IlB;6HIZ}ft+bP&9QFm<!ww}EEQxNdlDD>ezfcpF8nd}FHJRT)zt)zNztlRG{E<Xs{ncx?6'
            'Bogc+3R^+JBaYU)nIPa~)K6ELf$bMWMc<sdJ{+75Xu~U8s%7igFl~>zGHCktMYU;Dxt=_^C=08n+3wI=sxWre!m72!7#Iw{O0#GU'
            'x8l5x<UjLdogw>r8X<sb76lJw;SNE^7O)dS*Y|?k@o+Fs8Pr}t+M8us`7Z;sdxy1*aQ`Yn{b6}+LZR|kp46X|<e*e6AR%W}D}PU{'
            'J_nCZGXALJmhC>#a6AccHk_sm^3^cOoJ`}N7b$Afah_tPv<$=u5cLQ7I-l^43Rd|9^B&{OX3?bXq!^cd+wyl*Cq~_xg7T-9RP#^V'
            'WP;9lXnWm+%4-?j5}bfX33n6>Zn^)hd+yif;aw&312sn3c)duE|E)I^4~w~tzJ|}?*{lzgkjU?Ja1v(naZo$lQ%k?GeRX>r5??J$'
            'U%lcHc2Wb^;NLuCI^8jx1=@uA=)9MXjn&E<ROCyt7DG2syWfTuYe_;4zep1FK%o@+VCk(YSh@*&wQki%fL}|8@=52s^Wmy<7DT2D'
            '7=h@QAG>(r_lVM)p$G*tjo4z97kjUKAQM6uj^ljBIh`0-JWD?7i#AKva4M|FbEMz<>>S9|SX^`BH9XVA<CX;A$3r`9SZCz|+k*47'
            'uCi!^KK#%@ZYp%A3){pr2@0WEAuicq%Lf6AHvaJ1!F^hw^P*T;Y%Z<VqM&yln5}XOlw0)J9rF998h@20Ndt4zGkYwfp@LB!rLjsf'
            '{-^p^Pt{Gy8xigB2Jxfsm!K}kKO-1AiEEen9O=kOnvz=s!SJ#j5Eh@VdIk2BG6TY6^#9^}R#nkRrqf$$u+1b|&!Y(^nGI><6dAfe'
            'gSbp?aw2hFAZcMt2^WTo<vJzSwm6{r@mGrOC4P4tKaj<cAR51f$5Yn$L~BA;$>oYds71{=Lx$(z)hnWT-w$pgG<-BZ-i|XQT-y)c'
            'uCFjx?g?3c;q!DnNEaE#{A1pyg07Ai);yZ7GeV2N{yS#0NOdCKJ?48~MYAR1K=3{Lgv{H~_E=$m)R|Ir7O~L+d(L=Fug+DThmxf1'
            'pYt8tJGw3BFY7`_!M!oFn;;*;Rsjh(m_(}?3`hBB{_y_Sv(7mt#G{-Iy0#vus_h<V!<bs<1^o6QpJ1f@B*3#9WVsBJQ$_1E8aeVe'
            '3hcuO<aWewkCHT*Ovk}(G?{`z22$nico{6FaX!M!V3n*v6;4*xkuMe*y_DhJosZ|E8ZAj!t4HP;U`PS)KeR7TkG5t;CR9)}cpc4n'
            'neXF*78I=>D<f!yhiyE^2}bnpm_RoV4c5;&s4h?##)comfir3)y=Lpk#OP0gR>5Z-5%a%-R9E}'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
video_analysis = _local_pipeline_modules['video_analysis']
render_utils = _local_pipeline_modules['render_utils']
google_ai = _local_pipeline_modules['google_ai']



## 1. Test Videosu

İki seçenek: (A) kendi dosyanı yükle, (B) bir YouTube URL'sinden kısa bir klip indir. Gerçekçi bir karşılaştırma için **3–6 dakikalık** bir örnek yeterli (tüm videoyu değil).

**YouTube "Sign in to confirm you're not a bot" hatası alırsan** (Colab IP'leri sık engellenir), iki yoldan biri:
- **En garantili — SECENEK A (upload):** Videoyu kendi bilgisayarına indir (ör. tarayıcı uzantısıyla ya da yerelde `yt-dlp`), Colab sol paneldeki 📁 **Files**'tan yükle, `SOURCE_MODE = 'upload'` yap ve `UPLOADED_PATH`'i dosya adına eşitle.
- **YouTube'dan indirmeye devam etmek istersen — cookies:** Chrome/Firefox'a "Get cookies.txt LOCALLY" gibi bir uzantı kur, YouTube'da oturum açıkken `cookies.txt` dışa aktar, Colab Files'a yükle ve aşağıda `COOKIES_FILE = 'cookies.txt'` yaz. Bu, yt-dlp'ye "gerçek bir tarayıcı oturumu" gösterip bot kontrolünü çoğu zaman aşar.

In [ ]:
# SECENEK A: Kendi dosyani kullan. Colab ve yerel Jupyter otomatik algilanir.
# SECENEK B: YouTube URL'sinden indir.

SOURCE_MODE = 'upload'  # 'upload' veya 'youtube'
YOUTUBE_URL = ''
VIDEO_FILENAME = 'THE ABSOLUTE BEST OF DAEMON TARGARYEN - ABC CINEMA (1080p).mp4'
COOKIES_FILE = ''
CLIP_START_SEC = 0
CLIP_DURATION_SEC = 240

if IS_COLAB:
    SEARCH_ROOTS = [Path('/content/sample_data'), Path('/content')]
else:
    SEARCH_ROOTS = [
        PROJECT_ROOT / 'sample_data',
        PROJECT_ROOT,
        Path.home() / 'Downloads',
        Path.home() / 'Videos',
    ]

UPLOADED_PATH = str(SEARCH_ROOTS[0] / VIDEO_FILENAME)
RAW_VIDEO = str(RUNTIME_DIR / 'source_raw.mp4')
VIDEO_PATH = str(RUNTIME_DIR / 'source_clip.mp4')
AUDIO_PATH = str(RUNTIME_DIR / 'source_clip.wav')

def resolve_uploaded_path(path):
    """Tam yol yoksa Colab veya yerel arama klasorlerinde kaynak videoyu bulur."""
    requested = Path(path)
    if requested.is_file():
        return str(requested.resolve())

    print(f'UYARI: tam yol bulunamadi -> {requested}')
    generated_names = {'source_clip.mp4', 'input_video.mp4'}
    exact_matches = []
    fallback_matches = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        exact = root / VIDEO_FILENAME
        if exact.is_file():
            exact_matches.append(exact)
        for extension in ('*.mp4', '*.mkv', '*.webm', '*.mov'):
            fallback_matches.extend(
                item for item in root.rglob(extension)
                if item.name not in generated_names
            )

    candidates = exact_matches or sorted(set(fallback_matches))
    print('Bulunan kaynak videolar:', [str(item) for item in candidates])
    if len(candidates) == 1:
        selected = str(candidates[0].resolve())
        print('Otomatik kullanilan video:', selected)
        return selected
    if len(candidates) > 1:
        raise FileNotFoundError(
            f'Birden fazla video bulundu. VIDEO_FILENAME veya UPLOADED_PATH degerini netlestir: {candidates}'
        )
    raise FileNotFoundError(
        f'Video bulunamadi. Aranan klasorler: {[str(root) for root in SEARCH_ROOTS]}'
    )

def download_youtube(url, out_path, cookies_file=''):
    base_cmd = [
        'yt-dlp',
        '-f', 'bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]/b',
        '--merge-output-format', 'mp4',
        '--no-playlist',
        '-o', out_path,
    ]
    attempts = [['--cookies', cookies_file]] if cookies_file else []
    attempts += [[], ['--extractor-args', 'youtube:player_client=android']]
    last_stderr = ''
    for extra in attempts:
        proc = subprocess.run(base_cmd + extra + [url], capture_output=True, text=True)
        if proc.returncode == 0:
            print('Indirme basarili.')
            return
        last_stderr = proc.stderr
    print(last_stderr[-3000:])
    raise RuntimeError('yt-dlp indirme basarisiz oldu; upload modunu veya cookies dosyasini kullan.')

if SOURCE_MODE == 'youtube':
    assert YOUTUBE_URL, 'YOUTUBE_URL bos olamaz'
    download_youtube(YOUTUBE_URL, RAW_VIDEO, cookies_file=COOKIES_FILE)
else:
    RAW_VIDEO = resolve_uploaded_path(UPLOADED_PATH)

file_size_mb = os.path.getsize(RAW_VIDEO) / (1024 ** 2)
print(f'Kaynak video: {RAW_VIDEO}')
print(f'Dosya boyutu: {file_size_mb:.2f} MB')
if file_size_mb < 1:
    raise RuntimeError('Video dosyasi cok kucuk; yukleme veya kopyalama tamamlanmamis olabilir.')

if FFPROBE_BIN:
    probe = subprocess.run(
        [
            FFPROBE_BIN, '-v', 'error',
            '-show_entries', 'stream=index,codec_type,codec_name,width,height,duration',
            '-of', 'json', RAW_VIDEO,
        ],
        capture_output=True,
        text=True,
    )
    if probe.returncode != 0:
        raise RuntimeError(f'FFprobe kaynak videoyu okuyamadi: {probe.stderr[-3000:]}')
    streams = json.loads(probe.stdout).get('streams', [])
    print('Bulunan akislar:', streams)
    if not any(stream.get('codec_type') == 'video' for stream in streams):
        raise RuntimeError('Dosyada video akisi bulunamadi.')
    if not any(stream.get('codec_type') == 'audio' for stream in streams):
        raise RuntimeError('Dosyada ses akisi bulunamadi; transcript uretilemez.')
else:
    print('FFprobe yok; akis kontrolu FFmpeg donusumu sirasinda yapilacak.')

# Kanonik klip+ses cikarimi (master notebook ve verify_local.py ile ayni fonksiyon).
video_analysis.extract_clip_and_audio(
    FFMPEG_BIN, RAW_VIDEO, VIDEO_PATH, AUDIO_PATH, CLIP_START_SEC, CLIP_DURATION_SEC,
)

print('Klip hazir:', VIDEO_PATH)
print('Ses hazir:', AUDIO_PATH)

## 2. Transcript Çıkarımı — Aday Modeller

Adaylar: `faster-whisper` (tiny / small / large-v3-turbo, lokal & ücretsiz) ve `OpenAI gpt-4o-mini-transcribe` (API, ücretli — anahtar varsa).

In [ ]:
import gc

transcripts = {}
transcript_segments = {}

try:
    cuda_check = subprocess.run(['nvidia-smi'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    cuda_available = cuda_check.returncode == 0
except FileNotFoundError:
    cuda_available = False
local_gpu_opt_in = os.environ.get('YOUTUBE_AUTOMATION_USE_GPU') == '1'
cuda_available = cuda_available and (IS_COLAB or local_gpu_opt_in)
TRANSCRIBE_DEVICE = 'cuda' if cuda_available else 'cpu'
TRANSCRIBE_COMPUTE_TYPE = 'float16' if TRANSCRIBE_DEVICE == 'cuda' else 'int8'
# Karar (gercek test sonuclarina gore): large-v3-turbo GPU'da varsayilan
# (~%91 ortalama kelime guveni), small CPU fallback (~%85). tiny (~%72) sadece
# karsilastirma icin tutuluyor, production icin onerilmez.
TRANSCRIPT_MODELS = ['tiny', 'small']
if TRANSCRIBE_DEVICE == 'cuda':
    TRANSCRIPT_MODELS.append('large-v3-turbo')
else:
    print('GPU bulunamadi; large-v3-turbo cok yavas olacagi icin atlandi. CPU fallback: small.')

print('Transkripsiyon cihazi:', TRANSCRIBE_DEVICE, '| modeller:', TRANSCRIPT_MODELS)

# Bu notebook'un katkisi: AYNI kanonik transcribe_audio() fonksiyonunu birden
# fazla model boyutuyla calistirip karsilastirmak.
for size in TRANSCRIPT_MODELS:
    candidate_name = f'faster-whisper-{size}'
    t0 = time.time()
    text, segment_rows = video_analysis.transcribe_audio(
        AUDIO_PATH, model_size=size, device=TRANSCRIBE_DEVICE, compute_type=TRANSCRIBE_COMPUTE_TYPE,
    )
    latency = time.time() - t0
    transcripts[candidate_name] = text
    transcript_segments[candidate_name] = segment_rows
    log_result(
        'transcript', candidate_name, quality_score=None, cost_usd=0.0, latency_sec=latency,
        notes=f'{len(text)} karakter, {len(segment_rows)} segment, VAD+timestamps',
    )
    gc.collect()

if OPENAI_API_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    t0 = time.time()
    # gpt-4o-mini-transcribe: whisper-1'in yerini alan guncel, daha ucuz model.
    with open(AUDIO_PATH, 'rb') as f:
        resp = client.audio.transcriptions.create(model='gpt-4o-mini-transcribe', file=f)
    latency = time.time() - t0
    transcripts['openai-gpt-4o-mini-transcribe'] = resp.text
    cost = (CLIP_DURATION_SEC / 60) * 0.003  # yaklasik; platform.openai.com/docs/pricing
    log_result('transcript', 'openai-gpt-4o-mini-transcribe', quality_score=None, cost_usd=cost,
               latency_sec=latency, notes=f'{len(resp.text)} karakter')
else:
    print('OpenAI key yok, openai-gpt-4o-mini-transcribe atlandi.')


In [ ]:
# Ciktilari yan yana oku ve kalite puanini (1-5) kendin gir.
for name, text in transcripts.items():
    print('=' * 20, name, '=' * 20)
    print(text[:800], '...\n')

transcript_quality = {
    # ornek: 'faster-whisper-tiny': 2, 'faster-whisper-small': 3, 'faster-whisper-large-v3-turbo': 4
}

for r in results:
    if r['task'] == 'transcript' and r['candidate'] in transcript_quality:
        r['quality_score_1_5'] = transcript_quality[r['candidate']]
print('Kalite puanlarini transcript_quality sozlugune girip bu hucreyi tekrar calistir.')

## 3. Sahne Tespiti — Aday Yöntemler

Adaylar: PySceneDetect `ContentDetector` ve `AdaptiveDetector`. Kalite ölçütü: bulunan sahne sayısının videoyu mantıklı şekilde bölüp bölmediği (elle kontrol).

In [ ]:
scene_results = {}

# Kanonik detect_scenes() iki dedektorle karsilastirmali calistirilir.
for name, detector_key in [('content-detector', 'content'), ('adaptive-detector', 'adaptive')]:
    t0 = time.time()
    scene_list = video_analysis.detect_scenes(VIDEO_PATH, detector=detector_key)
    latency = time.time() - t0
    scene_results[name] = scene_list
    log_result('scene-detection', name, quality_score=None, cost_usd=0.0, latency_sec=latency,
               notes=f'{len(scene_list)} sahne bulundu')


In [ ]:
scene_quality = {
    # ornek: 'content-detector': 4, 'adaptive-detector': 3
}
for r in results:
    if r['task'] == 'scene-detection' and r['candidate'] in scene_quality:
        r['quality_score_1_5'] = scene_quality[r['candidate']]
print('Sahne sayilarini yukarida gordukten sonra scene_quality sozlugunu doldurup bu hucreyi tekrar calistir.')

## 4. Ses Enerjisi Analizi (Destek Sinyali)

Bu adım tek bir sabit yöntemle (librosa RMS enerji tepe noktaları) çalışır — aday model karşılaştırması gerektirmez, sadece hot-moment skorlamasına ek sinyal sağlar. Bu yüzden burada kalite/maliyet karşılaştırması yapılmaz.

In [ ]:
import numpy as np

t0 = time.time()
# Kanonik compute_audio_peaks(): 16kHz yukleme + RMS + min-mesafeli tepe gruplama.
audio_info = video_analysis.compute_audio_peaks(AUDIO_PATH, top_n=10, min_distance_sec=1.5)
times, rms = audio_info['times'], audio_info['rms']
audio_peak_details = audio_info['peaks']
audio_peaks = [item['time_sec'] for item in audio_peak_details]
latency = time.time() - t0
log_result('audio-energy', 'librosa-rms', quality_score='n/a', cost_usd=0.0, latency_sec=latency,
           notes=f'gruplanmis en yuksek enerjili anlar: {[round(t, 1) for t in audio_peaks]}')


## 5. Yerel + Opsiyonel LLM Hot-Moment Skorlama

Önce transcript, sahne ve ses sinyallerinden API gerektirmeyen yerel hot-moment adayları üretilir. Aynı adaylar, API anahtarı varsa LLM sağlayıcılarıyla karşılaştırılır.

In [ ]:
t0 = time.time()
TRANSCRIPT_PREFERENCE = [
    'faster-whisper-large-v3-turbo',
    'faster-whisper-small',
    'faster-whisper-tiny',
]
BEST_TRANSCRIPT_CANDIDATE = next(
    name for name in TRANSCRIPT_PREFERENCE
    if name in transcript_segments and transcripts.get(name, '').strip()
)
print('Kullanilan transcript:', BEST_TRANSCRIPT_CANDIDATE)

LOCAL_WINDOW_SEC = 30.0
LOCAL_STRIDE_SEC = 10.0
TOP_HOT_MOMENTS = 5

analysis_duration = min(float(CLIP_DURATION_SEC), float(times[-1]))
content_scenes = scene_results.get('content-detector', [])
scene_boundaries = [end for _, end in content_scenes[:-1]] if content_scenes else []

# Kanonik skorlama: master notebook da AYNI score_hot_moments() fonksiyonunu cagirir.
hot_moments = video_analysis.score_hot_moments(
    transcript_segments[BEST_TRANSCRIPT_CANDIDATE], times, rms, scene_boundaries,
    analysis_duration, CLIP_START_SEC, top_n=TOP_HOT_MOMENTS,
    window_sec=LOCAL_WINDOW_SEC, stride_sec=LOCAL_STRIDE_SEC,
)

# Kanonik onizleme klibi uretimi (her ana bir 720p klip).
video_analysis.generate_preview_clips(FFMPEG_BIN, VIDEO_PATH, hot_moments, RESULTS_DIR / 'hot_moment_previews')

# Kanonik manifest: status HER ZAMAN 'review_required' (asla otomatik approved).
hot_moments_manifest = video_analysis.build_hot_moments_manifest(
    hot_moments, transcript_candidate=BEST_TRANSCRIPT_CANDIDATE,
    window_sec=LOCAL_WINDOW_SEC, stride_sec=LOCAL_STRIDE_SEC,
    weights=video_analysis.DEFAULT_WEIGHTS,
    raw_video_path=RAW_VIDEO, analysis_clip_path=VIDEO_PATH,
    clip_start_sec=CLIP_START_SEC, clip_duration_sec=CLIP_DURATION_SEC,
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
hot_moments_path = RESULTS_DIR / 'hot_moments.json'
with open(hot_moments_path, 'w', encoding='utf-8') as hot_file:
    json.dump(hot_moments_manifest, hot_file, ensure_ascii=False, indent=2)

local_latency = time.time() - t0
log_result(
    'hotmoment-scoring', 'local-multisignal-v2', quality_score=None, cost_usd=0.0,
    latency_sec=local_latency,
    notes=str([(item['start_sec'], item['end_sec'], item['score']) for item in hot_moments]),
    status='review_required',
)
print('Yerel hot-moment dosyasi:', hot_moments_path, '(status: review_required)')
display(pd.DataFrame(hot_moments)[['rank', 'aligned_start_sec', 'aligned_end_sec', 'score', 'reason', 'preview_path', 'transcript']])

# Opsiyonel LLM karsilastirmasinda yerel yontemin sectigi ayni pencereleri kullan.
windows = [item['transcript'] for item in hot_moments if item['transcript']]
print(f'{len(windows)} zaman kodlu pencere olusturuldu.')

SCORING_PROMPT = (
    "Asagidaki video transkript parcasini oku. 0-100 arasi bir 'hot moment' (vurucu, "
    "paylasilabilir, dikkat cekici an) skoru ver ve 1 cumlelik gerekce yaz. "
    "SADECE JSON don: {{\"score\": <int>, \"reason\": \"...\"}}\n\nMetin:\n{chunk}"
)


In [ ]:
def parse_json_score(raw):
    try:
        start = raw.find('{'); end = raw.rfind('}') + 1
        return json.loads(raw[start:end])
    except Exception:
        return {'score': None, 'reason': raw[:200]}

llm_outputs = {'local-multisignal-v2': hot_moments}

# --- OpenAI gpt-5.6-luna (gpt-4o-mini'nin yerini alan dusuk maliyetli model) ---
if OPENAI_API_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    t0 = time.time()
    scored = []
    for chunk in windows:
        resp = client.chat.completions.create(
            model='gpt-5.6-luna',
            messages=[{'role': 'user', 'content': SCORING_PROMPT.format(chunk=chunk)}],
            temperature=0,
        )
        scored.append(parse_json_score(resp.choices[0].message.content))
    latency = time.time() - t0
    llm_outputs['openai-gpt-5.6-luna'] = scored
    approx_tokens = sum(len(c) for c in windows) / 4
    cost = (approx_tokens / 1_000_000) * 1.0  # yaklasik input fiyati, guncelini platform.openai.com/docs/pricing adresinden kontrol et
    log_result('hotmoment-scoring', 'openai-gpt-5.6-luna', quality_score=None, cost_usd=cost, latency_sec=latency,
               notes=str([s.get('score') for s in scored]))
else:
    print('OpenAI key yok, gpt-5.6-luna atlandi.')

# --- Anthropic claude-haiku-4-5 ---
if ANTHROPIC_API_KEY:
    import anthropic
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    t0 = time.time()
    scored = []
    for chunk in windows:
        resp = client.messages.create(
            model='claude-haiku-4-5',
            max_tokens=200,
            messages=[{'role': 'user', 'content': SCORING_PROMPT.format(chunk=chunk)}],
        )
        scored.append(parse_json_score(resp.content[0].text))
    latency = time.time() - t0
    llm_outputs['anthropic-claude-haiku-4-5'] = scored
    approx_tokens = sum(len(c) for c in windows) / 4
    cost = (approx_tokens / 1_000_000) * 1.00  # yaklasik input fiyati, guncelini platform.claude.com/docs/en/pricing adresinden kontrol et
    log_result('hotmoment-scoring', 'anthropic-claude-haiku-4-5', quality_score=None, cost_usd=cost, latency_sec=latency,
               notes=str([s.get('score') for s in scored]))
else:
    print('Anthropic key yok, claude-haiku-4-5 atlandi.')

# --- Google: SABIT MODEL YOK. Her cagri kanonik router uzerinden gider -------
# (local_pipeline/google_ai.py: Rota A = gemini-3.6-flash -> gemini-3.5-flash ->
#  gemini-3.5-flash-lite -> gemma-4-31b-it, ilk basarida durur, kota defteri
#  tutulur, hatalar siniflandirilir ve sanitize edilir.)
#
# Ucretsiz katman veri politikasi: buraya SADECE yerel olarak uretilmis transcript
# pencereleri gonderilir; ham video ASLA yuklenmez.
GOOGLE_CREDENTIAL = google_ai.resolve_secret('google_api', get_secret)
print('Gemini kimlik bilgisi:', google_ai.describe_credential_state(GOOGLE_CREDENTIAL))

if GOOGLE_CREDENTIAL.configured:
    _g_client = google_ai.build_google_client(GOOGLE_CREDENTIAL)
    _g_manifest = google_ai.build_google_capability_manifest(
        _g_client, credential=GOOGLE_CREDENTIAL)
    _g_ledger = google_ai.QuotaLedger()
    _g_selection = google_ai.select_google_route(
        'hotmoment_review', manifest=_g_manifest, ledger=_g_ledger)
    print(google_ai.format_route_line(_g_selection, GOOGLE_CREDENTIAL))

    t0 = time.time()
    scored, _g_models, _g_failed = [], [], None
    for chunk in windows:
        _g_result = google_ai.run_google_hotmoment_review(
            SCORING_PROMPT.format(chunk=chunk), client=_g_client,
            manifest=_g_manifest, ledger=_g_ledger,
            required_fields=(), checks=(),
        )
        if _g_result['status'] != 'ok':
            _g_failed = _g_result['status_reason']
            break
        _g_models.append(_g_result['model'])
        scored.append(parse_json_score(_g_result['output']))
    latency = time.time() - t0

    if _g_failed:
        # Cikti UYDURULMAZ: basarisizlik acikca 'failed' olarak loglanir.
        log_result('hotmoment-scoring', 'google-router', quality_score=None, cost_usd=0.0,
                   latency_sec=latency, notes=f'Google rotasi basarisiz: {_g_failed}',
                   status='failed')
        print('Google hot-moment incelemesi basarisiz:', _g_failed)
    else:
        _g_label = 'google-' + (sorted(set(_g_models))[0] if _g_models else 'router')
        llm_outputs[_g_label] = scored
        # Ucretsiz katman: dolar maliyeti 0; gercek butce RPM/TPM/RPD kotasidir.
        log_result('hotmoment-scoring', _g_label, quality_score=None, cost_usd=0.0,
                   latency_sec=latency,
                   notes=str([s.get('score') for s in scored])
                         + f' | kullanilan modeller: {sorted(set(_g_models))}')
else:
    print('Google key yok (GeminiKey / GOOGLE_API_KEY), Google adayi atlandi.')
    print(google_ai.credential_help_text('google_api'))

In [ ]:
for name, scored in llm_outputs.items():
    print('=' * 20, name, '=' * 20)
    for s in scored:
        print(s)
    print()

hotmoment_quality = {
    # ornek: 'local-multisignal-v2': 4, 'openai-gpt-5.6-luna': 4
}
for r in results:
    if r['task'] == 'hotmoment-scoring' and r['candidate'] in hotmoment_quality:
        r['quality_score_1_5'] = hotmoment_quality[r['candidate']]
print('Skorlarin tutarliligini/isabetini degerlendirip hotmoment_quality sozlugunu doldur, sonra tekrar calistir.')

## 6. Karşılaştırma Tablosu ve Dışa Aktarım

In [ ]:
df = pd.DataFrame(results)
# Not: satir bazli status artik log_result tarafindan ayarlaniyor; eksikleri guvenli varsayilana tamamla.
if 'status' not in df.columns:
    df['status'] = 'experimental'
else:
    df['status'] = df['status'].fillna('experimental')
display(df)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
out_path = RESULTS_DIR / '01_video_analysis_hotmoment_sonuclar.csv'
df.to_csv(out_path, index=False)
print('Kaydedildi:', out_path)
print('\nBu dosyayi indirip experiments/results/ altina koy; roadmap Faz 1deki provider registry karari bu sonuclara gore verilecek.')

## 7. Tanılama Paketini İndir

Bu hücre transcriptleri, sahne zamanlarını, ses tepe noktalarını ve karşılaştırma CSV'sini tek bir ZIP dosyasında toplar.

In [ ]:
import zipfile

scene_report = {}
for detector_name, scenes in globals().get('scene_results', {}).items():
    scene_report[detector_name] = {
        'count': len(scenes),
        # detect_scenes() zaten (start_sec, end_sec) float ikilileri dondurur.
        'scenes': [{'start_sec': round(start, 3), 'end_sec': round(end, 3)} for start, end in scenes],
    }

diagnostic_report = {
    'status': 'experimental',
    'status_reason': 'provider comparison notebook output; not a final production artifact',
    'source': {
        'raw_video': globals().get('RAW_VIDEO'),
        'video_path': globals().get('VIDEO_PATH'),
        'audio_path': globals().get('AUDIO_PATH'),
        'clip_start_sec': globals().get('CLIP_START_SEC'),
        'clip_duration_sec': globals().get('CLIP_DURATION_SEC'),
    },
    'transcripts': {
        name: {
            'character_count': len(text),
            'full_text': text,
            'segments': globals().get('transcript_segments', {}).get(name, []),
        }
        for name, text in globals().get('transcripts', {}).items()
    },
    'scene_detection': scene_report,
    'audio_peaks_sec': globals().get('audio_peaks', []),
    'audio_peak_details': globals().get('audio_peak_details', []),
    'local_hot_moments': globals().get('hot_moments', []),
    'llm_hotmoment_outputs': globals().get('llm_outputs', {}),
    'benchmark_results': globals().get('results', []),
}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
report_path = RESULTS_DIR / '01_video_analysis_tanilama.json'
zip_path = RESULTS_DIR / '01_video_analysis_paylasim_paketi.zip'

with open(report_path, 'w', encoding='utf-8') as report_file:
    json.dump(diagnostic_report, report_file, ensure_ascii=False, indent=2)

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(report_path, arcname=report_path.name)
    if Path(out_path).exists():
        archive.write(out_path, arcname=Path(out_path).name)
    if Path(globals().get('hot_moments_path', '')).exists():
        archive.write(hot_moments_path, arcname=Path(hot_moments_path).name)
    for preview_file in (RESULTS_DIR / 'hot_moment_previews').glob('*.mp4'):
        archive.write(preview_file, arcname=f'hot_moment_previews/{preview_file.name}')

print('Paylasim paketi hazir:', zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    print('Otomatik indirme baslatilamadi. Files panelinden elle indir:', zip_path)
    print('Detay:', exc)


## 8. Karar (elle doldurulacak)

Yukarıdaki tabloyu inceledikten sonra bu hücreyi düzenleyip kararını ve kısa gerekçeni yaz. Bu karar Faz 1'de `packages/providers/vision-analysis` ve `packages/providers/llm` adaptörlerinin varsayılan sağlayıcısını belirleyecek.

- **Transcript:** _(seçilen aday)_ — gerekçe: _...
- **Sahne tespiti:** _(seçilen aday)_ — gerekçe: _...
- **Hot-moment skorlama:** _(seçilen aday)_ — gerekçe: _...
